# RNN

Working

Post-HPO Performs about the same as other methods

Epoch [39/100], Train Loss: 1.0418, Val Loss: 1.1461, Val AUC: 0.7355, Val AUPRC: 0.2195
Validation loss did not improve for 15 epoch(s).
Early stopping triggered after 39 epochs.
Training finished.
Loading best model from /home/server/Projects/data/AKI/models/best_plain_rnn_model.pth for final evaluation.

Starting final evaluation with the best/last model...
                                                                          

--- Final Test Performance (Best/Last Model) ---
Precision: 0.0851
Sensitivity: 0.8536
Accuracy: 0.4406
rc_auc: 0.7343
pr_auc: 0.2178
Specificity: 0.4142
Negative Predictive Value: 0.9779
F1 Score: 0.1547
Results saved to /home/server/Projects/data/AKI/results/plain_rnn_results.pkl
Script finished.


Pre-HPO: Performs significantly worse than other methods.

Epoch [33/100], Train Loss: 1.1237, Val Loss: 1.1855, Val AUC: 0.6977, Val AUPRC: 0.1768
Validation loss did not improve for 15 epoch(s).
Early stopping triggered after 33 epochs.
Training finished.
Loading best model from /home/server/Projects/data/AKI/models/best_plain_rnn_model.pth for final evaluation.

Starting final evaluation with the best/last model...
                                                                          

--- Final Test Performance (Best/Last Model) ---
Precision: 0.0750
Sensitivity: 0.8814
Accuracy: 0.3412
rc_auc: 0.6993
pr_auc: 0.1718
Specificity: 0.3067
Negative Predictive Value: 0.9759
F1 Score: 0.1383
Results saved to /home/server/Projects/data/AKI/results/plain_rnn_results.pkl
Script finished.

In [8]:
# %%
import os
import numpy as np
import torch
from torch.nn.utils.rnn import pack_padded_sequence # pad_packed_sequence might not be strictly needed if only using final hidden state
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
import torch.optim as optim

import random
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_curve, auc, precision_recall_curve, average_precision_score)
from sklearn.model_selection import train_test_split
import pandas as pd
# Ensure utils.py is in the same directory or Python path
from utils import performance_dict, sigmoid
from tqdm import tqdm

# --- Configuration for the Plain RNN Model ---
CONFIG = {
    "data_file": '/home/server/Projects/data/AKI/lstm_trainable.pkl', # !!! UPDATE THIS PATH !!!
    "results_output_file": '/home/server/Projects/data/AKI/results/plain_rnn_results.pkl',
    "tensorboard_log_dir_base": '/home/server/Projects/data/AKI/runs/plain_rnn/',
    "best_model_save_path": '/home/server/Projects/data/AKI/models/best_plain_rnn_model.pth',
    "test_split_size": 0.2,
    "random_seed": 42,
    "num_epochs": 100,
    "learning_rate": 0.0002, 
    "weight_decay": 4.212169152071669e-06,
    "gradient_clip_value": 1.0,
    "lr_scheduler_patience": 7,
    "lr_scheduler_factor": 0.1,
    "early_stopping_patience": 15,
    
    # --- Architecture Specifics for Plain RNN ---
    "rnn_hidden_dim": 128,     # Hidden dim for RNN
    "num_rnn_layers": 1,       # Single Layer RNN
    "rnn_nonlinearity": 'relu', # 'tanh' or 'relu' for nn.RNN
    "dropout_rate": 0.5,       # Dropout after RNN before FC
    
    "batch_size": 640,          
    "eval_batch_size": 640,     
    "classification_threshold": 0.3, 
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load data (once)
try:
    df_full = pd.read_pickle(CONFIG["data_file"])
except FileNotFoundError:
    print(f"ERROR: Data file not found at {CONFIG['data_file']}")
    print("Please ensure the path in CONFIG['data_file'] is correct.")
    exit()
except Exception as e:
    print(f"Error loading data file {CONFIG['data_file']}: {e}")
    exit()


# %%
# --- Data Preparation ---
def df_to_tensors(df, all_columns):
    non_tabular_cols = ['op_id', 'time_tensors', 'seq_len', 'aki', 'aki_boolean']
    tabular_feature_cols = [col for col in all_columns if col not in non_tabular_cols]
    if tabular_feature_cols and not df[tabular_feature_cols].empty:
        X_tab = torch.tensor(df[tabular_feature_cols].values).float()
    else:
        X_tab = torch.empty((len(df), 0)).float()
    X_time = torch.stack(df['time_tensors'].tolist()).float()
    y = torch.tensor(df['aki'].values).unsqueeze(1).float()
    y_binary = df['aki_boolean'].values.astype(bool)
    sequence_lengths = torch.tensor(df['seq_len'].tolist()).long()
    return X_tab, X_time, sequence_lengths, y, y_binary

def prepare_data(full_df, test_size, random_seed):
    df_train, df_test = train_test_split(full_df, test_size=test_size, random_state=random_seed, stratify=full_df['aki_boolean'])
    print(f"Original train size: {len(df_train)}")
    print(f"Test size: {len(df_test)}")
    all_columns = full_df.columns.tolist()
    num_positive_aki_train = df_train['aki_boolean'].sum()
    num_negative_aki_train = len(df_train) - num_positive_aki_train
    print(f"Training set: Positive AKI cases: {num_positive_aki_train}, Negative AKI cases: {num_negative_aki_train}")
    X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train = df_to_tensors(df_train, all_columns)
    X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test = df_to_tensors(df_test, all_columns)
    
    CONFIG["input_features"] = X_time_train.shape[2]
    print(f"Number of input time-series features: {CONFIG['input_features']}")
    
    return (X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train,
            X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test,
            num_positive_aki_train, num_negative_aki_train)

(X_tab_train_full, X_time_train_full, seq_len_train_full, y_train_full, y_binary_train_full,
 X_tab_test_full, X_time_test_full, seq_len_test_full, y_test_full, y_binary_test_full,
 num_pos_train_full, num_neg_train_full) = prepare_data(
    df_full, CONFIG["test_split_size"], CONFIG["random_seed"]
)
input_features_global = CONFIG["input_features"]

# %%
# --- Utility Functions ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["random_seed"])

def save_single_run_results(model_name, metrics_dict, output_file):
    results_to_save = {k: [v] if not isinstance(v, (list, np.ndarray)) else v for k,v in metrics_dict.items()}
    df_result = pd.DataFrame(results_to_save)
    df_result['model_name'] = model_name
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    if os.path.exists(output_file):
        try:
            df_output = pd.read_pickle(output_file)
            df_output = df_output[df_output['model_name'] != model_name]
            df_output = pd.concat([df_output, df_result], ignore_index=True)
        except EOFError: df_output = df_result
    else: df_output = df_result
    df_output.to_pickle(output_file)
    print(f"Results saved to {output_file}")

# %%
# --- Model Definition for Plain RNN ---
class SimpleRNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes, nonlinearity='tanh', dropout_rate=0.2):
        super(SimpleRNNModel, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers # Expected to be 1 for "plain"
        self.num_classes = num_classes
        
        self.rnn = nn.RNN(input_size=input_dim, 
                          hidden_size=hidden_dim,
                          num_layers=num_layers,
                          nonlinearity=nonlinearity, # 'tanh' or 'relu'
                          batch_first=True,
                          bidirectional=False, # Unidirectional
                          dropout=dropout_rate if num_layers > 1 else 0) # RNN internal dropout for stacked

        # Classifier part
        self.dropout = nn.Dropout(dropout_rate) # Dropout after RNN output
        self.fc_intermediate_dim = max(1, hidden_dim // 2)
        self.fc0 = nn.Linear(hidden_dim, self.fc_intermediate_dim) 
        self.relu_fc = nn.ReLU()
        self.fc_final = nn.Linear(self.fc_intermediate_dim, num_classes)

    def forward(self, x_time, seq_lens):
        packed_input = pack_padded_sequence(x_time, seq_lens.cpu(), batch_first=True, enforce_sorted=False)
        # output_packed contains all hidden states for each t from the last layer
        # hn is the final hidden state: (num_layers * num_directions, batch, hidden_size)
        # For unidirectional, num_directions = 1. For num_layers = 1, hn is (1, batch, hidden_size)
        _, hn = self.rnn(packed_input) 
        
        # Get the hidden state of the last layer
        # hn is (num_layers, batch, hidden_size) for unidirectional
        fc_input = hn.squeeze(0) if self.num_layers == 1 else hn[-1,:,:]
        
        out = self.dropout(fc_input)
        out = self.fc0(out)
        out = self.relu_fc(out)
        out = self.fc_final(out)
        return out

# %%
# --- Main Training and Evaluation ---
model = SimpleRNNModel( 
    input_dim=input_features_global,
    hidden_dim=CONFIG["rnn_hidden_dim"],
    num_layers=CONFIG["num_rnn_layers"],
    num_classes=1,
    nonlinearity=CONFIG["rnn_nonlinearity"],
    dropout_rate=CONFIG["dropout_rate"]
).to(device)

print(model)
print(f"Model Parameter Count: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

if num_pos_train_full > 0:
    pos_weight_value = num_neg_train_full / num_pos_train_full
else:
    pos_weight_value = 1.0; print("Warning: No positive samples in train for pos_weight.")
pos_weight = torch.tensor([pos_weight_value], device=device)
print(f"Calculated pos_weight for BCEWithLogitsLoss: {pos_weight.item():.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=CONFIG["lr_scheduler_factor"],
    patience=CONFIG["lr_scheduler_patience"])

run_name = f"SimpleRNN_h{CONFIG['rnn_hidden_dim']}_l{CONFIG['num_rnn_layers']}_{CONFIG['rnn_nonlinearity']}_classWeighted"
tensorboard_run_path = os.path.join(CONFIG["tensorboard_log_dir_base"], run_name)
os.makedirs(tensorboard_run_path, exist_ok=True)
writer = SummaryWriter(tensorboard_run_path)

num_train_samples = X_time_train_full.shape[0]
train_batch_size = CONFIG["batch_size"]
num_train_batches = int(np.ceil(num_train_samples / train_batch_size))
eval_batch_size = CONFIG["eval_batch_size"]
num_test_samples = X_time_test_full.shape[0]
num_eval_batches = int(np.ceil(num_test_samples / eval_batch_size))

best_val_loss = float('inf')
epochs_no_improve = 0
os.makedirs(os.path.dirname(CONFIG["best_model_save_path"]), exist_ok=True)

print(f"Starting training for {CONFIG['num_epochs']} epochs on {num_train_samples} original samples...")
for epoch in tqdm(range(CONFIG["num_epochs"]), desc="Epochs"):
    model.train()
    epoch_train_loss = 0.0
    for i in tqdm(range(num_train_batches), desc="Training Batches", leave=False):
        start_idx = i * train_batch_size
        end_idx = min(num_train_samples, (i + 1) * train_batch_size)
        batch_x_time = X_time_train_full[start_idx:end_idx].to(device)
        batch_seq_len = seq_len_train_full[start_idx:end_idx]
        batch_y_binary = torch.tensor(y_binary_train_full[start_idx:end_idx]).unsqueeze(1).float().to(device)
        optimizer.zero_grad()
        outputs = model(batch_x_time, batch_seq_len)
        loss = criterion(outputs, batch_y_binary)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CONFIG["gradient_clip_value"])
        optimizer.step()
        epoch_train_loss += loss.item()
    avg_epoch_train_loss = epoch_train_loss / num_train_batches
    writer.add_scalar('Loss/train', avg_epoch_train_loss, epoch)

    model.eval()
    all_val_outputs_logits = []; all_val_y_for_loss = []
    with torch.no_grad():
        for i in tqdm(range(num_eval_batches), desc="Validation Batches", leave=False):
            start_idx = i * eval_batch_size; end_idx = min(num_test_samples, (i + 1) * eval_batch_size)
            batch_x_time_val = X_time_test_full[start_idx:end_idx].to(device)
            batch_seq_len_val = seq_len_test_full[start_idx:end_idx]
            batch_y_binary_val_for_loss = torch.tensor(y_binary_test_full[start_idx:end_idx]).unsqueeze(1).float().to(device)
            outputs_batch_logits = model(batch_x_time_val, batch_seq_len_val)
            all_val_outputs_logits.append(outputs_batch_logits.cpu()); all_val_y_for_loss.append(batch_y_binary_val_for_loss.cpu())
    val_outputs_logits = torch.cat(all_val_outputs_logits, dim=0); val_y_for_loss = torch.cat(all_val_y_for_loss, dim=0)
    current_val_loss = criterion(val_outputs_logits.to(device), val_y_for_loss.to(device)).item()
    writer.add_scalar('Loss/val', current_val_loss, epoch)
    val_probs = torch.sigmoid(val_outputs_logits).numpy().squeeze()
    if val_probs.ndim == 0: val_probs = np.array([val_probs])
    val_pred_binary = (val_probs > CONFIG["classification_threshold"])
    current_y_true_for_metric = y_binary_test_full[:len(val_pred_binary)]
    
    try:
        dic = performance_dict(current_y_true_for_metric, val_pred_binary, val_probs)
        writer.add_scalar('Val/AUC', dic.get('rc_auc', np.nan), epoch)
        writer.add_scalar('Val/AUPRC', dic.get('pr_auc', np.nan), epoch)
        # ... (log other metrics)
        print(f"Epoch [{epoch+1}/{CONFIG['num_epochs']}], Train Loss: {avg_epoch_train_loss:.4f}, Val Loss: {current_val_loss:.4f}, Val AUC: {dic.get('rc_auc', np.nan):.4f}, Val AUPRC: {dic.get('pr_auc', np.nan):.4f}")
    except Exception as e:
        print(f"Error calculating performance_dict in validation: {e}")
        print(f"Epoch [{epoch+1}/{CONFIG['num_epochs']}], Train Loss: {avg_epoch_train_loss:.4f}, Val Loss: {current_val_loss:.4f}")

    scheduler.step(current_val_loss)
    if current_val_loss < best_val_loss:
        best_val_loss = current_val_loss; epochs_no_improve = 0
        torch.save(model.state_dict(), CONFIG["best_model_save_path"])
        print(f"Validation loss improved. Saved best model to {CONFIG['best_model_save_path']}")
    else:
        epochs_no_improve += 1
        print(f"Validation loss did not improve for {epochs_no_improve} epoch(s).")
    if epochs_no_improve >= CONFIG["early_stopping_patience"]:
        print(f"Early stopping triggered after {epoch+1} epochs.")
        break
writer.close()
print("Training finished.")

if os.path.exists(CONFIG["best_model_save_path"]):
    print(f"Loading best model from {CONFIG['best_model_save_path']} for final evaluation.")
    model.load_state_dict(torch.load(CONFIG["best_model_save_path"]))
else:
    print("No best model found. Using last model state.")

model.eval()
final_all_test_outputs_logits = []
print("\nStarting final evaluation with the best/last model...")
with torch.no_grad():
    for i in tqdm(range(num_eval_batches), desc="Final Evaluation Batches", leave=False):
        start_idx = i * eval_batch_size; end_idx = min(num_test_samples, (i + 1) * eval_batch_size)
        batch_x_time_test = X_time_test_full[start_idx:end_idx].to(device)
        batch_seq_len_test = seq_len_test_full[start_idx:end_idx]
        outputs_batch_logits = model(batch_x_time_test, batch_seq_len_test)
        final_all_test_outputs_logits.append(outputs_batch_logits.cpu())
final_test_outputs_logits = torch.cat(final_all_test_outputs_logits, dim=0)
final_test_probs = torch.sigmoid(final_test_outputs_logits).numpy().squeeze()
if final_test_probs.ndim == 0: final_test_probs = np.array([final_test_probs])
final_y_pred_binary = (final_test_probs > CONFIG["classification_threshold"])
current_y_true_for_final_metric = y_binary_test_full[:len(final_y_pred_binary)]

try:
    final_metrics = performance_dict(current_y_true_for_final_metric, final_y_pred_binary, final_test_probs)
    print("\n--- Final Test Performance (Best/Last Model) ---")
    for metric, value in final_metrics.items():
        if isinstance(value, (float, int, np.number)): print(f"{metric}: {value:.4f}")
    save_single_run_results(run_name, final_metrics, CONFIG["results_output_file"])
except Exception as e:
    print(f"Error calculating final performance_dict: {e}")

print("Script finished.")

Using device: cuda
Original train size: 43271
Test size: 10818
Training set: Positive AKI cases: 2595.0, Negative AKI cases: 40676.0
Number of input time-series features: 24
SimpleRNNModel(
  (rnn): RNN(24, 128, batch_first=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc0): Linear(in_features=128, out_features=64, bias=True)
  (relu_fc): ReLU()
  (fc_final): Linear(in_features=64, out_features=1, bias=True)
)
Model Parameter Count: 28033
Calculated pos_weight for BCEWithLogitsLoss: 15.67
Starting training for 100 epochs on 43271 original samples...


Epochs:   0%|          | 0/100 [00:00<?, ?it/s]/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no 

Epoch [1/100], Train Loss: 1.3022, Val Loss: 1.2944, Val AUC: 0.5857, Val AUPRC: 0.0977
Validation loss improved. Saved best model to /home/server/Projects/data/AKI/models/best_plain_rnn_model.pth


Epochs:   2%|▏         | 2/100 [00:04<03:18,  2.03s/it]

Epoch [2/100], Train Loss: 1.2688, Val Loss: 1.2464, Val AUC: 0.6483, Val AUPRC: 0.1201
Validation loss improved. Saved best model to /home/server/Projects/data/AKI/models/best_plain_rnn_model.pth


...

Epochs:  38%|███▊      | 38/100 [01:13<02:00,  1.94s/it]

Epoch [38/100], Train Loss: 1.0416, Val Loss: 1.1459, Val AUC: 0.7356, Val AUPRC: 0.2196
Validation loss did not improve for 14 epoch(s).


Epochs:  38%|███▊      | 38/100 [01:14<02:02,  1.97s/it]


Epoch [39/100], Train Loss: 1.0418, Val Loss: 1.1461, Val AUC: 0.7355, Val AUPRC: 0.2195
Validation loss did not improve for 15 epoch(s).
Early stopping triggered after 39 epochs.
Training finished.
Loading best model from /home/server/Projects/data/AKI/models/best_plain_rnn_model.pth for final evaluation.

Starting final evaluation with the best/last model...



--- Final Test Performance (Best/Last Model) ---
Precision: 0.0851
Sensitivity: 0.8536
Accuracy: 0.4406
rc_auc: 0.7343
pr_auc: 0.2178
Specificity: 0.4142
Negative Predictive Value: 0.9779
F1 Score: 0.1547
Results saved to /home/server/Projects/data/AKI/results/plain_rnn_results.pkl
Script finished.


# HPO RNN

Working

--- HPO Study Finished ---
Number of finished trials: 50

Best trial for Plain RNN:
  Value (Best Validation Loss): 1.1361
  Params: 
    learning_rate: 0.0002095430664254445
    rnn_hidden_dim: 128
    rnn_nonlinearity: relu
    dropout_rate: 0.5
    weight_decay: 4.212169152071669e-06
Best hyperparameters for Plain RNN saved to /home/server/Projects/data/AKI/results/hpo_plain_rnn/best_plain_rnn_hpo_params.json

To train a final Plain RNN model with these best hyperparameters, update your CONFIG dict,
set num_epochs to a higher value, and run a standard (non-HPO) training script.
Full HPO study results for Plain RNN saved to /home/server/Projects/data/AKI/results/hpo_plain_rnn/hpo_plain_rnn_study_results.csv
Script finished.

In [5]:
# %%
import os
import numpy as np
import torch
from torch.nn.utils.rnn import pack_padded_sequence # pad_packed_sequence not strictly needed for this model
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
import torch.optim as optim
import optuna
import functools
import math # Not used by SimpleRNNModel directly, but good to keep if other models share this template

import random
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_curve, auc, precision_recall_curve, average_precision_score)
from sklearn.model_selection import train_test_split
import pandas as pd
# Ensure utils.py is in the same directory or Python path
from utils import performance_dict, sigmoid
from tqdm import tqdm

# --- Configuration ---
CONFIG = {
    "data_file": '/home/server/Projects/data/AKI/lstm_trainable.pkl', # !!! UPDATE THIS PATH !!!
    "results_output_file_base": '/home/server/Projects/data/AKI/results/hpo_plain_rnn/',
    "tensorboard_log_dir_base": '/home/server/Projects/data/AKI/runs/hpo_plain_rnn/',
    # "best_model_save_path_base": '/home/server/Projects/data/AKI/models/hpo_plain_rnn/', # Optional
    "test_split_size": 0.2,
    "random_seed": 42,
    "num_epochs_hpo": 50,
    "gradient_clip_value": 1.0,
    "lr_scheduler_patience": 5,
    "lr_scheduler_factor": 0.2,
    "early_stopping_patience": 10,
    
    # --- Fixed Architectural Choices for PLAIN RNN HPO ---
    "num_rnn_layers": 1,       # Plain RNN is single layer
    
    "batch_size": 1024,        # Can try larger batch size for simpler RNN
    "eval_batch_size": 512,
    "classification_threshold": 0.3,
    "n_hpo_trials": 50, 
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load data (once)
try:
    df_full = pd.read_pickle(CONFIG["data_file"])
except FileNotFoundError:
    print(f"ERROR: Data file not found at {CONFIG['data_file']}")
    print("Please ensure the path in CONFIG['data_file'] is correct.")
    exit()
except Exception as e:
    print(f"Error loading data file {CONFIG['data_file']}: {e}")
    exit()


# %%
# --- Data Preparation ---
def df_to_tensors(df, all_columns):
    non_tabular_cols = ['op_id', 'time_tensors', 'seq_len', 'aki', 'aki_boolean']
    tabular_feature_cols = [col for col in all_columns if col not in non_tabular_cols]
    if tabular_feature_cols and not df[tabular_feature_cols].empty:
        X_tab = torch.tensor(df[tabular_feature_cols].values).float()
    else:
        X_tab = torch.empty((len(df), 0)).float()
    X_time = torch.stack(df['time_tensors'].tolist()).float()
    y = torch.tensor(df['aki'].values).unsqueeze(1).float()
    y_binary = df['aki_boolean'].values.astype(bool)
    sequence_lengths = torch.tensor(df['seq_len'].tolist()).long()
    return X_tab, X_time, sequence_lengths, y, y_binary

def prepare_data(full_df, test_size, random_seed):
    df_train, df_test = train_test_split(full_df, test_size=test_size, random_state=random_seed, stratify=full_df['aki_boolean'])
    print(f"Original train size: {len(df_train)}")
    print(f"Test size: {len(df_test)}")
    all_columns = full_df.columns.tolist()
    num_positive_aki_train = df_train['aki_boolean'].sum()
    num_negative_aki_train = len(df_train) - num_positive_aki_train
    print(f"Training set: Positive AKI cases: {num_positive_aki_train}, Negative AKI cases: {num_negative_aki_train}")
    X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train = df_to_tensors(df_train, all_columns)
    X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test = df_to_tensors(df_test, all_columns)
    
    CONFIG["input_features"] = X_time_train.shape[2]
    print(f"Number of input time-series features: {CONFIG['input_features']}")
    
    return (X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train,
            X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test,
            num_positive_aki_train, num_negative_aki_train)

(X_tab_train_full, X_time_train_full, seq_len_train_full, y_train_full, y_binary_train_full,
 X_tab_test_full, X_time_test_full, seq_len_test_full, y_test_full, y_binary_test_full,
 num_pos_train_full, num_neg_train_full) = prepare_data(
    df_full, CONFIG["test_split_size"], CONFIG["random_seed"]
)
input_features_global = CONFIG["input_features"]


# %%
# --- Utility Functions ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["random_seed"])

# (save_hpo_trial_results function can be kept if you want to save individual trial outputs)

# %%
# --- Model Definition for Plain RNN ---
class SimpleRNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes, nonlinearity='tanh', dropout_rate=0.2):
        super(SimpleRNNModel, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers # Expected to be 1 for "plain"
        self.num_classes = num_classes
        
        self.rnn = nn.RNN(input_size=input_dim, 
                          hidden_size=hidden_dim,
                          num_layers=num_layers,
                          nonlinearity=nonlinearity, # 'tanh' or 'relu'
                          batch_first=True,
                          bidirectional=False, # Unidirectional
                          dropout=dropout_rate if num_layers > 1 else 0) # RNN internal dropout for stacked

        self.dropout = nn.Dropout(dropout_rate) 
        self.fc_intermediate_dim = max(1, hidden_dim // 2)
        self.fc0 = nn.Linear(hidden_dim, self.fc_intermediate_dim) 
        self.relu_fc = nn.ReLU()
        self.fc_final = nn.Linear(self.fc_intermediate_dim, num_classes)

    def forward(self, x_time, seq_lens):
        packed_input = pack_padded_sequence(x_time, seq_lens.cpu(), batch_first=True, enforce_sorted=False)
        _, hn = self.rnn(packed_input) 
        # hn shape: (num_layers * num_directions, batch, hidden_size)
        # For plain RNN (unidirectional, num_layers=1): (1, batch, hidden_size)
        fc_input = hn.squeeze(0) # if num_layers=1 and unidirectional
        
        out = self.dropout(fc_input)
        out = self.fc0(out)
        out = self.relu_fc(out)
        out = self.fc_final(out)
        return out

# %%
# --- Optuna Objective Function for Plain RNN ---
def objective(trial, fixed_config, data_globals):
    cfg_trial = {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True),
        "rnn_hidden_dim": trial.suggest_categorical("rnn_hidden_dim", [32, 64, 128, 256]),
        "rnn_nonlinearity": trial.suggest_categorical("rnn_nonlinearity", ['tanh', 'relu']),
        "dropout_rate": trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
    }

    print(f"\n--- Trial {trial.number} Starting ---")
    print(f"Hyperparameters for Plain RNN: {cfg_trial}")

    try:
        model = SimpleRNNModel(
            input_dim=data_globals["input_features"],
            hidden_dim=cfg_trial["rnn_hidden_dim"],
            num_layers=fixed_config["num_rnn_layers"], # Fixed to 1 for plain RNN
            num_classes=1,
            nonlinearity=cfg_trial["rnn_nonlinearity"],
            dropout_rate=cfg_trial["dropout_rate"]
        ).to(device)

        if data_globals["num_pos_train"] > 0:
            pos_weight_value = data_globals["num_neg_train"] / data_globals["num_pos_train"]
        else: pos_weight_value = 1.0
        pos_weight = torch.tensor([pos_weight_value], device=device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = optim.Adam(model.parameters(), lr=cfg_trial["learning_rate"], weight_decay=cfg_trial["weight_decay"])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=fixed_config["lr_scheduler_factor"],
            patience=fixed_config["lr_scheduler_patience"], verbose=False )

        num_train_samples = data_globals["X_time_train_full"].shape[0]
        train_batch_size = fixed_config["batch_size"]
        num_train_batches = int(np.ceil(num_train_samples / train_batch_size))
        eval_batch_size = fixed_config["eval_batch_size"]
        num_test_samples = data_globals["X_time_test_full"].shape[0]
        num_eval_batches = int(np.ceil(num_test_samples / eval_batch_size))

        best_val_loss_for_trial = float('inf')
        epochs_no_improve_for_trial = 0
        
        for epoch in range(fixed_config["num_epochs_hpo"]):
            model.train()
            epoch_train_loss = 0.0
            for i in range(num_train_batches):
                start_idx = i * train_batch_size; end_idx = min(num_train_samples, (i + 1) * train_batch_size)
                batch_x_time = data_globals["X_time_train_full"][start_idx:end_idx].to(device)
                batch_seq_len = data_globals["seq_len_train_full"][start_idx:end_idx]
                batch_y_binary = torch.tensor(data_globals["y_binary_train_full"][start_idx:end_idx]).unsqueeze(1).float().to(device)
                optimizer.zero_grad(); outputs = model(batch_x_time, batch_seq_len); loss = criterion(outputs, batch_y_binary)
                loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=fixed_config["gradient_clip_value"]); optimizer.step()
                epoch_train_loss += loss.item()
            avg_epoch_train_loss = epoch_train_loss / num_train_batches if num_train_batches > 0 else 0.0

            model.eval()
            all_val_outputs_logits = []; all_val_y_for_loss = []
            with torch.no_grad():
                for i in range(num_eval_batches):
                    start_idx = i * eval_batch_size; end_idx = min(num_test_samples, (i + 1) * eval_batch_size)
                    batch_x_time_val = data_globals["X_time_test_full"][start_idx:end_idx].to(device)
                    batch_seq_len_val = data_globals["seq_len_test_full"][start_idx:end_idx]
                    batch_y_binary_val_for_loss = torch.tensor(data_globals["y_binary_test_full"][start_idx:end_idx]).unsqueeze(1).float().to(device)
                    outputs_batch_logits = model(batch_x_time_val, batch_seq_len_val)
                    all_val_outputs_logits.append(outputs_batch_logits.cpu()); all_val_y_for_loss.append(batch_y_binary_val_for_loss.cpu())
            
            if not all_val_outputs_logits: current_val_loss = float('inf')
            else:
                val_outputs_logits = torch.cat(all_val_outputs_logits, dim=0); val_y_for_loss = torch.cat(all_val_y_for_loss, dim=0)
                current_val_loss = criterion(val_outputs_logits.to(device), val_y_for_loss.to(device)).item()

            scheduler.step(current_val_loss)
            if current_val_loss < best_val_loss_for_trial:
                best_val_loss_for_trial = current_val_loss; epochs_no_improve_for_trial = 0
            else: epochs_no_improve_for_trial += 1
            if epochs_no_improve_for_trial >= fixed_config["early_stopping_patience"]:
                print(f"Trial {trial.number} early stopping at epoch {epoch+1} (val_loss: {current_val_loss:.4f})")
                break
            trial.report(current_val_loss, epoch)
            if trial.should_prune():
                print(f"Trial {trial.number} pruned at epoch {epoch+1} (val_loss: {current_val_loss:.4f}).")
                raise optuna.exceptions.TrialPruned()
            if (epoch + 1) % 10 == 0 or epoch == fixed_config["num_epochs_hpo"] -1 :
                 print(f"Trial {trial.number}, Epoch [{epoch+1}/{fixed_config['num_epochs_hpo']}], Train Loss: {avg_epoch_train_loss:.4f}, Val Loss: {current_val_loss:.4f}")
        
        if 'val_outputs_logits' in locals() and val_outputs_logits is not None and len(val_outputs_logits) > 0:
            val_probs = torch.sigmoid(val_outputs_logits).numpy().squeeze()
            if val_probs.ndim == 0: val_probs = np.array([val_probs])
            # val_pred_binary = (val_probs > fixed_config["classification_threshold"]) # Not strictly needed for HPO metric
            # current_y_true_for_metric = data_globals["y_binary_test_full"][:len(val_probs)]
            # try:
            #     dic = performance_dict(current_y_true_for_metric, val_pred_binary, val_probs)
            #     val_auprc = dic.get('pr_auc', 0.0)
            # except Exception as e:
            #     print(f"Warning: Could not compute performance_dict for trial {trial.number} final metrics: {e}"); val_auprc = 0.0
            # print(f"Trial {trial.number} finished. Best Val Loss: {best_val_loss_for_trial:.4f}, Final Epoch Val AUPRC: {val_auprc:.4f}")
            print(f"Trial {trial.number} finished. Best Val Loss: {best_val_loss_for_trial:.4f}")
        else:
             print(f"Trial {trial.number} finished. Best Val Loss: {best_val_loss_for_trial:.4f} (No validation outputs for AUPRC).")
        return best_val_loss_for_trial

    except torch.cuda.OutOfMemoryError:
        print(f"Trial {trial.number} failed due to CUDA OutOfMemoryError. Pruning trial.")
        torch.cuda.empty_cache(); raise optuna.exceptions.TrialPruned()
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"Trial {trial.number} failed due to RuntimeError (OOM). Pruning trial. Error: {e}")
            torch.cuda.empty_cache(); raise optuna.exceptions.TrialPruned()
        else: print(f"Trial {trial.number} failed with other RuntimeError: {e}"); import traceback; traceback.print_exc(); raise

# %%
# --- HPO Study Execution ---
if __name__ == "__main__":
    data_globals_for_objective = {
        "input_features": input_features_global,
        "num_pos_train": num_pos_train_full,
        "num_neg_train": num_neg_train_full,
        "X_time_train_full": X_time_train_full,
        "seq_len_train_full": seq_len_train_full,
        "y_binary_train_full": y_binary_train_full,
        "X_time_test_full": X_time_test_full,
        "seq_len_test_full": seq_len_test_full,
        "y_binary_test_full": y_binary_test_full,
    }
    os.makedirs(CONFIG["results_output_file_base"], exist_ok=True)
    os.makedirs(CONFIG["tensorboard_log_dir_base"], exist_ok=True) # Not actively used in objective

    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=max(1, CONFIG["early_stopping_patience"]//2), interval_steps=1)
    study = optuna.create_study(direction="minimize", pruner=pruner)
    objective_with_config = functools.partial(objective, fixed_config=CONFIG, data_globals=data_globals_for_objective)

    print(f"Starting HPO study for PLAIN RNN with {CONFIG['n_hpo_trials']} trials...")
    try:
        study.optimize(objective_with_config, n_trials=CONFIG['n_hpo_trials'])
    except KeyboardInterrupt: print("HPO study interrupted by user.")
    except Exception as e: print(f"An error occurred during HPO study: {e}"); import traceback; traceback.print_exc()

    print("\n--- HPO Study Finished ---")
    print(f"Number of finished trials: {len(study.trials)}")
    if len(study.trials) > 0 and hasattr(study, 'best_trial') and study.best_trial is not None:
        best_trial = study.best_trial
        print("\nBest trial for Plain RNN:")
        print(f"  Value (Best Validation Loss): {best_trial.value:.4f}")
        print("  Params: "); [print(f"    {key}: {value}") for key, value in best_trial.params.items()]
        best_params_path = os.path.join(CONFIG["results_output_file_base"], "best_plain_rnn_hpo_params.json")
        import json
        with open(best_params_path, 'w') as f: json.dump(best_trial.params, f, indent=4)
        print(f"Best hyperparameters for Plain RNN saved to {best_params_path}")
    else: print("No successful trials completed or no best trial found.")
    print("\nTo train a final Plain RNN model with these best hyperparameters, update your CONFIG dict,")
    print("set num_epochs to a higher value, and run a standard (non-HPO) training script.")
    study_results_df = study.trials_dataframe()
    study_results_path = os.path.join(CONFIG["results_output_file_base"], "hpo_plain_rnn_study_results.csv")
    study_results_df.to_csv(study_results_path, index=False)
    print(f"Full HPO study results for Plain RNN saved to {study_results_path}")

print("Script finished.")

Using device: cuda
Original train size: 43271
Test size: 10818
Training set: Positive AKI cases: 2595.0, Negative AKI cases: 40676.0


[I 2025-05-25 06:26:05,934] A new study created in memory with name: no-name-4c723857-9f52-482d-8cd7-4cf9f6aabeab


Number of input time-series features: 24
Starting HPO study for PLAIN RNN with 50 trials...

--- Trial 0 Starting ---
Hyperparameters for Plain RNN: {'learning_rate': 0.002421041410424487, 'rnn_hidden_dim': 256, 'rnn_nonlinearity': 'tanh', 'dropout_rate': 0.30000000000000004, 'weight_decay': 4.217825903566072e-05}


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Trial 0, Epoch [10/50], Train Loss: 1.1888, Val Loss: 1.2041
Trial 0, Epoch [20/50], Train Loss: 1.1710, Val Loss: 1.2000
Trial 0, Epoch [30/50], Train Loss: 1.1656, Val Loss: 1.1999


...

Trial 47 pruned at epoch 6 (val_loss: 1.2743).

--- Trial 48 Starting ---
Hyperparameters for Plain RNN: {'learning_rate': 9.362910999601996e-05, 'rnn_hidden_dim': 256, 'rnn_nonlinearity': 'tanh', 'dropout_rate': 0.30000000000000004, 'weight_decay': 1.9761763013503094e-05}


[I 2025-05-25 06:41:08,975] Trial 48 pruned. 


Trial 48 pruned at epoch 6 (val_loss: 1.1954).

--- Trial 49 Starting ---
Hyperparameters for Plain RNN: {'learning_rate': 0.0011079265531979306, 'rnn_hidden_dim': 64, 'rnn_nonlinearity': 'relu', 'dropout_rate': 0.4, 'weight_decay': 4.27988086539259e-06}


[I 2025-05-25 06:41:16,047] Trial 49 pruned. 


Trial 49 pruned at epoch 6 (val_loss: 1.1762).

--- HPO Study Finished ---
Number of finished trials: 50

Best trial for Plain RNN:
  Value (Best Validation Loss): 1.1361
  Params: 
    learning_rate: 0.0002095430664254445
    rnn_hidden_dim: 128
    rnn_nonlinearity: relu
    dropout_rate: 0.5
    weight_decay: 4.212169152071669e-06
Best hyperparameters for Plain RNN saved to /home/server/Projects/data/AKI/results/hpo_plain_rnn/best_plain_rnn_hpo_params.json

To train a final Plain RNN model with these best hyperparameters, update your CONFIG dict,
set num_epochs to a higher value, and run a standard (non-HPO) training script.
Full HPO study results for Plain RNN saved to /home/server/Projects/data/AKI/results/hpo_plain_rnn/hpo_plain_rnn_study_results.csv
Script finished.


# LSTM

Working

Epoch [26/100], Train Loss: 0.7107, Val Loss: 1.6407, Val AUC: 0.7141, Val AUPRC: 0.2078
Validation loss did not improve for 15 epoch(s).
Early stopping triggered after 26 epochs.
Training finished.
Loading best model from /home/server/Projects/data/AKI/models/best_base_lstm_model.pth for final evaluation.

Starting final evaluation with the best/last model...
                                                                         

--- Final Test Performance (Best/Last Model) ---
Precision: 0.0822
Sensitivity: 0.8675
Accuracy: 0.4108
rc_auc: 0.7409
pr_auc: 0.2119
Specificity: 0.3817
Negative Predictive Value: 0.9783
F1 Score: 0.1501
Results saved to /home/server/Projects/data/AKI/results/base_lstm_results.pkl
Script finished.

In [1]:
# %%
import os
import numpy as np
import torch
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence # pad_packed_sequence might not be needed for base LSTM if only using hn
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
import torch.optim as optim

import random
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_curve, auc, precision_recall_curve, average_precision_score)
from sklearn.model_selection import train_test_split
import pandas as pd
# Ensure utils.py is in the same directory or Python path
from utils import performance_dict, sigmoid
from tqdm import tqdm

# --- Configuration for the Base LSTM Model ---
CONFIG = {
    "data_file": '/home/server/Projects/data/AKI/lstm_trainable.pkl', # !!! UPDATE THIS PATH !!!
    "results_output_file": '/home/server/Projects/data/AKI/results/base_lstm_results.pkl',
    "tensorboard_log_dir_base": '/home/server/Projects/data/AKI/runs/base_lstm/',
    "best_model_save_path": '/home/server/Projects/data/AKI/models/best_base_lstm_model.pth',
    "test_split_size": 0.2,
    "random_seed": 42,
    "num_epochs": 100,
    "learning_rate": 0.00178, # Starting with the lowered LR
    "weight_decay": 5e-05,
    "gradient_clip_value": 1.0,
    "lr_scheduler_patience": 7,
    "lr_scheduler_factor": 0.1,
    "early_stopping_patience": 15,
    
    # --- Architecture Specifics for Base LSTM ---
    "lstm_hidden_dim": 128,     # For a unidirectional LSTM, might use more hidden units than one direction of BiLSTM
                                # Or keep it at 64 for a truly simpler base model. Let's start with 128.
    "num_lstm_layers": 1,       # Single Layer
    "bidirectional": False,     # Unidirectional
    "use_attention": False,     # No Attention
    "use_se_block": False,      # No SE Block
    "dropout_rate": 0.4,        # Dropout after LSTM before FC
    
    "batch_size": 2000,          # As per your decision
    "eval_batch_size": 512,     # Using the same for evaluation, monitor VRAM. Can be reduced to 512 if needed.
    "classification_threshold": 0.3, # Review this after training
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load data (once)
try:
    df_full = pd.read_pickle(CONFIG["data_file"])
except FileNotFoundError:
    print(f"ERROR: Data file not found at {CONFIG['data_file']}")
    print("Please ensure the path in CONFIG['data_file'] is correct.")
    exit()
except Exception as e:
    print(f"Error loading data file {CONFIG['data_file']}: {e}")
    exit()


# %%
# --- Data Preparation ---
def df_to_tensors(df, all_columns):
    non_tabular_cols = ['op_id', 'time_tensors', 'seq_len', 'aki', 'aki_boolean']
    tabular_feature_cols = [col for col in all_columns if col not in non_tabular_cols]
    if tabular_feature_cols and not df[tabular_feature_cols].empty:
        X_tab = torch.tensor(df[tabular_feature_cols].values).float()
    else:
        X_tab = torch.empty((len(df), 0)).float()
    X_time = torch.stack(df['time_tensors'].tolist()).float()
    y = torch.tensor(df['aki'].values).unsqueeze(1).float()
    y_binary = df['aki_boolean'].values.astype(bool)
    sequence_lengths = torch.tensor(df['seq_len'].tolist()).long()
    return X_tab, X_time, sequence_lengths, y, y_binary

def prepare_data(full_df, test_size, random_seed):
    df_train, df_test = train_test_split(full_df, test_size=test_size, random_state=random_seed, stratify=full_df['aki_boolean'])
    print(f"Original train size: {len(df_train)}")
    print(f"Test size: {len(df_test)}")
    all_columns = full_df.columns.tolist()
    num_positive_aki_train = df_train['aki_boolean'].sum()
    num_negative_aki_train = len(df_train) - num_positive_aki_train
    print(f"Training set: Positive AKI cases: {num_positive_aki_train}, Negative AKI cases: {num_negative_aki_train}")
    X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train = df_to_tensors(df_train, all_columns)
    X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test = df_to_tensors(df_test, all_columns)
    return (X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train,
            X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test,
            num_positive_aki_train, num_negative_aki_train)

(X_tab_train_full, X_time_train_full, seq_len_train_full, y_train_full, y_binary_train_full,
 X_tab_test_full, X_time_test_full, seq_len_test_full, y_test_full, y_binary_test_full,
 num_pos_train_full, num_neg_train_full) = prepare_data(
    df_full, CONFIG["test_split_size"], CONFIG["random_seed"]
)
lstm_input_dim_global = X_time_train_full.shape[2]

# %%
# --- Utility Functions ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["random_seed"])

def save_single_run_results(model_name, metrics_dict, output_file):
    results_to_save = {k: [v] if not isinstance(v, (list, np.ndarray)) else v for k,v in metrics_dict.items()}
    df_result = pd.DataFrame(results_to_save)
    df_result['model_name'] = model_name
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    if os.path.exists(output_file):
        try:
            df_output = pd.read_pickle(output_file)
            df_output = df_output[df_output['model_name'] != model_name] # Remove old result
            df_output = pd.concat([df_output, df_result], ignore_index=True)
        except EOFError: df_output = df_result
    else: df_output = df_result
    df_output.to_pickle(output_file)
    print(f"Results saved to {output_file}")

# %%
# --- Model Definition for Base LSTM ---
class BaseLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes, dropout_rate=0.2):
        super(BaseLSTM, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers # Will be 1 for this base model
        self.num_classes = num_classes
        
        self.lstm = nn.LSTM(input_size=input_dim, 
                            hidden_size=hidden_dim,
                            num_layers=num_layers,
                            batch_first=True,
                            bidirectional=False) # Unidirectional

        # Classifier part
        self.dropout = nn.Dropout(dropout_rate)
        self.fc_intermediate_dim = max(1, hidden_dim // 2)
        self.fc0 = nn.Linear(hidden_dim, self.fc_intermediate_dim) # Input is just hidden_dim (not doubled)
        self.relu_fc = nn.ReLU()
        self.fc_final = nn.Linear(self.fc_intermediate_dim, num_classes)

    def forward(self, x_time, seq_lens):
        packed_input = pack_padded_sequence(x_time, seq_lens.cpu(), batch_first=True, enforce_sorted=False)
        # packed_output contains all hidden states for each t
        # hn is the final hidden state: (num_layers * num_directions, batch, hidden_size)
        # For unidirectional, num_directions = 1
        _, (hn, _) = self.lstm(packed_input) 
        
        # Get the hidden state of the last layer
        # hn is (num_layers, batch, hidden_size) for unidirectional
        # If num_layers is 1, hn[0] or hn.squeeze(0) or hn[-1,:,:] all work
        fc_input = hn[-1,:,:] # Takes the hidden state from the last (and only) layer
        
        out = self.dropout(fc_input) # Apply dropout to the LSTM output features
        out = self.fc0(out)
        out = self.relu_fc(out)
        # No dropout typically right before the very last layer, but can be added if desired
        out = self.fc_final(out)
        return out

# %%
# --- Main Training and Evaluation ---
model = BaseLSTM( # Using the new BaseLSTM class
    input_dim=lstm_input_dim_global,
    hidden_dim=CONFIG["lstm_hidden_dim"],
    num_layers=CONFIG["num_lstm_layers"], # This will be 1 based on CONFIG
    num_classes=1,
    dropout_rate=CONFIG["dropout_rate"]
).to(device)

print(model)
print(f"Model Parameter Count: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

if num_pos_train_full > 0:
    pos_weight_value = num_neg_train_full / num_pos_train_full
else:
    pos_weight_value = 1.0; print("Warning: No positive samples in train for pos_weight.")
pos_weight = torch.tensor([pos_weight_value], device=device)
print(f"Calculated pos_weight for BCEWithLogitsLoss: {pos_weight.item():.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=CONFIG["lr_scheduler_factor"],
    patience=CONFIG["lr_scheduler_patience"])

run_name = f"BaseLSTM_h{CONFIG['lstm_hidden_dim']}_l{CONFIG['num_lstm_layers']}_classWeighted"
tensorboard_run_path = os.path.join(CONFIG["tensorboard_log_dir_base"], run_name)
os.makedirs(tensorboard_run_path, exist_ok=True)
writer = SummaryWriter(tensorboard_run_path)

num_train_samples = X_time_train_full.shape[0]
train_batch_size = CONFIG["batch_size"]
num_train_batches = int(np.ceil(num_train_samples / train_batch_size))
eval_batch_size = CONFIG["eval_batch_size"]
num_test_samples = X_time_test_full.shape[0]
num_eval_batches = int(np.ceil(num_test_samples / eval_batch_size))

best_val_loss = float('inf')
epochs_no_improve = 0
os.makedirs(os.path.dirname(CONFIG["best_model_save_path"]), exist_ok=True)

print(f"Starting training for {CONFIG['num_epochs']} epochs on {num_train_samples} original samples...")
for epoch in tqdm(range(CONFIG["num_epochs"]), desc="Epochs"):
    model.train()
    epoch_train_loss = 0.0
    for i in tqdm(range(num_train_batches), desc="Training Batches", leave=False):
        start_idx = i * train_batch_size
        end_idx = min(num_train_samples, (i + 1) * train_batch_size)
        batch_x_time = X_time_train_full[start_idx:end_idx].to(device)
        batch_seq_len = seq_len_train_full[start_idx:end_idx]
        batch_y_binary = torch.tensor(y_binary_train_full[start_idx:end_idx]).unsqueeze(1).float().to(device)
        optimizer.zero_grad()
        outputs = model(batch_x_time, batch_seq_len)
        loss = criterion(outputs, batch_y_binary)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CONFIG["gradient_clip_value"])
        optimizer.step()
        epoch_train_loss += loss.item()
    avg_epoch_train_loss = epoch_train_loss / num_train_batches
    writer.add_scalar('Loss/train', avg_epoch_train_loss, epoch)

    model.eval()
    all_val_outputs_logits = []; all_val_y_for_loss = []
    with torch.no_grad():
        for i in tqdm(range(num_eval_batches), desc="Validation Batches", leave=False):
            start_idx = i * eval_batch_size; end_idx = min(num_test_samples, (i + 1) * eval_batch_size)
            batch_x_time_val = X_time_test_full[start_idx:end_idx].to(device)
            batch_seq_len_val = seq_len_test_full[start_idx:end_idx]
            batch_y_binary_val_for_loss = torch.tensor(y_binary_test_full[start_idx:end_idx]).unsqueeze(1).float().to(device)
            outputs_batch_logits = model(batch_x_time_val, batch_seq_len_val)
            all_val_outputs_logits.append(outputs_batch_logits.cpu()); all_val_y_for_loss.append(batch_y_binary_val_for_loss.cpu())
    val_outputs_logits = torch.cat(all_val_outputs_logits, dim=0); val_y_for_loss = torch.cat(all_val_y_for_loss, dim=0)
    current_val_loss = criterion(val_outputs_logits.to(device), val_y_for_loss.to(device)).item()
    writer.add_scalar('Loss/val', current_val_loss, epoch)
    val_probs = torch.sigmoid(val_outputs_logits).numpy().squeeze()
    if val_probs.ndim == 0: val_probs = np.array([val_probs])
    val_pred_binary = (val_probs > CONFIG["classification_threshold"])
    current_y_true_for_metric = y_binary_test_full[:len(val_pred_binary)]
    
    try:
        dic = performance_dict(current_y_true_for_metric, val_pred_binary, val_probs)
        writer.add_scalar('Val/AUC', dic.get('rc_auc', np.nan), epoch)
        writer.add_scalar('Val/AUPRC', dic.get('pr_auc', np.nan), epoch)
        writer.add_scalar('Val/Accuracy', dic.get('Accuracy', np.nan), epoch)
        writer.add_scalar('Val/Sensitivity', dic.get('Sensitivity', np.nan), epoch)
        writer.add_scalar('Val/Specificity', dic.get('Specificity', np.nan), epoch)
        writer.add_scalar('Val/F1', dic.get('F1 Score', np.nan), epoch)
        print(f"Epoch [{epoch+1}/{CONFIG['num_epochs']}], Train Loss: {avg_epoch_train_loss:.4f}, Val Loss: {current_val_loss:.4f}, Val AUC: {dic.get('rc_auc', np.nan):.4f}, Val AUPRC: {dic.get('pr_auc', np.nan):.4f}")
    except Exception as e:
        print(f"Error calculating performance_dict in validation: {e}")
        print(f"Epoch [{epoch+1}/{CONFIG['num_epochs']}], Train Loss: {avg_epoch_train_loss:.4f}, Val Loss: {current_val_loss:.4f}")

    scheduler.step(current_val_loss)
    if current_val_loss < best_val_loss:
        best_val_loss = current_val_loss; epochs_no_improve = 0
        torch.save(model.state_dict(), CONFIG["best_model_save_path"])
        print(f"Validation loss improved. Saved best model to {CONFIG['best_model_save_path']}")
    else:
        epochs_no_improve += 1
        print(f"Validation loss did not improve for {epochs_no_improve} epoch(s).")
    if epochs_no_improve >= CONFIG["early_stopping_patience"]:
        print(f"Early stopping triggered after {epoch+1} epochs.")
        break
writer.close()
print("Training finished.")

if os.path.exists(CONFIG["best_model_save_path"]):
    print(f"Loading best model from {CONFIG['best_model_save_path']} for final evaluation.")
    model.load_state_dict(torch.load(CONFIG["best_model_save_path"]))
else:
    print("No best model found (e.g. early stopping triggered on epoch 0 or training didn't improve). Using last model state.")

model.eval()
final_all_test_outputs_logits = []
print("\nStarting final evaluation with the best/last model...")
with torch.no_grad():
    for i in tqdm(range(num_eval_batches), desc="Final Evaluation Batches", leave=False):
        start_idx = i * eval_batch_size; end_idx = min(num_test_samples, (i + 1) * eval_batch_size)
        batch_x_time_test = X_time_test_full[start_idx:end_idx].to(device)
        batch_seq_len_test = seq_len_test_full[start_idx:end_idx]
        outputs_batch_logits = model(batch_x_time_test, batch_seq_len_test)
        final_all_test_outputs_logits.append(outputs_batch_logits.cpu())
final_test_outputs_logits = torch.cat(final_all_test_outputs_logits, dim=0)
final_test_probs = torch.sigmoid(final_test_outputs_logits).numpy().squeeze()
if final_test_probs.ndim == 0: final_test_probs = np.array([final_test_probs])
final_y_pred_binary = (final_test_probs > CONFIG["classification_threshold"])
current_y_true_for_final_metric = y_binary_test_full[:len(final_y_pred_binary)]

try:
    final_metrics = performance_dict(current_y_true_for_final_metric, final_y_pred_binary, final_test_probs)
    print("\n--- Final Test Performance (Best/Last Model) ---")
    for metric, value in final_metrics.items():
        if isinstance(value, (float, int, np.number)): print(f"{metric}: {value:.4f}")
    save_single_run_results(run_name, final_metrics, CONFIG["results_output_file"])
except Exception as e:
    print(f"Error calculating final performance_dict: {e}")

print("Script finished.")

Using device: cuda
Original train size: 43271
Test size: 10818
Training set: Positive AKI cases: 2595.0, Negative AKI cases: 40676.0
BaseLSTM(
  (lstm): LSTM(24, 128, batch_first=True)
  (dropout): Dropout(p=0.4, inplace=False)
  (fc0): Linear(in_features=128, out_features=64, bias=True)
  (relu_fc): ReLU()
  (fc_final): Linear(in_features=64, out_features=1, bias=True)
)
Model Parameter Count: 87169
Calculated pos_weight for BCEWithLogitsLoss: 15.67


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Starting training for 100 epochs on 43271 original samples...


Epochs:   1%|          | 1/100 [00:02<04:37,  2.80s/it]

Epoch [1/100], Train Loss: 1.2458, Val Loss: 1.2211, Val AUC: 0.6697, Val AUPRC: 0.1614
Validation loss improved. Saved best model to /home/server/Projects/data/AKI/models/best_base_lstm_model.pth


...

Epochs:  25%|██▌       | 25/100 [00:56<02:47,  2.23s/it]

Epoch [25/100], Train Loss: 0.7276, Val Loss: 1.6059, Val AUC: 0.7153, Val AUPRC: 0.2079
Validation loss did not improve for 14 epoch(s).


Epochs:  25%|██▌       | 25/100 [00:58<02:55,  2.34s/it]
/tmp/ipykernel_144335/2797236322.py:281: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.l

Epoch [26/100], Train Loss: 0.7107, Val Loss: 1.6407, Val AUC: 0.7141, Val AUPRC: 0.2078
Validation loss did not improve for 15 epoch(s).
Early stopping triggered after 26 epochs.
Training finished.
Loading best model from /home/server/Projects/data/AKI/models/best_base_lstm_model.pth for final evaluation.

Starting final evaluation with the best/last model...



--- Final Test Performance (Best/Last Model) ---
Precision: 0.0822
Sensitivity: 0.8675
Accuracy: 0.4108
rc_auc: 0.7409
pr_auc: 0.2119
Specificity: 0.3817
Negative Predictive Value: 0.9783
F1 Score: 0.1501
Results saved to /home/server/Projects/data/AKI/results/base_lstm_results.pkl
Script finished.


# BiLSTM

Working

Epoch [47/100], Train Loss: 1.0104, Val Loss: 1.1501, Val AUC: 0.7381, Val AUPRC: 0.1961
Validation loss did not improve for 15 epoch(s).
Early stopping triggered after 47 epochs.
Training finished.
Loading best model from /home/server/Projects/data/AKI/models/best_bilstm_only_model.pth for final evaluation.

Starting final evaluation with the best/last model...
                                                                         

--- Final Test Performance (Best/Last Model) ---
Precision: 0.0830
Sensitivity: 0.8737
Accuracy: 0.4132
rc_auc: 0.7377
pr_auc: 0.1967
Specificity: 0.3838
Negative Predictive Value: 0.9794
F1 Score: 0.1516
Results saved to /home/server/Projects/data/AKI/results/bilstm_only_results.pkl
Script finished.

In [3]:
# %%
import os
import numpy as np
import torch
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
import torch.optim as optim

import random
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_curve, auc, precision_recall_curve, average_precision_score)
from sklearn.model_selection import train_test_split
import pandas as pd
# Ensure utils.py is in the same directory or Python path
from utils import performance_dict, sigmoid
from tqdm import tqdm

# --- Configuration for the BiLSTM Only Model ---
CONFIG = {
    "data_file": '/home/server/Projects/data/AKI/lstm_trainable.pkl', # !!! UPDATE THIS PATH !!!
    "results_output_file": '/home/server/Projects/data/AKI/results/bilstm_only_results.pkl',
    "tensorboard_log_dir_base": '/home/server/Projects/data/AKI/runs/bilstm_only/',
    "best_model_save_path": '/home/server/Projects/data/AKI/models/best_bilstm_only_model.pth',
    "test_split_size": 0.2,
    "random_seed": 42,
    "num_epochs": 100,
    "learning_rate": 0.0001,
    "weight_decay": 1e-5,
    "gradient_clip_value": 1.0,
    "lr_scheduler_patience": 7,
    "lr_scheduler_factor": 0.1,
    "early_stopping_patience": 15,
    
    # --- Architecture Specifics for BiLSTM Only ---
    "lstm_hidden_dim": 64,      # Hidden dim for each LSTM direction
    "num_lstm_layers": 1,       # Single Layer BiLSTM
    "bidirectional": True,      # BiLSTM
    "use_attention": False,     # <<< --- NO Attention --- >>>
    "num_attention_heads": 4,   # Not used, but kept for consistency if EnhancedLSTM expects it
    "use_se_block": False,      # <<< --- NO SE Block --- >>>
    "se_reduction_ratio": 16,   # Not used
    "dropout_rate": 0.3,
    
    "batch_size": 640,         
    "eval_batch_size": 640,     
    "classification_threshold": 0.3, 
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load data (once)
try:
    df_full = pd.read_pickle(CONFIG["data_file"])
except FileNotFoundError:
    print(f"ERROR: Data file not found at {CONFIG['data_file']}")
    print("Please ensure the path in CONFIG['data_file'] is correct.")
    exit()
except Exception as e:
    print(f"Error loading data file {CONFIG['data_file']}: {e}")
    exit()


# %%
# --- Data Preparation ---
def df_to_tensors(df, all_columns):
    non_tabular_cols = ['op_id', 'time_tensors', 'seq_len', 'aki', 'aki_boolean']
    tabular_feature_cols = [col for col in all_columns if col not in non_tabular_cols]
    if tabular_feature_cols and not df[tabular_feature_cols].empty:
        X_tab = torch.tensor(df[tabular_feature_cols].values).float()
    else:
        X_tab = torch.empty((len(df), 0)).float()
    X_time = torch.stack(df['time_tensors'].tolist()).float()
    y = torch.tensor(df['aki'].values).unsqueeze(1).float()
    y_binary = df['aki_boolean'].values.astype(bool)
    sequence_lengths = torch.tensor(df['seq_len'].tolist()).long()
    return X_tab, X_time, sequence_lengths, y, y_binary

def prepare_data(full_df, test_size, random_seed):
    df_train, df_test = train_test_split(full_df, test_size=test_size, random_state=random_seed, stratify=full_df['aki_boolean'])
    print(f"Original train size: {len(df_train)}")
    print(f"Test size: {len(df_test)}")
    all_columns = full_df.columns.tolist()
    num_positive_aki_train = df_train['aki_boolean'].sum()
    num_negative_aki_train = len(df_train) - num_positive_aki_train
    print(f"Training set: Positive AKI cases: {num_positive_aki_train}, Negative AKI cases: {num_negative_aki_train}")
    X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train = df_to_tensors(df_train, all_columns)
    X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test = df_to_tensors(df_test, all_columns)
    return (X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train,
            X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test,
            num_positive_aki_train, num_negative_aki_train)

(X_tab_train_full, X_time_train_full, seq_len_train_full, y_train_full, y_binary_train_full,
 X_tab_test_full, X_time_test_full, seq_len_test_full, y_test_full, y_binary_test_full,
 num_pos_train_full, num_neg_train_full) = prepare_data(
    df_full, CONFIG["test_split_size"], CONFIG["random_seed"]
)
lstm_input_dim_global = X_time_train_full.shape[2]

# %%
# --- Utility Functions ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["random_seed"])

def save_single_run_results(model_name, metrics_dict, output_file):
    results_to_save = {k: [v] if not isinstance(v, (list, np.ndarray)) else v for k,v in metrics_dict.items()}
    df_result = pd.DataFrame(results_to_save)
    df_result['model_name'] = model_name
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    if os.path.exists(output_file):
        try:
            df_output = pd.read_pickle(output_file)
            df_output = df_output[df_output['model_name'] != model_name]
            df_output = pd.concat([df_output, df_result], ignore_index=True)
        except EOFError: df_output = df_result
    else: df_output = df_result
    df_output.to_pickle(output_file)
    print(f"Results saved to {output_file}")

# %%
# --- Model Definition (Using EnhancedLSTM with flags for simplicity) ---
# SequenceSEBlock and Attention classes are defined but won't be used if flags are False
class SequenceSEBlock(nn.Module):
    def __init__(self, channels, reduction_ratio=16):
        super(SequenceSEBlock, self).__init__()
        self.channels = channels; self.reduction_ratio = reduction_ratio
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, max(1, channels // reduction_ratio), bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(max(1, channels // reduction_ratio), channels, bias=False),
            nn.Sigmoid())
    def forward(self, x): # x shape: (batch, seq_len, channels)
        b, s, c = x.shape; y = x.permute(0, 2, 1); y = self.avg_pool(y); y = y.squeeze(-1)
        y = self.fc(y); return x * y.unsqueeze(1)

class EnhancedLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes,
                 bidirectional=True, use_attention=True, num_attention_heads=4, # num_attention_heads needed for MHA
                 dropout_rate=0.2, use_se_block=False, se_reduction_ratio=16):
        super(EnhancedLSTM, self).__init__()
        self.input_dim = input_dim; self.hidden_dim = hidden_dim; self.num_layers = num_layers
        self.num_classes = num_classes; self.bidirectional = bidirectional
        self.use_attention = use_attention; self.use_se_block = use_se_block

        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=bidirectional,
                            dropout=dropout_rate if num_layers > 1 else 0)
        
        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        
        self.se_block_instance = None
        if self.use_se_block:
            self.se_block_instance = SequenceSEBlock(channels=lstm_output_dim, reduction_ratio=se_reduction_ratio)
        
        self.attention_instance = None
        if self.use_attention:
            if lstm_output_dim % num_attention_heads != 0:
                # This check might be too strict if num_attention_heads is from CONFIG but not used by a simpler attention
                print(f"Warning: lstm_output_dim ({lstm_output_dim}) may not be divisible by num_attention_heads ({num_attention_heads}) for MultiHeadAttention.")
                # For a simple BiLSTM only, this part is skipped. If you had a custom single-head attention, it would be here.
                # For nn.MultiheadAttention, it would be instantiated here.
                # Since CONFIG["use_attention"] = False for BiLSTM only, this path is not taken.
            self.attention_instance = nn.MultiheadAttention( # This will NOT be created if use_attention=False
                embed_dim=lstm_output_dim, 
                num_heads=num_attention_heads, 
                dropout=dropout_rate,
                batch_first=True 
            )
            fc_input_dim = lstm_output_dim
        else:
            fc_input_dim = lstm_output_dim # This will be 2 * hidden_dim if bidirectional
            
        self.fc_intermediate_dim = max(1, fc_input_dim // 2)
        self.fc0 = nn.Linear(fc_input_dim, self.fc_intermediate_dim)
        self.relu_fc = nn.ReLU(); self.dropout_fc = nn.Dropout(dropout_rate)
        self.fc_final = nn.Linear(self.fc_intermediate_dim, num_classes)

    def forward(self, x_time, seq_lens):
        packed_input = pack_padded_sequence(x_time, seq_lens.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, (hn, cn) = self.lstm(packed_input)
        
        # For BiLSTM only (no attention, no SE on sequence output for this specific use case)
        # We use the final hidden states 'hn'
        if self.use_attention and self.attention_instance is not None: # This block will be SKIPPED
            lstm_sequence_output, _ = pad_packed_sequence(packed_output, batch_first=True)
            processed_sequence_output = lstm_sequence_output
            if self.use_se_block and self.se_block_instance is not None:
                processed_sequence_output = self.se_block_instance(processed_sequence_output)
            
            max_len = processed_sequence_output.size(1)
            idx = torch.arange(max_len, device=seq_lens.device).unsqueeze(0)
            key_padding_mask = (idx >= seq_lens.cpu().unsqueeze(1)).to(device)
            
            attn_output, _ = self.attention_instance(
                query=processed_sequence_output, key=processed_sequence_output, value=processed_sequence_output,
                key_padding_mask=key_padding_mask)
            fc_input = attn_output.mean(dim=1)
        else: # This block WILL BE USED for BiLSTM only
            if self.bidirectional:
                # Concatenate the final forward (hn[-2]) and backward (hn[-1]) hidden states from the last layer
                # hn shape is (num_layers * num_directions, batch, hidden_size)
                # For num_layers=1, BiLSTM: hn is (2, batch, hidden_size). hn[0] is fwd, hn[1] is bwd.
                # Or, more generally, for multi-layer BiLSTM, last layer's fwd/bwd.
                if self.num_layers == 1:
                    fc_input = torch.cat((hn[0,:,:], hn[1,:,:]), dim=1)
                else: #Stacked BiLSTM
                    fc_input = torch.cat((hn[-2,:,:], hn[-1,:,:]), dim=1)

            else: # Unidirectional LSTM (not hit if bidirectional=True)
                fc_input = hn[-1,:,:]
                
        out = self.fc0(fc_input); out = self.relu_fc(out); out = self.dropout_fc(out)
        out = self.fc_final(out)
        return out

# %%
# --- Main Training and Evaluation ---
model = EnhancedLSTM( # Using EnhancedLSTM configured as a BiLSTM
    input_dim=lstm_input_dim_global,
    hidden_dim=CONFIG["lstm_hidden_dim"],
    num_layers=CONFIG["num_lstm_layers"],
    num_classes=1,
    bidirectional=CONFIG["bidirectional"],
    use_attention=CONFIG["use_attention"], # This will be False
    num_attention_heads=CONFIG["num_attention_heads"], # Not used by model logic if use_attention=False
    dropout_rate=CONFIG["dropout_rate"],
    use_se_block=CONFIG["use_se_block"],   # This will be False
    se_reduction_ratio=CONFIG["se_reduction_ratio"] # Not used
).to(device)

print(model)
print(f"Model Parameter Count: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

if num_pos_train_full > 0:
    pos_weight_value = num_neg_train_full / num_pos_train_full
else:
    pos_weight_value = 1.0; print("Warning: No positive samples in train for pos_weight.")
pos_weight = torch.tensor([pos_weight_value], device=device)
print(f"Calculated pos_weight for BCEWithLogitsLoss: {pos_weight.item():.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=CONFIG["lr_scheduler_factor"],
    patience=CONFIG["lr_scheduler_patience"])

run_name = f"BiLSTMonly_h{CONFIG['lstm_hidden_dim']}_l{CONFIG['num_lstm_layers']}_classWeighted"
tensorboard_run_path = os.path.join(CONFIG["tensorboard_log_dir_base"], run_name)
os.makedirs(tensorboard_run_path, exist_ok=True)
writer = SummaryWriter(tensorboard_run_path)

num_train_samples = X_time_train_full.shape[0]
train_batch_size = CONFIG["batch_size"]
num_train_batches = int(np.ceil(num_train_samples / train_batch_size))
eval_batch_size = CONFIG["eval_batch_size"]
num_test_samples = X_time_test_full.shape[0]
num_eval_batches = int(np.ceil(num_test_samples / eval_batch_size))

best_val_loss = float('inf')
epochs_no_improve = 0
os.makedirs(os.path.dirname(CONFIG["best_model_save_path"]), exist_ok=True)

print(f"Starting training for {CONFIG['num_epochs']} epochs on {num_train_samples} original samples...")
for epoch in tqdm(range(CONFIG["num_epochs"]), desc="Epochs"):
    model.train()
    epoch_train_loss = 0.0
    for i in tqdm(range(num_train_batches), desc="Training Batches", leave=False):
        start_idx = i * train_batch_size
        end_idx = min(num_train_samples, (i + 1) * train_batch_size)
        batch_x_time = X_time_train_full[start_idx:end_idx].to(device)
        batch_seq_len = seq_len_train_full[start_idx:end_idx]
        batch_y_binary = torch.tensor(y_binary_train_full[start_idx:end_idx]).unsqueeze(1).float().to(device)
        optimizer.zero_grad()
        outputs = model(batch_x_time, batch_seq_len)
        loss = criterion(outputs, batch_y_binary)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CONFIG["gradient_clip_value"])
        optimizer.step()
        epoch_train_loss += loss.item()
    avg_epoch_train_loss = epoch_train_loss / num_train_batches
    writer.add_scalar('Loss/train', avg_epoch_train_loss, epoch)

    model.eval()
    all_val_outputs_logits = []; all_val_y_for_loss = []
    with torch.no_grad():
        for i in tqdm(range(num_eval_batches), desc="Validation Batches", leave=False):
            start_idx = i * eval_batch_size; end_idx = min(num_test_samples, (i + 1) * eval_batch_size)
            batch_x_time_val = X_time_test_full[start_idx:end_idx].to(device)
            batch_seq_len_val = seq_len_test_full[start_idx:end_idx]
            batch_y_binary_val_for_loss = torch.tensor(y_binary_test_full[start_idx:end_idx]).unsqueeze(1).float().to(device)
            outputs_batch_logits = model(batch_x_time_val, batch_seq_len_val)
            all_val_outputs_logits.append(outputs_batch_logits.cpu()); all_val_y_for_loss.append(batch_y_binary_val_for_loss.cpu())
    val_outputs_logits = torch.cat(all_val_outputs_logits, dim=0); val_y_for_loss = torch.cat(all_val_y_for_loss, dim=0)
    current_val_loss = criterion(val_outputs_logits.to(device), val_y_for_loss.to(device)).item()
    writer.add_scalar('Loss/val', current_val_loss, epoch)
    val_probs = torch.sigmoid(val_outputs_logits).numpy().squeeze()
    if val_probs.ndim == 0: val_probs = np.array([val_probs])
    val_pred_binary = (val_probs > CONFIG["classification_threshold"])
    current_y_true_for_metric = y_binary_test_full[:len(val_pred_binary)]
    
    try:
        dic = performance_dict(current_y_true_for_metric, val_pred_binary, val_probs)
        writer.add_scalar('Val/AUC', dic.get('rc_auc', np.nan), epoch)
        writer.add_scalar('Val/AUPRC', dic.get('pr_auc', np.nan), epoch)
        writer.add_scalar('Val/Accuracy', dic.get('Accuracy', np.nan), epoch)
        # ... (log other metrics from dic)
        print(f"Epoch [{epoch+1}/{CONFIG['num_epochs']}], Train Loss: {avg_epoch_train_loss:.4f}, Val Loss: {current_val_loss:.4f}, Val AUC: {dic.get('rc_auc', np.nan):.4f}, Val AUPRC: {dic.get('pr_auc', np.nan):.4f}")
    except Exception as e:
        print(f"Error calculating performance_dict in validation: {e}")
        print(f"Epoch [{epoch+1}/{CONFIG['num_epochs']}], Train Loss: {avg_epoch_train_loss:.4f}, Val Loss: {current_val_loss:.4f}")

    scheduler.step(current_val_loss)
    if current_val_loss < best_val_loss:
        best_val_loss = current_val_loss; epochs_no_improve = 0
        torch.save(model.state_dict(), CONFIG["best_model_save_path"])
        print(f"Validation loss improved. Saved best model to {CONFIG['best_model_save_path']}")
    else:
        epochs_no_improve += 1
        print(f"Validation loss did not improve for {epochs_no_improve} epoch(s).")
    if epochs_no_improve >= CONFIG["early_stopping_patience"]:
        print(f"Early stopping triggered after {epoch+1} epochs.")
        break
writer.close()
print("Training finished.")

if os.path.exists(CONFIG["best_model_save_path"]):
    print(f"Loading best model from {CONFIG['best_model_save_path']} for final evaluation.")
    model.load_state_dict(torch.load(CONFIG["best_model_save_path"]))
else:
    print("No best model found. Using last model state.")

model.eval()
final_all_test_outputs_logits = []
print("\nStarting final evaluation with the best/last model...")
with torch.no_grad():
    for i in tqdm(range(num_eval_batches), desc="Final Evaluation Batches", leave=False):
        start_idx = i * eval_batch_size; end_idx = min(num_test_samples, (i + 1) * eval_batch_size)
        batch_x_time_test = X_time_test_full[start_idx:end_idx].to(device)
        batch_seq_len_test = seq_len_test_full[start_idx:end_idx]
        outputs_batch_logits = model(batch_x_time_test, batch_seq_len_test)
        final_all_test_outputs_logits.append(outputs_batch_logits.cpu())
final_test_outputs_logits = torch.cat(final_all_test_outputs_logits, dim=0)
final_test_probs = torch.sigmoid(final_test_outputs_logits).numpy().squeeze()
if final_test_probs.ndim == 0: final_test_probs = np.array([final_test_probs])
final_y_pred_binary = (final_test_probs > CONFIG["classification_threshold"])
current_y_true_for_final_metric = y_binary_test_full[:len(final_y_pred_binary)]

try:
    final_metrics = performance_dict(current_y_true_for_final_metric, final_y_pred_binary, final_test_probs)
    print("\n--- Final Test Performance (Best/Last Model) ---")
    for metric, value in final_metrics.items():
        if isinstance(value, (float, int, np.number)): print(f"{metric}: {value:.4f}")
    save_single_run_results(run_name, final_metrics, CONFIG["results_output_file"])
except Exception as e:
    print(f"Error calculating final performance_dict: {e}")

print("Script finished.")

Using device: cuda
Original train size: 43271
Test size: 10818
Training set: Positive AKI cases: 2595.0, Negative AKI cases: 40676.0


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


EnhancedLSTM(
  (lstm): LSTM(24, 64, batch_first=True, bidirectional=True)
  (fc0): Linear(in_features=128, out_features=64, bias=True)
  (relu_fc): ReLU()
  (dropout_fc): Dropout(p=0.3, inplace=False)
  (fc_final): Linear(in_features=64, out_features=1, bias=True)
)
Model Parameter Count: 54401
Calculated pos_weight for BCEWithLogitsLoss: 15.67
Starting training for 100 epochs on 43271 original samples...


Epochs:   0%|          | 0/100 [00:00<?, ?it/s]/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no 

Epoch [1/100], Train Loss: 1.2997, Val Loss: 1.2944, Val AUC: 0.6173, Val AUPRC: 0.1097
Validation loss improved. Saved best model to /home/server/Projects/data/AKI/models/best_bilstm_only_model.pth


...

Epochs:  46%|████▌     | 46/100 [02:33<03:02,  3.37s/it]

Epoch [46/100], Train Loss: 1.0128, Val Loss: 1.1494, Val AUC: 0.7383, Val AUPRC: 0.1963
Validation loss did not improve for 14 epoch(s).


Epochs:  46%|████▌     | 46/100 [02:36<03:03,  3.40s/it]
/tmp/ipykernel_127091/2854436513.py:340: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.l

Epoch [47/100], Train Loss: 1.0104, Val Loss: 1.1501, Val AUC: 0.7381, Val AUPRC: 0.1961
Validation loss did not improve for 15 epoch(s).
Early stopping triggered after 47 epochs.
Training finished.
Loading best model from /home/server/Projects/data/AKI/models/best_bilstm_only_model.pth for final evaluation.

Starting final evaluation with the best/last model...



--- Final Test Performance (Best/Last Model) ---
Precision: 0.0830
Sensitivity: 0.8737
Accuracy: 0.4132
rc_auc: 0.7377
pr_auc: 0.1967
Specificity: 0.3838
Negative Predictive Value: 0.9794
F1 Score: 0.1516
Results saved to /home/server/Projects/data/AKI/results/bilstm_only_results.pkl
Script finished.


# BiLSTM + SAttention
Working

Epoch [65/100], Train Loss: 1.0182, Val Loss: 1.1449, Val AUC: 0.7362, Val AUPRC: 0.2186
Validation loss did not improve for 15 epoch(s).
Early stopping triggered after 65 epochs.
Training finished.
Loading best model from /home/server/Projects/data/AKI/models/best_lstm_v2_model.pth for final evaluation.

Starting final evaluation with the best/last model...
                                                                         

--- Final Test Performance (Best/Last Model) ---
Precision: 0.0813
Sensitivity: 0.8737
Accuracy: 0.4004
rc_auc: 0.7368
pr_auc: 0.2186
Specificity: 0.3701
Negative Predictive Value: 0.9787
F1 Score: 0.1488
Results saved to /home/server/Projects/data/AKI/results/class_weighted_lstm_v2_results.pkl
Script finished.

In [1]:
# %%
import os
import numpy as np
import torch
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
import torch.optim as optim

import random
# import scipy.stats as stats

from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_curve, auc, precision_recall_curve, average_precision_score)
# import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pandas as pd
from utils import performance_dict, sigmoid # Assuming these are your custom functions
from tqdm import tqdm

# --- Configuration ---
CONFIG = {
    "data_file": '/home/server/Projects/data/AKI/lstm_trainable.pkl',
    "results_output_file": '/home/server/Projects/data/AKI/results/class_weighted_lstm_v2_results.pkl', # Updated
    "tensorboard_log_dir_base": '/home/server/Projects/data/AKI/runs/class_weighted_lstm_v2/', # Updated
    "best_model_save_path": '/home/server/Projects/data/AKI/models/best_lstm_v2_model.pth', # Path to save best model
    "test_split_size": 0.2,
    "random_seed": 42,
    "num_epochs": 100, # You can keep this high; early stopping will terminate if needed
    "learning_rate": 0.0001,  # <<--- REDUCED LEARNING RATE
    "weight_decay": 1e-5,     # <<--- ADDED WEIGHT DECAY
    "gradient_clip_value": 1.0, # <<--- ADDED GRADIENT CLIPPING VALUE
    "lr_scheduler_patience": 7, # <<--- LR scheduler: epochs to wait for improvement
    "lr_scheduler_factor": 0.1, # <<--- LR scheduler: factor to reduce LR by
    "early_stopping_patience": 15, # <<--- Early stopping: epochs to wait for improvement
    "lstm_hidden_dim": 64,
    "num_lstm_layers": 1,
    "bidirectional": True,
    "use_attention": True,
    "attention_dim": 32,
    "use_se_block": False,
    "batch_size": 2000,
    "eval_batch_size": 512,
    "classification_threshold": 0.3, # This might need re-evaluation
    "dropout_rate": 0.3, # <<--- SLIGHTLY INCREASED DROPOUT
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load data (once)
try:
    df_full = pd.read_pickle(CONFIG["data_file"])
except FileNotFoundError:
    print(f"ERROR: Data file not found at {CONFIG['data_file']}")
    print("Please ensure the path in CONFIG['data_file'] is correct.")
    exit()


# %%
# --- Data Preparation (remains the same as your class-weighted version) ---

def df_to_tensors(df, all_columns):
    non_tabular_cols = ['op_id', 'time_tensors', 'seq_len', 'aki', 'aki_boolean']
    tabular_feature_cols = [col for col in all_columns if col not in non_tabular_cols]
    
    if tabular_feature_cols and not df[tabular_feature_cols].empty:
        X_tab = torch.tensor(df[tabular_feature_cols].values).float()
    else:
        X_tab = torch.empty((len(df), 0)).float()

    X_time = torch.stack(df['time_tensors'].tolist()).float()
    y = torch.tensor(df['aki'].values).unsqueeze(1).float()
    y_binary = df['aki_boolean'].values.astype(bool)
    sequence_lengths = torch.tensor(df['seq_len'].tolist()).long()
    
    return X_tab, X_time, sequence_lengths, y, y_binary

def prepare_data(full_df, test_size, random_seed):
    df_train, df_test = train_test_split(full_df,
                                         test_size=test_size,
                                         random_state=random_seed,
                                         stratify=full_df['aki_boolean'])
    
    print(f"Original train size: {len(df_train)}")
    print(f"Test size: {len(df_test)}")

    all_columns = full_df.columns.tolist()

    num_positive_aki_train = df_train['aki_boolean'].sum()
    num_negative_aki_train = len(df_train) - num_positive_aki_train
    
    print(f"Training set: Positive AKI cases: {num_positive_aki_train}, Negative AKI cases: {num_negative_aki_train}")

    X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train = df_to_tensors(df_train, all_columns)
    X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test = df_to_tensors(df_test, all_columns)
    
    return (X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train,
            X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test,
            num_positive_aki_train, num_negative_aki_train)

# %%
# --- Utility Functions (set_seed, save_single_run_results remain the same) ---

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["random_seed"])

def save_single_run_results(model_name, metrics_dict, output_file):
    results_to_save = {k: [v] if not isinstance(v, (list, np.ndarray)) else v for k,v in metrics_dict.items()}
    df_result = pd.DataFrame(results_to_save)
    df_result['model_name'] = model_name
    
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    if os.path.exists(output_file):
        try:
            df_output = pd.read_pickle(output_file)
            df_output = df_output[df_output['model_name'] != model_name]
            df_output = pd.concat([df_output, df_result], ignore_index=True)
        except EOFError:
             df_output = df_result
    else:
        df_output = df_result
    df_output.to_pickle(output_file)
    print(f"Results saved to {output_file}")

# %%
# --- Model Definition (EnhancedLSTM and Attention remain the same structurally) ---
# Dropout rate is now passed from CONFIG

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.W = nn.Linear(hidden_dim, hidden_dim)
        self.V = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_outputs):
        e = torch.tanh(self.W(lstm_outputs))
        e = self.V(e).squeeze(2)
        alpha = F.softmax(e, dim=1)
        context = torch.bmm(alpha.unsqueeze(1), lstm_outputs).squeeze(1)
        return context, alpha

class EnhancedLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes,
                 bidirectional=True, use_attention=True, attention_dim=None, dropout_rate=0.2, # dropout_rate from CONFIG
                 use_se_block=False):
        super(EnhancedLSTM, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.num_classes = num_classes
        self.bidirectional = bidirectional
        self.use_attention = use_attention
        self.use_se_block = use_se_block

        self.lstm = nn.LSTM(input_size=input_dim,
                            hidden_size=hidden_dim,
                            num_layers=num_layers,
                            batch_first=True,
                            bidirectional=bidirectional,
                            dropout=dropout_rate if num_layers > 1 else 0) # LSTM internal dropout

        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim

        if self.use_attention:
            self.attention = Attention(lstm_output_dim)
            fc_input_dim = lstm_output_dim
        else:
            fc_input_dim = lstm_output_dim
        
        self.fc_intermediate_dim = max(1, fc_input_dim // 2)
        self.fc0 = nn.Linear(fc_input_dim, self.fc_intermediate_dim)
        self.relu_fc = nn.ReLU()
        self.dropout_fc = nn.Dropout(dropout_rate) # Dropout before final FC layer
        self.fc_final = nn.Linear(self.fc_intermediate_dim, num_classes)
        
    def forward(self, x_time, seq_lens):
        packed_input = pack_padded_sequence(x_time, seq_lens.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, (hn, cn) = self.lstm(packed_input)
        output, _ = pad_packed_sequence(packed_output, batch_first=True)

        if self.use_attention:
            context_vector, attention_weights = self.attention(output)
            fc_input = context_vector
        else:
            if self.bidirectional:
                hn_last_layer_fwd = hn[-2,:,:]
                hn_last_layer_bwd = hn[-1,:,:]
                fc_input = torch.cat((hn_last_layer_fwd, hn_last_layer_bwd), dim=1)
            else:
                fc_input = hn[-1,:,:]
        
        out = self.fc0(fc_input)
        out = self.relu_fc(out)
        out = self.dropout_fc(out)
        out = self.fc_final(out)
        return out

# %%
# --- Main Training and Evaluation ---

(X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train,
 X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test,
 num_pos_train, num_neg_train) = prepare_data(
    df_full, CONFIG["test_split_size"], CONFIG["random_seed"]
)

lstm_input_dim = X_time_train.shape[2] 

model = EnhancedLSTM(
    input_dim=lstm_input_dim,
    hidden_dim=CONFIG["lstm_hidden_dim"],
    num_layers=CONFIG["num_lstm_layers"],
    num_classes=1,
    bidirectional=CONFIG["bidirectional"],
    use_attention=CONFIG["use_attention"],
    attention_dim=CONFIG["attention_dim"],
    dropout_rate=CONFIG["dropout_rate"], # Pass dropout rate from CONFIG
    use_se_block=CONFIG["use_se_block"]
).to(device)

print(model)
print(f"Model Parameter Count: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

# --- Loss and Optimizer ---
if num_pos_train > 0:
    pos_weight_value = num_neg_train / num_pos_train
else:
    pos_weight_value = 1.0 
    print("Warning: No positive samples found in the training set for pos_weight calculation.")
pos_weight = torch.tensor([pos_weight_value], device=device)
print(f"Calculated pos_weight for BCEWithLogitsLoss: {pos_weight.item():.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"]) # Added weight_decay

# <<< --- ADDED: Learning Rate Scheduler --- >>>
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min', # Monitored quantity should be minimized (e.g., validation loss)
    factor=CONFIG["lr_scheduler_factor"],
    patience=CONFIG["lr_scheduler_patience"]
)

# --- TensorBoard ---
run_name = f"LSTM_h{CONFIG['lstm_hidden_dim']}_l{CONFIG['num_lstm_layers']}"
if CONFIG['bidirectional']: run_name += "_bi"
if CONFIG['use_attention']: run_name += "_attn"
run_name += "_classWeighted_v2" # Updated identifier
tensorboard_run_path = os.path.join(CONFIG["tensorboard_log_dir_base"], run_name)
os.makedirs(tensorboard_run_path, exist_ok=True)
writer = SummaryWriter(tensorboard_run_path)

# --- Training Loop ---
num_train_samples = X_time_train.shape[0]
train_batch_size = CONFIG["batch_size"]
num_train_batches = int(np.ceil(num_train_samples / train_batch_size))

eval_batch_size = CONFIG["eval_batch_size"]
num_test_samples = X_time_test.shape[0]
num_eval_batches = int(np.ceil(num_test_samples / eval_batch_size))

# <<< --- ADDED: Early Stopping Initialization --- >>>
best_val_loss = float('inf')
epochs_no_improve = 0
os.makedirs(os.path.dirname(CONFIG["best_model_save_path"]), exist_ok=True) # Ensure directory for best model exists

print(f"Starting training for {CONFIG['num_epochs']} epochs on {num_train_samples} original samples...")
for epoch in tqdm(range(CONFIG["num_epochs"]), desc="Epochs"):
    model.train()
    epoch_loss = 0.0
    for i in tqdm(range(num_train_batches), desc="Training Batches", leave=False):
        start_idx = i * train_batch_size
        end_idx = min(num_train_samples, (i + 1) * train_batch_size)

        batch_x_time = X_time_train[start_idx:end_idx].to(device)
        batch_seq_len = seq_len_train[start_idx:end_idx]
        batch_y_binary = torch.tensor(y_binary_train[start_idx:end_idx]).unsqueeze(1).float().to(device)

        optimizer.zero_grad()
        outputs = model(batch_x_time, batch_seq_len)
        loss = criterion(outputs, batch_y_binary)
        loss.backward()
        
        # <<< --- ADDED: Gradient Clipping --- >>>
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CONFIG["gradient_clip_value"])
        
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_epoch_loss = epoch_loss / num_train_batches
    writer.add_scalar('Loss/train', avg_epoch_loss, epoch)
    
    # --- Batched Validation (every epoch for LR scheduler and Early Stopping) ---
    model.eval()
    all_test_outputs_logits = []
    all_test_y_for_loss = []

    with torch.no_grad():
        for i in tqdm(range(num_eval_batches), desc="Validation Batches", leave=False):
            start_idx = i * eval_batch_size
            end_idx = min(num_test_samples, (i + 1) * eval_batch_size)

            batch_x_time_test = X_time_test[start_idx:end_idx].to(device)
            batch_seq_len_test = seq_len_test[start_idx:end_idx]
            batch_y_binary_test_for_loss = torch.tensor(y_binary_test[start_idx:end_idx]).unsqueeze(1).float().to(device)

            outputs_batch_logits = model(batch_x_time_test, batch_seq_len_test)
            all_test_outputs_logits.append(outputs_batch_logits.cpu())
            all_test_y_for_loss.append(batch_y_binary_test_for_loss.cpu())

    val_outputs_logits = torch.cat(all_test_outputs_logits, dim=0)
    val_y_for_loss = torch.cat(all_test_y_for_loss, dim=0)

    # Calculate overall validation loss (make sure criterion uses pos_weight correctly, which it does as it's part of its init)
    current_val_loss = criterion(val_outputs_logits.to(device), val_y_for_loss.to(device)).item()
    writer.add_scalar('Loss/val', current_val_loss, epoch)
    
    val_probs = torch.sigmoid(val_outputs_logits).numpy().squeeze()
    val_pred_binary = (val_probs > CONFIG["classification_threshold"])
    
    dic = performance_dict(y_binary_test, val_pred_binary, val_probs)

    writer.add_scalar('Val/AUC', dic.get('rc_auc', np.nan), epoch)
    writer.add_scalar('Val/AUPRC', dic.get('pr_auc', np.nan), epoch)
    writer.add_scalar('Val/Accuracy', dic.get('Accuracy', np.nan), epoch)
    # ... (rest of metric logging) ...
    
    print(f"Epoch [{epoch+1}/{CONFIG['num_epochs']}], Train Loss: {avg_epoch_loss:.4f}, Val Loss: {current_val_loss:.4f}, Val AUC: {dic.get('rc_auc', np.nan):.4f}, Val AUPRC: {dic.get('pr_auc', np.nan):.4f}")

    # <<< --- ADDED: Learning Rate Scheduler Step --- >>>
    scheduler.step(current_val_loss)

    # <<< --- ADDED: Early Stopping Logic --- >>>
    if current_val_loss < best_val_loss:
        best_val_loss = current_val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), CONFIG["best_model_save_path"])
        print(f"Validation loss improved. Saved best model to {CONFIG['best_model_save_path']}")
    else:
        epochs_no_improve += 1
        print(f"Validation loss did not improve for {epochs_no_improve} epoch(s).")

    if epochs_no_improve >= CONFIG["early_stopping_patience"]:
        print(f"Early stopping triggered after {epoch+1} epochs.")
        break # Exit training loop

writer.close()
print("Training finished.")

# <<< --- ADDED: Load best model if early stopping occurred --- >>>
if os.path.exists(CONFIG["best_model_save_path"]):
    print(f"Loading best model from {CONFIG['best_model_save_path']} for final evaluation.")
    model.load_state_dict(torch.load(CONFIG["best_model_save_path"]))
else:
    print("No best model found (early stopping might not have triggered improvement). Using last model state.")


# --- Batched Final Evaluation after all epochs (or after loading best model) ---
model.eval()
final_all_test_outputs_logits = []
print("\nStarting final evaluation with the best/last model...")
# ... (The rest of the final evaluation loop remains the same as your previous batched version) ...
with torch.no_grad():
    for i in tqdm(range(num_eval_batches), desc="Final Evaluation Batches", leave=False):
        start_idx = i * eval_batch_size
        end_idx = min(num_test_samples, (i + 1) * eval_batch_size)

        batch_x_time_test = X_time_test[start_idx:end_idx].to(device)
        batch_seq_len_test = seq_len_test[start_idx:end_idx]

        outputs_batch_logits = model(batch_x_time_test, batch_seq_len_test)
        final_all_test_outputs_logits.append(outputs_batch_logits.cpu())

final_test_outputs_logits = torch.cat(final_all_test_outputs_logits, dim=0)
final_test_probs = torch.sigmoid(final_test_outputs_logits).numpy().squeeze()
final_y_pred_binary = (final_test_probs > CONFIG["classification_threshold"])

final_metrics = performance_dict(y_binary_test, final_y_pred_binary, final_test_probs)

print("\n--- Final Test Performance (Best/Last Model) ---")
for metric, value in final_metrics.items():
    if isinstance(value, (float, int, np.number)):
        print(f"{metric}: {value:.4f}")

save_single_run_results(run_name, final_metrics, CONFIG["results_output_file"])
print("Script finished.")

Using device: cuda
Original train size: 43271
Test size: 10818
Training set: Positive AKI cases: 2595.0, Negative AKI cases: 40676.0
EnhancedLSTM(
  (lstm): LSTM(24, 64, batch_first=True, bidirectional=True)
  (attention): Attention(
    (W): Linear(in_features=128, out_features=128, bias=True)
    (V): Linear(in_features=128, out_features=1, bias=False)
  )
  (fc0): Linear(in_features=128, out_features=64, bias=True)
  (relu_fc): ReLU()
  (dropout_fc): Dropout(p=0.3, inplace=False)
  (fc_final): Linear(in_features=64, out_features=1, bias=True)
)
Model Parameter Count: 71041
Calculated pos_weight for BCEWithLogitsLoss: 15.67


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Starting training for 100 epochs on 43271 original samples...


Epochs:   0%|          | 0/100 [00:00<?, ?it/s]/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no 

Epoch [1/100], Train Loss: 1.3039, Val Loss: 1.3011, Val AUC: 0.6436, Val AUPRC: 0.1190
Validation loss improved. Saved best model to /home/server/Projects/data/AKI/models/best_lstm_v2_model.pth


...

Epochs:  64%|██████▍   | 64/100 [35:07<19:45, 32.94s/it]

Epoch [64/100], Train Loss: 1.0204, Val Loss: 1.1446, Val AUC: 0.7363, Val AUPRC: 0.2186
Validation loss did not improve for 14 epoch(s).


Epochs:  64%|██████▍   | 64/100 [35:40<20:04, 33.45s/it]
/tmp/ipykernel_125621/3759346896.py:366: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.l

Epoch [65/100], Train Loss: 1.0182, Val Loss: 1.1449, Val AUC: 0.7362, Val AUPRC: 0.2186
Validation loss did not improve for 15 epoch(s).
Early stopping triggered after 65 epochs.
Training finished.
Loading best model from /home/server/Projects/data/AKI/models/best_lstm_v2_model.pth for final evaluation.

Starting final evaluation with the best/last model...



--- Final Test Performance (Best/Last Model) ---
Precision: 0.0813
Sensitivity: 0.8737
Accuracy: 0.4004
rc_auc: 0.7368
pr_auc: 0.2186
Specificity: 0.3701
Negative Predictive Value: 0.9787
F1 Score: 0.1488
Results saved to /home/server/Projects/data/AKI/results/class_weighted_lstm_v2_results.pkl
Script finished.


# BiLSTM + SAttention + SE 

Working

Epoch [68/100], Train Loss: 1.0368, Val Loss: 1.1485, Val AUC: 0.7317, Val AUPRC: 0.2105
Validation loss did not improve for 15 epoch(s).
Early stopping triggered after 68 epochs.
Training finished.
Loading best model from /home/server/Projects/data/AKI/models/best_bilstm_singleattention_se_model.pth for final evaluation.

Starting final evaluation with the best/last model...
                                                                         

--- Final Test Performance (Best/Last Model) ---
Precision: 0.0818
Sensitivity: 0.8644
Accuracy: 0.4097
rc_auc: 0.7326
pr_auc: 0.2125
Specificity: 0.3807
Negative Predictive Value: 0.9778
F1 Score: 0.1494
Results saved to /home/server/Projects/data/AKI/results/bilstm_singleattention_se_results.pkl
Script finished.


In [2]:
# %%
import os
import numpy as np
import torch
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
import torch.optim as optim

import random
# import scipy.stats as stats

from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_curve, auc, precision_recall_curve, average_precision_score)
# import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pandas as pd
from utils import performance_dict, sigmoid # Assuming these are your custom functions
from tqdm import tqdm

# --- Configuration ---
CONFIG = {
    "data_file": '/home/server/Projects/data/AKI/lstm_trainable.pkl',
    # <<< --- MODIFIED: Output file and model save path --- >>>
    "results_output_file": '/home/server/Projects/data/AKI/results/bilstm_singleattention_se_results.pkl',
    "tensorboard_log_dir_base": '/home/server/Projects/data/AKI/runs/bilstm_singleattention_se/',
    "best_model_save_path": '/home/server/Projects/data/AKI/models/best_bilstm_singleattention_se_model.pth',
    "test_split_size": 0.2,
    "random_seed": 42,
    "num_epochs": 100,
    "learning_rate": 0.0001,
    "weight_decay": 1e-5,
    "gradient_clip_value": 1.0,
    "lr_scheduler_patience": 7,
    "lr_scheduler_factor": 0.1,
    "early_stopping_patience": 15,
    "lstm_hidden_dim": 64,
    "num_lstm_layers": 1,
    "bidirectional": True,
    "use_attention": True,
    "attention_dim": 32,
    "use_se_block": True,  # <<< --- ADDED: Enable SE Block --- >>>
    "se_reduction_ratio": 16, # <<< --- ADDED: SE Block reduction ratio --- >>>
    "batch_size": 2000,
    "eval_batch_size": 512,
    "classification_threshold": 0.3,
    "dropout_rate": 0.3,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load data (once)
try:
    df_full = pd.read_pickle(CONFIG["data_file"])
except FileNotFoundError:
    print(f"ERROR: Data file not found at {CONFIG['data_file']}")
    print("Please ensure the path in CONFIG['data_file'] is correct.")
    exit()


# %%
# --- Data Preparation (remains the same) ---

def df_to_tensors(df, all_columns):
    non_tabular_cols = ['op_id', 'time_tensors', 'seq_len', 'aki', 'aki_boolean']
    tabular_feature_cols = [col for col in all_columns if col not in non_tabular_cols]
    
    if tabular_feature_cols and not df[tabular_feature_cols].empty:
        X_tab = torch.tensor(df[tabular_feature_cols].values).float()
    else:
        X_tab = torch.empty((len(df), 0)).float()

    X_time = torch.stack(df['time_tensors'].tolist()).float()
    y = torch.tensor(df['aki'].values).unsqueeze(1).float()
    y_binary = df['aki_boolean'].values.astype(bool)
    sequence_lengths = torch.tensor(df['seq_len'].tolist()).long()
    
    return X_tab, X_time, sequence_lengths, y, y_binary

def prepare_data(full_df, test_size, random_seed):
    df_train, df_test = train_test_split(full_df,
                                         test_size=test_size,
                                         random_state=random_seed,
                                         stratify=full_df['aki_boolean'])
    
    print(f"Original train size: {len(df_train)}")
    print(f"Test size: {len(df_test)}")

    all_columns = full_df.columns.tolist()

    num_positive_aki_train = df_train['aki_boolean'].sum()
    num_negative_aki_train = len(df_train) - num_positive_aki_train
    
    print(f"Training set: Positive AKI cases: {num_positive_aki_train}, Negative AKI cases: {num_negative_aki_train}")

    X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train = df_to_tensors(df_train, all_columns)
    X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test = df_to_tensors(df_test, all_columns)
    
    return (X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train,
            X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test,
            num_positive_aki_train, num_negative_aki_train)

# %%
# --- Utility Functions (set_seed, save_single_run_results remain the same) ---

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["random_seed"])

def save_single_run_results(model_name, metrics_dict, output_file):
    results_to_save = {k: [v] if not isinstance(v, (list, np.ndarray)) else v for k,v in metrics_dict.items()}
    df_result = pd.DataFrame(results_to_save)
    df_result['model_name'] = model_name
    
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    if os.path.exists(output_file):
        try:
            df_output = pd.read_pickle(output_file)
            df_output = df_output[df_output['model_name'] != model_name] # Remove old result for this model name
            df_output = pd.concat([df_output, df_result], ignore_index=True)
        except EOFError: # Handle case where pickle file might be empty or corrupted
             df_output = df_result
    else:
        df_output = df_result
    df_output.to_pickle(output_file)
    print(f"Results saved to {output_file}")

# %%
# --- Model Definition ---

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.W = nn.Linear(hidden_dim, hidden_dim)
        self.V = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_outputs): # lstm_outputs shape: (batch, seq_len, hidden_dim_attention_input)
        e = torch.tanh(self.W(lstm_outputs))
        e = self.V(e).squeeze(2)
        alpha = F.softmax(e, dim=1)
        context = torch.bmm(alpha.unsqueeze(1), lstm_outputs).squeeze(1)
        return context, alpha

# <<< --- ADDED: SequenceSEBlock Definition --- >>>
class SequenceSEBlock(nn.Module):
    def __init__(self, channels, reduction_ratio=16):
        super(SequenceSEBlock, self).__init__()
        self.channels = channels
        self.reduction_ratio = reduction_ratio
        # Squeeze: Global Average Pooling across sequence length
        self.avg_pool = nn.AdaptiveAvgPool1d(1) # Output: (batch, channels, 1)
        # Excitation: Two fully connected layers
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction_ratio, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction_ratio, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x): # x shape: (batch, seq_len, channels)
        b, s, c = x.shape
        # Squeeze operation
        # Permute to (batch, channels, seq_len) for AdaptiveAvgPool1d
        y = x.permute(0, 2, 1)
        y = self.avg_pool(y) # Result: (batch, channels, 1)
        y = y.squeeze(-1) # Result: (batch, channels)
        
        # Excitation operation
        y = self.fc(y) # Result: (batch, channels)
        
        # Scale operation: Multiply original input by channel weights
        # Reshape y to (batch, 1, channels) to broadcast with x
        return x * y.unsqueeze(1)


class EnhancedLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes,
                 bidirectional=True, use_attention=True, attention_dim=None, dropout_rate=0.2,
                 use_se_block=False, se_reduction_ratio=16): # Added se_reduction_ratio
        super(EnhancedLSTM, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.num_classes = num_classes
        self.bidirectional = bidirectional
        self.use_attention = use_attention
        self.use_se_block = use_se_block

        self.lstm = nn.LSTM(input_size=input_dim,
                            hidden_size=hidden_dim,
                            num_layers=num_layers,
                            batch_first=True,
                            bidirectional=bidirectional,
                            dropout=dropout_rate if num_layers > 1 else 0)

        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim

        # <<< --- ADDED: Instantiate SEBlock if use_se_block is True --- >>>
        if self.use_se_block:
            self.se_block = SequenceSEBlock(channels=lstm_output_dim, reduction_ratio=se_reduction_ratio)
        # SE block does not change the dimensionality of lstm_output_dim

        if self.use_attention:
            self.attention = Attention(lstm_output_dim) # Attention input is lstm_output_dim
            fc_input_dim = lstm_output_dim # Attention output is also lstm_output_dim
        else:
            fc_input_dim = lstm_output_dim
        
        self.fc_intermediate_dim = max(1, fc_input_dim // 2)
        self.fc0 = nn.Linear(fc_input_dim, self.fc_intermediate_dim)
        self.relu_fc = nn.ReLU()
        self.dropout_fc = nn.Dropout(dropout_rate)
        self.fc_final = nn.Linear(self.fc_intermediate_dim, num_classes)
        
    def forward(self, x_time, seq_lens):
        packed_input = pack_padded_sequence(x_time, seq_lens.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, (hn, cn) = self.lstm(packed_input)
        # output shape: (batch, seq_max_len, num_directions * hidden_size)
        lstm_sequence_output, _ = pad_packed_sequence(packed_output, batch_first=True)

        # <<< --- ADDED: Apply SEBlock if enabled --- >>>
        processed_sequence_output = lstm_sequence_output
        if self.use_se_block:
            processed_sequence_output = self.se_block(processed_sequence_output)

        if self.use_attention:
            # Attention mechanism operates on the (potentially SE-enhanced) sequence output
            context_vector, attention_weights = self.attention(processed_sequence_output)
            fc_input = context_vector
        else:
            # Use the final hidden state (concatenated if bidirectional)
            # If not using attention on sequence, SE block might be applied differently (e.g., on hn)
            # For now, if no attention, SE block applied to sequence output is less typical if only hn is used.
            # Let's assume if SE is used, attention is likely also used for sequence processing.
            # If SE is used and attention is OFF, we should ideally apply SE to 'hn' before FC layers.
            # For simplicity with current structure, if attention is OFF, SE on sequence_output won't directly feed into 'hn' logic.
            # So, let's refine to apply SE to the features that will be used by FC layers.

            if self.bidirectional:
                hn_last_layer_fwd = hn[-2,:,:]
                hn_last_layer_bwd = hn[-1,:,:]
                fc_input_pre_se = torch.cat((hn_last_layer_fwd, hn_last_layer_bwd), dim=1)
            else:
                fc_input_pre_se = hn[-1,:,:]
            
            # If SE is used AND attention is OFF, we'd apply SE here.
            # This requires SEBlock to handle (batch, features) input.
            # The current SequenceSEBlock is for (batch, seq, features).
            # Let's keep it simple: if SE is on, it applies to the sequence output passed to attention.
            # If attention is off, the current SE block (SequenceSEBlock) won't be applied to 'hn'.
            # For this iteration, we will assume SE block is intended for the sequence output that attention uses.
            fc_input = fc_input_pre_se # If no attention, use hn directly
                                      # If SE needs to be applied to hn, a different SE block structure is needed.
                                      # Let's assume for now that SE is primarily for the sequence path with attention.

        out = self.fc0(fc_input)
        out = self.relu_fc(out)
        out = self.dropout_fc(out)
        out = self.fc_final(out)
        return out

# %%
# --- Main Training and Evaluation ---

(X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train,
 X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test,
 num_pos_train, num_neg_train) = prepare_data(
    df_full, CONFIG["test_split_size"], CONFIG["random_seed"]
)

lstm_input_dim = X_time_train.shape[2] 

model = EnhancedLSTM(
    input_dim=lstm_input_dim,
    hidden_dim=CONFIG["lstm_hidden_dim"],
    num_layers=CONFIG["num_lstm_layers"],
    num_classes=1,
    bidirectional=CONFIG["bidirectional"],
    use_attention=CONFIG["use_attention"],
    attention_dim=CONFIG["attention_dim"],
    dropout_rate=CONFIG["dropout_rate"],
    use_se_block=CONFIG["use_se_block"], # Pass this to the model
    se_reduction_ratio=CONFIG["se_reduction_ratio"] # Pass this to the model
).to(device)

print(model)
print(f"Model Parameter Count: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

# --- Loss and Optimizer ---
if num_pos_train > 0:
    pos_weight_value = num_neg_train / num_pos_train
else:
    pos_weight_value = 1.0
    print("Warning: No positive samples found in the training set for pos_weight calculation.")
pos_weight = torch.tensor([pos_weight_value], device=device)
print(f"Calculated pos_weight for BCEWithLogitsLoss: {pos_weight.item():.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=CONFIG["lr_scheduler_factor"],
    patience=CONFIG["lr_scheduler_patience"]
)

# --- TensorBoard ---
run_name = f"LSTM_h{CONFIG['lstm_hidden_dim']}_l{CONFIG['num_lstm_layers']}"
if CONFIG['bidirectional']: run_name += "_bi"
if CONFIG['use_attention']: run_name += "_attn"
if CONFIG['use_se_block']: run_name += "_SE" # <<< --- MODIFIED: Added SE to run name --- >>>
run_name += "_classWeighted_v3_SE" # Updated identifier
tensorboard_run_path = os.path.join(CONFIG["tensorboard_log_dir_base"], run_name)
os.makedirs(tensorboard_run_path, exist_ok=True)
writer = SummaryWriter(tensorboard_run_path)

# --- Training Loop ---
num_train_samples = X_time_train.shape[0]
train_batch_size = CONFIG["batch_size"]
num_train_batches = int(np.ceil(num_train_samples / train_batch_size))

eval_batch_size = CONFIG["eval_batch_size"]
num_test_samples = X_time_test.shape[0]
num_eval_batches = int(np.ceil(num_test_samples / eval_batch_size))

best_val_loss = float('inf')
epochs_no_improve = 0
# Ensure directory for best model saving exists
os.makedirs(os.path.dirname(CONFIG["best_model_save_path"]), exist_ok=True)

print(f"Starting training for {CONFIG['num_epochs']} epochs on {num_train_samples} original samples...")
# (The rest of the training loop, validation loop, and final evaluation loop
# remain structurally the same as the previous version with batched evaluation.
# No changes needed there as the model's forward pass now incorporates the SE block if enabled.)

# ... (Keep the existing training loop, validation, and final evaluation code here) ...
# It will automatically use the modified EnhancedLSTM with the SE block if CONFIG["use_se_block"] is True.

# --- Training Loop (Copied from previous, ensure it's complete) ---
for epoch in tqdm(range(CONFIG["num_epochs"]), desc="Epochs"):
    model.train()
    epoch_loss = 0.0
    for i in tqdm(range(num_train_batches), desc="Training Batches", leave=False):
        start_idx = i * train_batch_size
        end_idx = min(num_train_samples, (i + 1) * train_batch_size)

        batch_x_time = X_time_train[start_idx:end_idx].to(device)
        batch_seq_len = seq_len_train[start_idx:end_idx]
        batch_y_binary = torch.tensor(y_binary_train[start_idx:end_idx]).unsqueeze(1).float().to(device)

        optimizer.zero_grad()
        outputs = model(batch_x_time, batch_seq_len) # Model forward pass
        loss = criterion(outputs, batch_y_binary)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CONFIG["gradient_clip_value"])
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_epoch_loss = epoch_loss / num_train_batches
    writer.add_scalar('Loss/train', avg_epoch_loss, epoch)
    
    # --- Batched Validation ---
    model.eval()
    all_test_outputs_logits = []
    all_test_y_for_loss = []

    with torch.no_grad():
        for i in tqdm(range(num_eval_batches), desc="Validation Batches", leave=False):
            start_idx = i * eval_batch_size
            end_idx = min(num_test_samples, (i + 1) * eval_batch_size)

            batch_x_time_test = X_time_test[start_idx:end_idx].to(device)
            batch_seq_len_test = seq_len_test[start_idx:end_idx]
            batch_y_binary_test_for_loss = torch.tensor(y_binary_test[start_idx:end_idx]).unsqueeze(1).float().to(device)

            outputs_batch_logits = model(batch_x_time_test, batch_seq_len_test)
            all_test_outputs_logits.append(outputs_batch_logits.cpu())
            all_test_y_for_loss.append(batch_y_binary_test_for_loss.cpu())

    val_outputs_logits = torch.cat(all_test_outputs_logits, dim=0)
    val_y_for_loss = torch.cat(all_test_y_for_loss, dim=0)
    
    current_val_loss = criterion(val_outputs_logits.to(device), val_y_for_loss.to(device)).item()
    writer.add_scalar('Loss/val', current_val_loss, epoch)
    
    val_probs = torch.sigmoid(val_outputs_logits).numpy().squeeze()
    # Ensure val_pred_binary is calculated based on the full validation set size
    if val_probs.ndim == 0: # Handle case where only one sample might be in the last batch during eval
        val_pred_binary = np.array([val_probs > CONFIG["classification_threshold"]])
    else:
        val_pred_binary = (val_probs > CONFIG["classification_threshold"])

    # Ensure y_binary_test is used for the full test set evaluation
    dic = performance_dict(y_binary_test[:len(val_pred_binary)], val_pred_binary, val_probs) # Match lengths if necessary

    writer.add_scalar('Val/AUC', dic.get('rc_auc', np.nan), epoch)
    writer.add_scalar('Val/AUPRC', dic.get('pr_auc', np.nan), epoch)
    writer.add_scalar('Val/Accuracy', dic.get('Accuracy', np.nan), epoch)
    writer.add_scalar('Val/Sensitivity', dic.get('Sensitivity', np.nan), epoch)
    writer.add_scalar('Val/Specificity', dic.get('Specificity', np.nan), epoch)
    writer.add_scalar('Val/F1', dic.get('F1 Score', np.nan), epoch)
    
    print(f"Epoch [{epoch+1}/{CONFIG['num_epochs']}], Train Loss: {avg_epoch_loss:.4f}, Val Loss: {current_val_loss:.4f}, Val AUC: {dic.get('rc_auc', np.nan):.4f}, Val AUPRC: {dic.get('pr_auc', np.nan):.4f}")

    scheduler.step(current_val_loss)

    if current_val_loss < best_val_loss:
        best_val_loss = current_val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), CONFIG["best_model_save_path"])
        print(f"Validation loss improved. Saved best model to {CONFIG['best_model_save_path']}")
    else:
        epochs_no_improve += 1
        print(f"Validation loss did not improve for {epochs_no_improve} epoch(s).")

    if epochs_no_improve >= CONFIG["early_stopping_patience"]:
        print(f"Early stopping triggered after {epoch+1} epochs.")
        break

writer.close()
print("Training finished.")

if os.path.exists(CONFIG["best_model_save_path"]):
    print(f"Loading best model from {CONFIG['best_model_save_path']} for final evaluation.")
    model.load_state_dict(torch.load(CONFIG["best_model_save_path"]))
else:
    print("No best model found (early stopping might not have triggered improvement or ran for 0 epochs). Using last model state.")

# --- Batched Final Evaluation (remains the same) ---
model.eval()
final_all_test_outputs_logits = []
print("\nStarting final evaluation with the best/last model...")
with torch.no_grad():
    for i in tqdm(range(num_eval_batches), desc="Final Evaluation Batches", leave=False):
        start_idx = i * eval_batch_size
        end_idx = min(num_test_samples, (i + 1) * eval_batch_size)

        batch_x_time_test = X_time_test[start_idx:end_idx].to(device)
        batch_seq_len_test = seq_len_test[start_idx:end_idx]

        outputs_batch_logits = model(batch_x_time_test, batch_seq_len_test)
        final_all_test_outputs_logits.append(outputs_batch_logits.cpu())

final_test_outputs_logits = torch.cat(final_all_test_outputs_logits, dim=0)

final_test_probs = torch.sigmoid(final_test_outputs_logits).numpy().squeeze()
if final_test_probs.ndim == 0: # Handle case for single sample in last batch if test set not perfectly divisible
    final_y_pred_binary = np.array([final_test_probs > CONFIG["classification_threshold"]])
else:
    final_y_pred_binary = (final_test_probs > CONFIG["classification_threshold"])

# Ensure y_binary_test is used for the full test set evaluation
final_metrics = performance_dict(y_binary_test[:len(final_y_pred_binary)], final_y_pred_binary, final_test_probs) # Match lengths

print("\n--- Final Test Performance (Best/Last Model) ---")
for metric, value in final_metrics.items():
    if isinstance(value, (float, int, np.number)):
        print(f"{metric}: {value:.4f}")

save_single_run_results(run_name, final_metrics, CONFIG["results_output_file"])
print("Script finished.")

Using device: cuda
Original train size: 43271
Test size: 10818
Training set: Positive AKI cases: 2595.0, Negative AKI cases: 40676.0


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


EnhancedLSTM(
  (lstm): LSTM(24, 64, batch_first=True, bidirectional=True)
  (se_block): SequenceSEBlock(
    (avg_pool): AdaptiveAvgPool1d(output_size=1)
    (fc): Sequential(
      (0): Linear(in_features=128, out_features=8, bias=False)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=8, out_features=128, bias=False)
      (3): Sigmoid()
    )
  )
  (attention): Attention(
    (W): Linear(in_features=128, out_features=128, bias=True)
    (V): Linear(in_features=128, out_features=1, bias=False)
  )
  (fc0): Linear(in_features=128, out_features=64, bias=True)
  (relu_fc): ReLU()
  (dropout_fc): Dropout(p=0.3, inplace=False)
  (fc_final): Linear(in_features=64, out_features=1, bias=True)
)
Model Parameter Count: 73089
Calculated pos_weight for BCEWithLogitsLoss: 15.67
Starting training for 100 epochs on 43271 original samples...


Epochs:   0%|          | 0/100 [00:00<?, ?it/s]/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no 

Epoch [1/100], Train Loss: 1.3064, Val Loss: 1.3051, Val AUC: 0.6483, Val AUPRC: 0.1314
Validation loss improved. Saved best model to /home/server/Projects/data/AKI/models/best_bilstm_singleattention_se_model.pth


...

Epochs:  67%|██████▋   | 67/100 [37:46<18:37, 33.85s/it]

Epoch [67/100], Train Loss: 1.0364, Val Loss: 1.1485, Val AUC: 0.7317, Val AUPRC: 0.2104
Validation loss did not improve for 14 epoch(s).


Epochs:  67%|██████▋   | 67/100 [38:20<18:53, 34.34s/it]
/tmp/ipykernel_125621/54040442.py:435: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.loa

Epoch [68/100], Train Loss: 1.0368, Val Loss: 1.1485, Val AUC: 0.7317, Val AUPRC: 0.2105
Validation loss did not improve for 15 epoch(s).
Early stopping triggered after 68 epochs.
Training finished.
Loading best model from /home/server/Projects/data/AKI/models/best_bilstm_singleattention_se_model.pth for final evaluation.

Starting final evaluation with the best/last model...



--- Final Test Performance (Best/Last Model) ---
Precision: 0.0818
Sensitivity: 0.8644
Accuracy: 0.4097
rc_auc: 0.7326
pr_auc: 0.2125
Specificity: 0.3807
Negative Predictive Value: 0.9778
F1 Score: 0.1494
Results saved to /home/server/Projects/data/AKI/results/bilstm_singleattention_se_results.pkl
Script finished.


# 2x BiLSTM + MAttention + SE

Working

Epoch [31/100], Train Loss: 1.0553, Val Loss: 1.1386, Val AUC: 0.7384, Val AUPRC: 0.2197
Validation loss did not improve for 15 epoch(s).
Early stopping triggered after 31 epochs.
Training finished.
Loading best model from /home/server/Projects/data/AKI/models/best_dual_bilstm_se_mha_model.pth for final evaluation.

Starting final evaluation with the best/last model...
                                                                         

--- Final Test Performance (Best/Last Model) ---
Precision: 0.0804
Sensitivity: 0.8860
Accuracy: 0.3854
rc_auc: 0.7362
pr_auc: 0.2156
Specificity: 0.3534
Negative Predictive Value: 0.9798
F1 Score: 0.1475
Results saved to /home/server/Projects/data/AKI/results/dual_bilstm_se_mha_results.pkl
Script finished.

In [1]:
# %%
import os
import numpy as np
import torch
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
import torch.optim as optim

import random
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_curve, auc, precision_recall_curve, average_precision_score)
from sklearn.model_selection import train_test_split
import pandas as pd
# Ensure utils.py is in the same directory or Python path
from utils import performance_dict, sigmoid
from tqdm import tqdm

# --- Configuration for the Specific Model ---
CONFIG = {
    "data_file": '/home/server/Projects/data/AKI/lstm_trainable.pkl', # !!! UPDATE THIS PATH !!!
    "results_output_file": '/home/server/Projects/data/AKI/results/dual_bilstm_se_mha_results.pkl',
    "tensorboard_log_dir_base": '/home/server/Projects/data/AKI/runs/dual_bilstm_se_mha/',
    "best_model_save_path": '/home/server/Projects/data/AKI/models/best_dual_bilstm_se_mha_model.pth',
    "test_split_size": 0.2,
    "random_seed": 42,
    "num_epochs": 100, # Max epochs for this run (early stopping will likely trigger)
    "learning_rate": 0.0001,
    "weight_decay": 1e-5,
    "gradient_clip_value": 1.0,
    "lr_scheduler_patience": 7,
    "lr_scheduler_factor": 0.1,
    "early_stopping_patience": 15,
    
    # --- Architecture Specifics ---
    "lstm_hidden_dim": 64,      # Hidden dim for each LSTM direction
    "num_lstm_layers": 2,       # Dual Layer
    "bidirectional": True,      # BiLSTM
    "use_attention": True,      # Enable Attention (will be Multi-Head)
    "num_attention_heads": 4,   # <<--- For Multi-Head Attention (ensure lstm_output_dim is divisible by this)
    "use_se_block": True,       # Enable SE Block
    "se_reduction_ratio": 16,
    "dropout_rate": 0.3,
    
    "batch_size": 640,
    "eval_batch_size": 512,
    "classification_threshold": 0.3, # Review this after training
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load data (once)
try:
    df_full = pd.read_pickle(CONFIG["data_file"])
except FileNotFoundError:
    print(f"ERROR: Data file not found at {CONFIG['data_file']}")
    print("Please ensure the path in CONFIG['data_file'] is correct.")
    exit()
except Exception as e:
    print(f"Error loading data file {CONFIG['data_file']}: {e}")
    exit()


# %%
# --- Data Preparation ---
def df_to_tensors(df, all_columns):
    non_tabular_cols = ['op_id', 'time_tensors', 'seq_len', 'aki', 'aki_boolean']
    tabular_feature_cols = [col for col in all_columns if col not in non_tabular_cols]
    if tabular_feature_cols and not df[tabular_feature_cols].empty:
        X_tab = torch.tensor(df[tabular_feature_cols].values).float()
    else:
        X_tab = torch.empty((len(df), 0)).float()
    X_time = torch.stack(df['time_tensors'].tolist()).float()
    y = torch.tensor(df['aki'].values).unsqueeze(1).float()
    y_binary = df['aki_boolean'].values.astype(bool)
    sequence_lengths = torch.tensor(df['seq_len'].tolist()).long()
    return X_tab, X_time, sequence_lengths, y, y_binary

def prepare_data(full_df, test_size, random_seed):
    df_train, df_test = train_test_split(full_df, test_size=test_size, random_state=random_seed, stratify=full_df['aki_boolean'])
    print(f"Original train size: {len(df_train)}")
    print(f"Test size: {len(df_test)}")
    all_columns = full_df.columns.tolist()
    num_positive_aki_train = df_train['aki_boolean'].sum()
    num_negative_aki_train = len(df_train) - num_positive_aki_train
    print(f"Training set: Positive AKI cases: {num_positive_aki_train}, Negative AKI cases: {num_negative_aki_train}")
    X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train = df_to_tensors(df_train, all_columns)
    X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test = df_to_tensors(df_test, all_columns)
    return (X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train,
            X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test,
            num_positive_aki_train, num_negative_aki_train)

(X_tab_train_full, X_time_train_full, seq_len_train_full, y_train_full, y_binary_train_full,
 X_tab_test_full, X_time_test_full, seq_len_test_full, y_test_full, y_binary_test_full,
 num_pos_train_full, num_neg_train_full) = prepare_data(
    df_full, CONFIG["test_split_size"], CONFIG["random_seed"]
)
lstm_input_dim_global = X_time_train_full.shape[2]

# %%
# --- Utility Functions ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["random_seed"])

def save_single_run_results(model_name, metrics_dict, output_file):
    results_to_save = {k: [v] if not isinstance(v, (list, np.ndarray)) else v for k,v in metrics_dict.items()}
    df_result = pd.DataFrame(results_to_save)
    df_result['model_name'] = model_name
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    if os.path.exists(output_file):
        try:
            df_output = pd.read_pickle(output_file)
            df_output = df_output[df_output['model_name'] != model_name]
            df_output = pd.concat([df_output, df_result], ignore_index=True)
        except EOFError: df_output = df_result
    else: df_output = df_result
    df_output.to_pickle(output_file)
    print(f"Results saved to {output_file}")

# %%
# --- Model Definition ---
class SequenceSEBlock(nn.Module):
    def __init__(self, channels, reduction_ratio=16):
        super(SequenceSEBlock, self).__init__()
        self.channels = channels; self.reduction_ratio = reduction_ratio
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, max(1, channels // reduction_ratio), bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(max(1, channels // reduction_ratio), channels, bias=False),
            nn.Sigmoid())
    def forward(self, x): # x shape: (batch, seq_len, channels)
        b, s, c = x.shape; y = x.permute(0, 2, 1); y = self.avg_pool(y); y = y.squeeze(-1)
        y = self.fc(y); return x * y.unsqueeze(1)

class EnhancedLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes,
                 bidirectional=True, use_attention=True, num_attention_heads=4, # Changed from attention_dim
                 dropout_rate=0.2, use_se_block=False, se_reduction_ratio=16):
        super(EnhancedLSTM, self).__init__()
        self.input_dim = input_dim; self.hidden_dim = hidden_dim; self.num_layers = num_layers
        self.num_classes = num_classes; self.bidirectional = bidirectional
        self.use_attention = use_attention; self.use_se_block = use_se_block

        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=bidirectional,
                            dropout=dropout_rate if num_layers > 1 else 0)
        
        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        
        self.se_block_instance = None
        if self.use_se_block:
            self.se_block_instance = SequenceSEBlock(channels=lstm_output_dim, reduction_ratio=se_reduction_ratio)
        
        self.attention_instance = None
        if self.use_attention:
            # Ensure lstm_output_dim is divisible by num_attention_heads
            if lstm_output_dim % num_attention_heads != 0:
                raise ValueError(f"lstm_output_dim ({lstm_output_dim}) must be divisible by num_attention_heads ({num_attention_heads})")
            self.attention_instance = nn.MultiheadAttention(
                embed_dim=lstm_output_dim, 
                num_heads=num_attention_heads, 
                dropout=dropout_rate, # Can use a separate attention dropout if desired
                batch_first=True # Important for (batch, seq, feature) input
            )
            fc_input_dim = lstm_output_dim # Output of MHA is also embed_dim
        else:
            fc_input_dim = lstm_output_dim
            
        self.fc_intermediate_dim = max(1, fc_input_dim // 2)
        self.fc0 = nn.Linear(fc_input_dim, self.fc_intermediate_dim)
        self.relu_fc = nn.ReLU(); self.dropout_fc = nn.Dropout(dropout_rate)
        self.fc_final = nn.Linear(self.fc_intermediate_dim, num_classes)

    def forward(self, x_time, seq_lens):
        # Create a mask for padded elements for nn.MultiheadAttention
        # True for positions that should NOT be attended to.
        # max_len = x_time.size(1)
        # idx = torch.arange(max_len, device=x_time.device).unsqueeze(0) # (1, max_len)
        # key_padding_mask = (idx >= seq_lens.cpu().unsqueeze(1)).bool() # (batch, max_len)

        packed_input = pack_padded_sequence(x_time, seq_lens.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, (hn, cn) = self.lstm(packed_input)
        lstm_sequence_output, _ = pad_packed_sequence(packed_output, batch_first=True)
        
        processed_sequence_output = lstm_sequence_output
        if self.use_se_block and self.se_block_instance is not None:
            processed_sequence_output = self.se_block_instance(processed_sequence_output)
            
        if self.use_attention and self.attention_instance is not None:
            # For nn.MultiheadAttention, query, key, value are usually the same for self-attention
            # key_padding_mask should be provided if sequences have different lengths
            # Create mask based on actual sequence lengths
            max_len = processed_sequence_output.size(1)
            idx = torch.arange(max_len, device=seq_lens.device).unsqueeze(0)
            # seq_lens might be on CPU from pack_padded_sequence, ensure it's on the correct device if not already
            # or that idx is on CPU if seq_lens is on CPU.
            # For key_padding_mask, lengths should be on the same device as the mask.
            # Let's use seq_lens.cpu() to be safe with idx on CPU.
            key_padding_mask = (idx >= seq_lens.cpu().unsqueeze(1)).to(device) # (batch, max_len) True where padded

            attn_output, attn_weights = self.attention_instance(
                query=processed_sequence_output,
                key=processed_sequence_output,
                value=processed_sequence_output,
                key_padding_mask=key_padding_mask # Use this mask
            )
            # attn_output is (batch, seq_len, embed_dim). We need a single vector.
            # Option 1: Mean pooling of attended outputs
            fc_input = attn_output.mean(dim=1)
            # Option 2: Use the output corresponding to the last valid token (more complex with padding)
            # For now, mean pooling is simpler.
        else:
            if self.bidirectional:
                fc_input = torch.cat((hn[-2,:,:], hn[-1,:,:]), dim=1)
            else:
                fc_input = hn[-1,:,:]
                
        out = self.fc0(fc_input); out = self.relu_fc(out); out = self.dropout_fc(out)
        out = self.fc_final(out)
        return out

# %%
# --- Main Training and Evaluation ---
model = EnhancedLSTM(
    input_dim=lstm_input_dim_global,
    hidden_dim=CONFIG["lstm_hidden_dim"],
    num_layers=CONFIG["num_lstm_layers"],
    num_classes=1,
    bidirectional=CONFIG["bidirectional"],
    use_attention=CONFIG["use_attention"],
    num_attention_heads=CONFIG["num_attention_heads"], # Pass num_heads
    dropout_rate=CONFIG["dropout_rate"],
    use_se_block=CONFIG["use_se_block"],
    se_reduction_ratio=CONFIG["se_reduction_ratio"]
).to(device)

print(model)
print(f"Model Parameter Count: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

if num_pos_train_full > 0:
    pos_weight_value = num_neg_train_full / num_pos_train_full
else:
    pos_weight_value = 1.0; print("Warning: No positive samples in train for pos_weight.")
pos_weight = torch.tensor([pos_weight_value], device=device)
print(f"Calculated pos_weight for BCEWithLogitsLoss: {pos_weight.item():.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=CONFIG["lr_scheduler_factor"],
    patience=CONFIG["lr_scheduler_patience"])

run_name = f"DualBiLSTM_h{CONFIG['lstm_hidden_dim']}_l{CONFIG['num_lstm_layers']}"
if CONFIG['use_se_block']: run_name += "_SE"
if CONFIG['use_attention']: run_name += f"_MHA{CONFIG['num_attention_heads']}"
run_name += "_classWeighted_final"
tensorboard_run_path = os.path.join(CONFIG["tensorboard_log_dir_base"], run_name)
os.makedirs(tensorboard_run_path, exist_ok=True)
writer = SummaryWriter(tensorboard_run_path)

num_train_samples = X_time_train_full.shape[0]
train_batch_size = CONFIG["batch_size"]
num_train_batches = int(np.ceil(num_train_samples / train_batch_size))
eval_batch_size = CONFIG["eval_batch_size"]
num_test_samples = X_time_test_full.shape[0]
num_eval_batches = int(np.ceil(num_test_samples / eval_batch_size))

best_val_loss = float('inf')
epochs_no_improve = 0
os.makedirs(os.path.dirname(CONFIG["best_model_save_path"]), exist_ok=True)

print(f"Starting training for {CONFIG['num_epochs']} epochs on {num_train_samples} original samples...")
for epoch in tqdm(range(CONFIG["num_epochs"]), desc="Epochs"):
    model.train()
    epoch_train_loss = 0.0
    for i in tqdm(range(num_train_batches), desc="Training Batches", leave=False):
        start_idx = i * train_batch_size
        end_idx = min(num_train_samples, (i + 1) * train_batch_size)
        batch_x_time = X_time_train_full[start_idx:end_idx].to(device)
        batch_seq_len = seq_len_train_full[start_idx:end_idx]
        batch_y_binary = torch.tensor(y_binary_train_full[start_idx:end_idx]).unsqueeze(1).float().to(device)
        optimizer.zero_grad()
        outputs = model(batch_x_time, batch_seq_len)
        loss = criterion(outputs, batch_y_binary)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CONFIG["gradient_clip_value"])
        optimizer.step()
        epoch_train_loss += loss.item()
    avg_epoch_train_loss = epoch_train_loss / num_train_batches
    writer.add_scalar('Loss/train', avg_epoch_train_loss, epoch)

    model.eval()
    all_val_outputs_logits = []; all_val_y_for_loss = []
    with torch.no_grad():
        for i in tqdm(range(num_eval_batches), desc="Validation Batches", leave=False):
            start_idx = i * eval_batch_size; end_idx = min(num_test_samples, (i + 1) * eval_batch_size)
            batch_x_time_val = X_time_test_full[start_idx:end_idx].to(device)
            batch_seq_len_val = seq_len_test_full[start_idx:end_idx]
            batch_y_binary_val_for_loss = torch.tensor(y_binary_test_full[start_idx:end_idx]).unsqueeze(1).float().to(device)
            outputs_batch_logits = model(batch_x_time_val, batch_seq_len_val)
            all_val_outputs_logits.append(outputs_batch_logits.cpu()); all_val_y_for_loss.append(batch_y_binary_val_for_loss.cpu())
    val_outputs_logits = torch.cat(all_val_outputs_logits, dim=0); val_y_for_loss = torch.cat(all_val_y_for_loss, dim=0)
    current_val_loss = criterion(val_outputs_logits.to(device), val_y_for_loss.to(device)).item()
    writer.add_scalar('Loss/val', current_val_loss, epoch)
    val_probs = torch.sigmoid(val_outputs_logits).numpy().squeeze()
    if val_probs.ndim == 0: val_probs = np.array([val_probs])
    val_pred_binary = (val_probs > CONFIG["classification_threshold"])
    current_y_true_for_metric = y_binary_test_full[:len(val_pred_binary)]
    try:
        dic = performance_dict(current_y_true_for_metric, val_pred_binary, val_probs)
        writer.add_scalar('Val/AUC', dic.get('rc_auc', np.nan), epoch)
        writer.add_scalar('Val/AUPRC', dic.get('pr_auc', np.nan), epoch)
        writer.add_scalar('Val/Accuracy', dic.get('Accuracy', np.nan), epoch)
        # ... (log other metrics from dic)
        print(f"Epoch [{epoch+1}/{CONFIG['num_epochs']}], Train Loss: {avg_epoch_train_loss:.4f}, Val Loss: {current_val_loss:.4f}, Val AUC: {dic.get('rc_auc', np.nan):.4f}, Val AUPRC: {dic.get('pr_auc', np.nan):.4f}")
    except Exception as e:
        print(f"Error calculating performance_dict in validation: {e}")
        print(f"Epoch [{epoch+1}/{CONFIG['num_epochs']}], Train Loss: {avg_epoch_train_loss:.4f}, Val Loss: {current_val_loss:.4f}")


    scheduler.step(current_val_loss)
    if current_val_loss < best_val_loss:
        best_val_loss = current_val_loss; epochs_no_improve = 0
        torch.save(model.state_dict(), CONFIG["best_model_save_path"])
        print(f"Validation loss improved. Saved best model to {CONFIG['best_model_save_path']}")
    else:
        epochs_no_improve += 1
        print(f"Validation loss did not improve for {epochs_no_improve} epoch(s).")
    if epochs_no_improve >= CONFIG["early_stopping_patience"]:
        print(f"Early stopping triggered after {epoch+1} epochs.")
        break
writer.close()
print("Training finished.")

if os.path.exists(CONFIG["best_model_save_path"]):
    print(f"Loading best model from {CONFIG['best_model_save_path']} for final evaluation.")
    model.load_state_dict(torch.load(CONFIG["best_model_save_path"]))
else:
    print("No best model found. Using last model state.")

model.eval()
final_all_test_outputs_logits = []
print("\nStarting final evaluation with the best/last model...")
with torch.no_grad():
    for i in tqdm(range(num_eval_batches), desc="Final Evaluation Batches", leave=False):
        start_idx = i * eval_batch_size; end_idx = min(num_test_samples, (i + 1) * eval_batch_size)
        batch_x_time_test = X_time_test_full[start_idx:end_idx].to(device)
        batch_seq_len_test = seq_len_test_full[start_idx:end_idx]
        outputs_batch_logits = model(batch_x_time_test, batch_seq_len_test)
        final_all_test_outputs_logits.append(outputs_batch_logits.cpu())
final_test_outputs_logits = torch.cat(final_all_test_outputs_logits, dim=0)
final_test_probs = torch.sigmoid(final_test_outputs_logits).numpy().squeeze()
if final_test_probs.ndim == 0: final_test_probs = np.array([final_test_probs])
final_y_pred_binary = (final_test_probs > CONFIG["classification_threshold"])
current_y_true_for_final_metric = y_binary_test_full[:len(final_y_pred_binary)]
try:
    final_metrics = performance_dict(current_y_true_for_final_metric, final_y_pred_binary, final_test_probs)
    print("\n--- Final Test Performance (Best/Last Model) ---")
    for metric, value in final_metrics.items():
        if isinstance(value, (float, int, np.number)): print(f"{metric}: {value:.4f}")
    save_single_run_results(run_name, final_metrics, CONFIG["results_output_file"])
except Exception as e:
    print(f"Error calculating final performance_dict: {e}")

print("Script finished.")

Using device: cuda
Original train size: 43271
Test size: 10818
Training set: Positive AKI cases: 2595.0, Negative AKI cases: 40676.0
EnhancedLSTM(
  (lstm): LSTM(24, 64, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (se_block_instance): SequenceSEBlock(
    (avg_pool): AdaptiveAvgPool1d(output_size=1)
    (fc): Sequential(
      (0): Linear(in_features=128, out_features=8, bias=False)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=8, out_features=128, bias=False)
      (3): Sigmoid()
    )
  )
  (attention_instance): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
  )
  (fc0): Linear(in_features=128, out_features=64, bias=True)
  (relu_fc): ReLU()
  (dropout_fc): Dropout(p=0.3, inplace=False)
  (fc_final): Linear(in_features=64, out_features=1, bias=True)
)
Model Parameter Count: 221825
Calculated pos_weight for BCEWithLogitsLoss: 15.67


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Starting training for 100 epochs on 43271 original samples...


Epochs:   0%|          | 0/100 [00:00<?, ?it/s]/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no 

Epoch [1/100], Train Loss: 1.3038, Val Loss: 1.2965, Val AUC: 0.6494, Val AUPRC: 0.1042
Validation loss improved. Saved best model to /home/server/Projects/data/AKI/models/best_dual_bilstm_se_mha_model.pth


...

Epochs:  30%|███       | 30/100 [25:01<58:23, 50.05s/it]

Epoch [30/100], Train Loss: 1.0528, Val Loss: 1.1391, Val AUC: 0.7385, Val AUPRC: 0.2197
Validation loss did not improve for 14 epoch(s).


Epochs:  30%|███       | 30/100 [25:51<1:00:20, 51.72s/it]
/tmp/ipykernel_127091/1657546830.py:349: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch

Epoch [31/100], Train Loss: 1.0553, Val Loss: 1.1386, Val AUC: 0.7384, Val AUPRC: 0.2197
Validation loss did not improve for 15 epoch(s).
Early stopping triggered after 31 epochs.
Training finished.
Loading best model from /home/server/Projects/data/AKI/models/best_dual_bilstm_se_mha_model.pth for final evaluation.

Starting final evaluation with the best/last model...



--- Final Test Performance (Best/Last Model) ---
Precision: 0.0804
Sensitivity: 0.8860
Accuracy: 0.3854
rc_auc: 0.7362
pr_auc: 0.2156
Specificity: 0.3534
Negative Predictive Value: 0.9798
F1 Score: 0.1475
Results saved to /home/server/Projects/data/AKI/results/dual_bilstm_se_mha_results.pkl
Script finished.


# HPO LSTM

Working

--- HPO Study Finished ---
Number of finished trials: 100

Best trial for Plain LSTM:
  Value (Best Validation Loss): 1.1224
  Params: 
    learning_rate: 0.0012395393822216506
    lstm_hidden_dim: 64
    dropout_rate: 0.2
    weight_decay: 2.4834455388189246e-06
Best hyperparameters for Plain LSTM saved to /home/server/Projects/data/AKI/results/hpo_plain_lstm/best_plain_lstm_hpo_params.json

To train a final Plain LSTM model with these best hyperparameters, update your CONFIG dict,
set num_epochs to a higher value (e.g., original 100), and run a standard (non-HPO) training script.
Full HPO study results for Plain LSTM saved to /home/server/Projects/data/AKI/results/hpo_plain_lstm/hpo_plain_lstm_study_results.csv
Script finished.

In [2]:
# %%
import os
import numpy as np
import torch
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
import torch.optim as optim
import optuna
import functools

import random
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_curve, auc, precision_recall_curve, average_precision_score)
from sklearn.model_selection import train_test_split
import pandas as pd
from utils import performance_dict, sigmoid # Ensure utils.py is present
from tqdm import tqdm

# --- Configuration ---
CONFIG = {
    "data_file": '/home/server/Projects/data/AKI/lstm_trainable.pkl', # !!! UPDATE THIS PATH !!!
    "results_output_file_base": '/home/server/Projects/data/AKI/results/hpo_plain_lstm/',
    "tensorboard_log_dir_base": '/home/server/Projects/data/AKI/runs/hpo_plain_lstm/',
    "best_model_save_path_base": '/home/server/Projects/data/AKI/models/hpo_plain_lstm/',
    "test_split_size": 0.2,
    "random_seed": 42,
    "num_epochs_hpo": 50,
    "gradient_clip_value": 1.0,
    "lr_scheduler_patience": 5,
    "lr_scheduler_factor": 0.2,
    "early_stopping_patience": 10,
    
    # --- Fixed Architectural Choices for PLAIN LSTM HPO ---
    "num_lstm_layers": 1,       # Plain LSTM is single layer
    "bidirectional": False,     # Plain LSTM is unidirectional
    "use_attention": False,     # No attention for plain LSTM
    "attention_dim": 32,        # Not used, placeholder
    "num_attention_heads": 4,   # Not used, placeholder
    "use_se_block": False,      # No SE Block for plain LSTM
    "se_reduction_ratio": 16,   # Not used
    
    "batch_size": 1024,
    "eval_batch_size": 512,
    "classification_threshold": 0.3,
    "n_hpo_trials": 100, # Number of HPO trials
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load data (once)
try:
    df_full = pd.read_pickle(CONFIG["data_file"])
except FileNotFoundError:
    print(f"ERROR: Data file not found at {CONFIG['data_file']}")
    print("Please ensure the path in CONFIG['data_file'] is correct.")
    exit()
except Exception as e:
    print(f"Error loading data file {CONFIG['data_file']}: {e}")
    exit()


# %%
# --- Data Preparation ---
def df_to_tensors(df, all_columns):
    non_tabular_cols = ['op_id', 'time_tensors', 'seq_len', 'aki', 'aki_boolean']
    tabular_feature_cols = [col for col in all_columns if col not in non_tabular_cols]
    if tabular_feature_cols and not df[tabular_feature_cols].empty:
        X_tab = torch.tensor(df[tabular_feature_cols].values).float()
    else:
        X_tab = torch.empty((len(df), 0)).float()
    X_time = torch.stack(df['time_tensors'].tolist()).float()
    y = torch.tensor(df['aki'].values).unsqueeze(1).float()
    y_binary = df['aki_boolean'].values.astype(bool)
    sequence_lengths = torch.tensor(df['seq_len'].tolist()).long()
    return X_tab, X_time, sequence_lengths, y, y_binary

def prepare_data(full_df, test_size, random_seed):
    df_train, df_test = train_test_split(full_df, test_size=test_size, random_state=random_seed, stratify=full_df['aki_boolean'])
    print(f"Original train size: {len(df_train)}")
    print(f"Test size: {len(df_test)}")
    all_columns = full_df.columns.tolist()
    num_positive_aki_train = df_train['aki_boolean'].sum()
    num_negative_aki_train = len(df_train) - num_positive_aki_train
    print(f"Training set: Positive AKI cases: {num_positive_aki_train}, Negative AKI cases: {num_negative_aki_train}")
    X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train = df_to_tensors(df_train, all_columns)
    X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test = df_to_tensors(df_test, all_columns)
    
    CONFIG["input_features"] = X_time_train.shape[2] # Set dynamically
    print(f"Number of input time-series features: {CONFIG['input_features']}")
    
    return (X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train,
            X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test,
            num_positive_aki_train, num_negative_aki_train)

(X_tab_train_full, X_time_train_full, seq_len_train_full, y_train_full, y_binary_train_full,
 X_tab_test_full, X_time_test_full, seq_len_test_full, y_test_full, y_binary_test_full,
 num_pos_train_full, num_neg_train_full) = prepare_data(
    df_full, CONFIG["test_split_size"], CONFIG["random_seed"]
)
lstm_input_dim_global = CONFIG["input_features"]


# %%
# --- Utility Functions ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["random_seed"])

# (save_hpo_trial_results function can be kept if you want to save individual trial outputs,
#  otherwise, Optuna's study.trials_dataframe() is the main HPO result)

# %%
# --- Model Definition (Using EnhancedLSTM, configured as plain LSTM via flags) ---
# SequenceSEBlock and Attention classes are defined but won't be instantiated or used
# by EnhancedLSTM if the corresponding flags in fixed_config are False.

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.W = nn.Linear(hidden_dim, hidden_dim)
        self.V = nn.Linear(hidden_dim, 1, bias=False)
    def forward(self, lstm_outputs):
        e = torch.tanh(self.W(lstm_outputs))
        e = self.V(e).squeeze(2); alpha = F.softmax(e, dim=1)
        context = torch.bmm(alpha.unsqueeze(1), lstm_outputs).squeeze(1)
        return context, alpha

class SequenceSEBlock(nn.Module):
    def __init__(self, channels, reduction_ratio=16):
        super(SequenceSEBlock, self).__init__()
        self.channels = channels; self.reduction_ratio = reduction_ratio
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, max(1, channels // reduction_ratio), bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(max(1, channels // reduction_ratio), channels, bias=False),
            nn.Sigmoid())
    def forward(self, x):
        b, s, c = x.shape; y = x.permute(0, 2, 1); y = self.avg_pool(y); y = y.squeeze(-1)
        y = self.fc(y); return x * y.unsqueeze(1)

class EnhancedLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes,
                 bidirectional=True, use_attention=True, num_attention_heads=4,
                 dropout_rate=0.2, use_se_block=False, se_reduction_ratio=16):
        super(EnhancedLSTM, self).__init__()
        self.input_dim = input_dim; self.hidden_dim = hidden_dim; self.num_layers = num_layers
        self.num_classes = num_classes; self.bidirectional = bidirectional
        self.use_attention = use_attention; self.use_se_block = use_se_block

        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=bidirectional,
                            dropout=dropout_rate if num_layers > 1 else 0)
        
        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        
        self.se_block_instance = None
        if self.use_se_block:
            self.se_block_instance = SequenceSEBlock(channels=lstm_output_dim, reduction_ratio=se_reduction_ratio)
        
        self.attention_instance = None
        if self.use_attention:
            if lstm_output_dim % num_attention_heads != 0 and num_attention_heads > 0 : # Check num_attention_heads > 0
                print(f"Warning: lstm_output_dim ({lstm_output_dim}) is not divisible by num_attention_heads ({num_attention_heads}). Adjust for nn.MultiheadAttention if used.")
            # For this plain LSTM HPO, use_attention will be False, so this won't be created.
            # If it were True for a more complex model, MultiHeadAttention would be here.
            # self.attention_instance = nn.MultiheadAttention(...) 
            fc_input_dim = lstm_output_dim
        else:
            fc_input_dim = lstm_output_dim # For plain LSTM, this is just hidden_dim
            
        self.fc_intermediate_dim = max(1, fc_input_dim // 2)
        self.fc0 = nn.Linear(fc_input_dim, self.fc_intermediate_dim)
        self.relu_fc = nn.ReLU(); self.dropout_fc = nn.Dropout(dropout_rate)
        self.fc_final = nn.Linear(self.fc_intermediate_dim, num_classes)

    def forward(self, x_time, seq_lens):
        packed_input = pack_padded_sequence(x_time, seq_lens.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, (hn, cn) = self.lstm(packed_input)
        
        if self.use_attention and self.attention_instance is not None:
            # This path is NOT taken for plain LSTM
            lstm_sequence_output, _ = pad_packed_sequence(packed_output, batch_first=True)
            processed_sequence_output = lstm_sequence_output
            if self.use_se_block and self.se_block_instance is not None:
                processed_sequence_output = self.se_block_instance(processed_sequence_output)
            # ... (attention logic would be here) ...
            # fc_input = ... (from attention)
            pass # Placeholder, as this branch shouldn't be hit with use_attention=False
        else: # This path WILL BE USED for plain LSTM (bidirectional=False, use_attention=False)
            # hn shape: (num_layers * num_directions, batch, hidden_size)
            # For plain LSTM: (1 * 1, batch, hidden_size)
            fc_input = hn.squeeze(0) # if num_layers=1 and unidirectional, shape becomes (batch, hidden_size)
            # If you allow num_layers > 1 for "plain LSTM HPO" (stacked unidirectional):
            # fc_input = hn[-1,:,:] # Takes the hidden state from the last layer

        out = self.fc0(fc_input); out = self.relu_fc(out); out = self.dropout_fc(out)
        out = self.fc_final(out)
        return out

# %%
# --- Optuna Objective Function ---
def objective(trial, fixed_config, data_globals):
    cfg_trial = {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True),
        "lstm_hidden_dim": trial.suggest_categorical("lstm_hidden_dim", [32, 64, 128, 256, 512]),
        # "num_lstm_layers": trial.suggest_int("num_lstm_layers", 1, 2), # Kept fixed at 1 for "plain" LSTM via fixed_config
        "dropout_rate": trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1), # For FC layers
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        # SE and Attention params are not tuned as they are disabled by fixed_config
    }

    print(f"\n--- Trial {trial.number} ---")
    print(f"Hyperparameters for Plain LSTM: {cfg_trial}")

    model = EnhancedLSTM(
        input_dim=data_globals["lstm_input_dim"],
        hidden_dim=cfg_trial["lstm_hidden_dim"],
        num_layers=fixed_config["num_lstm_layers"], # Fixed to 1 for plain LSTM
        num_classes=1,
        bidirectional=fixed_config["bidirectional"], # Fixed to False
        use_attention=fixed_config["use_attention"],   # Fixed to False
        num_attention_heads=fixed_config["num_attention_heads"], # Not used
        dropout_rate=cfg_trial["dropout_rate"],
        use_se_block=fixed_config["use_se_block"],     # Fixed to False
        se_reduction_ratio=fixed_config["se_reduction_ratio"] # Not used
    ).to(device)

    if data_globals["num_pos_train"] > 0:
        pos_weight_value = data_globals["num_neg_train"] / data_globals["num_pos_train"]
    else:
        pos_weight_value = 1.0
    pos_weight = torch.tensor([pos_weight_value], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=cfg_trial["learning_rate"], weight_decay=cfg_trial["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=fixed_config["lr_scheduler_factor"],
        patience=fixed_config["lr_scheduler_patience"], verbose=False
    )

    num_train_samples = data_globals["X_time_train_full"].shape[0]
    train_batch_size = fixed_config["batch_size"]
    num_train_batches = int(np.ceil(num_train_samples / train_batch_size))
    eval_batch_size = fixed_config["eval_batch_size"]
    num_test_samples = data_globals["X_time_test_full"].shape[0]
    num_eval_batches = int(np.ceil(num_test_samples / eval_batch_size))

    best_val_loss_for_trial = float('inf')
    epochs_no_improve_for_trial = 0
    
    for epoch in range(fixed_config["num_epochs_hpo"]):
        model.train()
        epoch_train_loss = 0.0
        for i in range(num_train_batches):
            start_idx = i * train_batch_size
            end_idx = min(num_train_samples, (i + 1) * train_batch_size)
            batch_x_time = data_globals["X_time_train_full"][start_idx:end_idx].to(device)
            batch_seq_len = data_globals["seq_len_train_full"][start_idx:end_idx]
            batch_y_binary = torch.tensor(data_globals["y_binary_train_full"][start_idx:end_idx]).unsqueeze(1).float().to(device)
            optimizer.zero_grad(); outputs = model(batch_x_time, batch_seq_len); loss = criterion(outputs, batch_y_binary)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=fixed_config["gradient_clip_value"]); optimizer.step()
            epoch_train_loss += loss.item()
        
        model.eval()
        all_val_outputs_logits = []; all_val_y_for_loss = []
        with torch.no_grad():
            for i in range(num_eval_batches):
                start_idx = i * eval_batch_size; end_idx = min(num_test_samples, (i + 1) * eval_batch_size)
                batch_x_time_val = data_globals["X_time_test_full"][start_idx:end_idx].to(device)
                batch_seq_len_val = data_globals["seq_len_test_full"][start_idx:end_idx]
                batch_y_binary_val_for_loss = torch.tensor(data_globals["y_binary_test_full"][start_idx:end_idx]).unsqueeze(1).float().to(device)
                outputs_batch_logits = model(batch_x_time_val, batch_seq_len_val)
                all_val_outputs_logits.append(outputs_batch_logits.cpu()); all_val_y_for_loss.append(batch_y_binary_val_for_loss.cpu())
        val_outputs_logits = torch.cat(all_val_outputs_logits, dim=0); val_y_for_loss = torch.cat(all_val_y_for_loss, dim=0)
        current_val_loss = criterion(val_outputs_logits.to(device), val_y_for_loss.to(device)).item()

        scheduler.step(current_val_loss)

        if current_val_loss < best_val_loss_for_trial:
            best_val_loss_for_trial = current_val_loss
            epochs_no_improve_for_trial = 0
        else:
            epochs_no_improve_for_trial += 1
        
        if epochs_no_improve_for_trial >= fixed_config["early_stopping_patience"]:
            print(f"Trial {trial.number} early stopping at epoch {epoch+1} (val_loss: {current_val_loss:.4f})")
            break
        
        trial.report(current_val_loss, epoch)
        if trial.should_prune():
            print(f"Trial {trial.number} pruned at epoch {epoch+1} (val_loss: {current_val_loss:.4f}).")
            raise optuna.exceptions.TrialPruned()
        
        if (epoch + 1) % 10 == 0:
             print(f"Trial {trial.number}, Epoch [{epoch+1}/{fixed_config['num_epochs_hpo']}], Train Loss: {(epoch_train_loss / num_train_batches):.4f}, Val Loss: {current_val_loss:.4f}")

    val_probs = torch.sigmoid(val_outputs_logits).numpy().squeeze()
    if val_probs.ndim == 0: val_probs = np.array([val_probs])
    val_pred_binary = (val_probs > fixed_config["classification_threshold"])
    current_y_true_for_metric = data_globals["y_binary_test_full"][:len(val_probs)]
    try:
        dic = performance_dict(current_y_true_for_metric, val_pred_binary, val_probs)
        val_auprc = dic.get('pr_auc', 0.0)
    except Exception as e:
        print(f"Warning: Could not compute performance_dict for trial {trial.number}: {e}")
        val_auprc = 0.0

    print(f"Trial {trial.number} finished. Best Val Loss: {best_val_loss_for_trial:.4f}, Final Val AUPRC: {val_auprc:.4f}")
    return best_val_loss_for_trial

# %%
# --- HPO Study Execution ---
if __name__ == "__main__":
    data_globals_for_objective = {
        "lstm_input_dim": lstm_input_dim_global,
        "num_pos_train": num_pos_train_full,
        "num_neg_train": num_neg_train_full,
        "X_time_train_full": X_time_train_full,
        "seq_len_train_full": seq_len_train_full,
        "y_binary_train_full": y_binary_train_full,
        "X_time_test_full": X_time_test_full,
        "seq_len_test_full": seq_len_test_full,
        "y_binary_test_full": y_binary_test_full,
    }

    os.makedirs(CONFIG["results_output_file_base"], exist_ok=True)
    os.makedirs(CONFIG["tensorboard_log_dir_base"], exist_ok=True) # Not actively used in objective currently
    os.makedirs(CONFIG["best_model_save_path_base"], exist_ok=True) # Not actively used in objective

    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=CONFIG["early_stopping_patience"]//2, interval_steps=1) # Adjusted n_warmup_steps
    study = optuna.create_study(direction="minimize", pruner=pruner)

    objective_with_config = functools.partial(objective, fixed_config=CONFIG, data_globals=data_globals_for_objective)

    print(f"Starting HPO study for PLAIN LSTM with {CONFIG['n_hpo_trials']} trials...")
    try:
        study.optimize(objective_with_config, n_trials=CONFIG['n_hpo_trials'])
    except KeyboardInterrupt:
        print("HPO study interrupted by user.")
    except Exception as e:
        print(f"An error occurred during HPO study: {e}")
        import traceback
        traceback.print_exc()

    print("\n--- HPO Study Finished ---")
    print(f"Number of finished trials: {len(study.trials)}")
    
    if len(study.trials) > 0 and hasattr(study, 'best_trial') and study.best_trial is not None:
        best_trial = study.best_trial
        print("\nBest trial for Plain LSTM:")
        print(f"  Value (Best Validation Loss): {best_trial.value:.4f}")
        print("  Params: ")
        for key, value in best_trial.params.items():
            print(f"    {key}: {value}")
        best_params_path = os.path.join(CONFIG["results_output_file_base"], "best_plain_lstm_hpo_params.json")
        import json
        with open(best_params_path, 'w') as f:
            json.dump(best_trial.params, f, indent=4)
        print(f"Best hyperparameters for Plain LSTM saved to {best_params_path}")
    else:
        print("No successful trials completed or no best trial found.")

    print("\nTo train a final Plain LSTM model with these best hyperparameters, update your CONFIG dict,")
    print("set num_epochs to a higher value (e.g., original 100), and run a standard (non-HPO) training script.")

    study_results_df = study.trials_dataframe()
    study_results_path = os.path.join(CONFIG["results_output_file_base"], "hpo_plain_lstm_study_results.csv")
    study_results_df.to_csv(study_results_path, index=False)
    print(f"Full HPO study results for Plain LSTM saved to {study_results_path}")

print("Script finished.")

Using device: cuda
Original train size: 43271
Test size: 10818
Training set: Positive AKI cases: 2595.0, Negative AKI cases: 40676.0


[I 2025-05-24 15:51:05,440] A new study created in memory with name: no-name-99285ceb-0027-4bc7-99f4-1a5f9142ab39


Number of input time-series features: 24
Starting HPO study for PLAIN LSTM with 100 trials...

--- Trial 0 ---
Hyperparameters for Plain LSTM: {'learning_rate': 0.008827739275315554, 'lstm_hidden_dim': 512, 'dropout_rate': 0.5, 'weight_decay': 3.7684020364645793e-06}


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Trial 0, Epoch [10/50], Train Loss: 1.3024, Val Loss: 1.2959
Trial 0, Epoch [20/50], Train Loss: 1.1555, Val Loss: 1.1589


...

Trial 97 early stopping at epoch 16 (val_loss: 1.3070)
Trial 97 finished. Best Val Loss: 1.1348, Final Val AUPRC: 0.1950

--- Trial 98 ---
Hyperparameters for Plain LSTM: {'learning_rate': 0.003780794577994938, 'lstm_hidden_dim': 512, 'dropout_rate': 0.4, 'weight_decay': 1.7759252103724518e-06}


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
[I 2025-05-24 16:53:00,586] Trial 98 pruned. 


Trial 98 pruned at epoch 6 (val_loss: 1.1618).

--- Trial 99 ---
Hyperparameters for Plain LSTM: {'learning_rate': 0.0009670316472367341, 'lstm_hidden_dim': 32, 'dropout_rate': 0.2, 'weight_decay': 2.4893953673342088e-06}


[I 2025-05-24 16:53:08,779] Trial 99 pruned. 


Trial 99 pruned at epoch 6 (val_loss: 1.1622).

--- HPO Study Finished ---
Number of finished trials: 100

Best trial for Plain LSTM:
  Value (Best Validation Loss): 1.1224
  Params: 
    learning_rate: 0.0012395393822216506
    lstm_hidden_dim: 64
    dropout_rate: 0.2
    weight_decay: 2.4834455388189246e-06
Best hyperparameters for Plain LSTM saved to /home/server/Projects/data/AKI/results/hpo_plain_lstm/best_plain_lstm_hpo_params.json

To train a final Plain LSTM model with these best hyperparameters, update your CONFIG dict,
set num_epochs to a higher value (e.g., original 100), and run a standard (non-HPO) training script.
Full HPO study results for Plain LSTM saved to /home/server/Projects/data/AKI/results/hpo_plain_lstm/hpo_plain_lstm_study_results.csv
Script finished.


# HPO BiLSTM + SAttention

Testing b/c this combo had best AUPRC

In [ ]:
# %%
import os
import numpy as np
import torch
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
import torch.optim as optim
import optuna
import functools
import math

import random
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_curve, auc, precision_recall_curve, average_precision_score)
from sklearn.model_selection import train_test_split
import pandas as pd
# Ensure utils.py is in the same directory or Python path
from utils import performance_dict, sigmoid
from tqdm import tqdm

# --- Configuration ---
CONFIG = {
    "data_file": '/home/server/Projects/data/AKI/lstm_trainable.pkl', # !!! UPDATE THIS PATH !!!
    "results_output_file_base": '/home/server/Projects/data/AKI/results/hpo_bilstm_attention/',
    "tensorboard_log_dir_base": '/home/server/Projects/data/AKI/runs/hpo_bilstm_attention/',
    # "best_model_save_path_base": '/home/server/Projects/data/AKI/models/hpo_bilstm_attention/', # Optional
    "test_split_size": 0.2,
    "random_seed": 42,
    "num_epochs_hpo": 50,
    "gradient_clip_value": 1.0,
    "lr_scheduler_patience": 5,
    "lr_scheduler_factor": 0.2,
    "early_stopping_patience": 10,
    
    # --- Fixed Architectural Choices for BiLSTM + Single Head Attention HPO ---
    "num_lstm_layers": 1,       # Single BiLSTM layer
    "bidirectional": True,      # Bidirectional
    "use_attention": True,      # Enable Single Head Attention (custom class)
    "attention_dim": None,      # Not directly used by custom Attention if it infers from lstm_output_dim
    "num_attention_heads": 0,   # Placeholder, not used for custom single head attention
    "use_se_block": False,      # No SE Block
    "se_reduction_ratio": 16,   # Not used
    
    "batch_size": 512,          # Adjusted batch size for potentially larger BiLSTM models
    "eval_batch_size": 512,
    "classification_threshold": 0.3,
    "n_hpo_trials": 50, 
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load data (once)
try:
    df_full = pd.read_pickle(CONFIG["data_file"])
except FileNotFoundError:
    print(f"ERROR: Data file not found at {CONFIG['data_file']}")
    print("Please ensure the path in CONFIG['data_file'] is correct.")
    exit()
except Exception as e:
    print(f"Error loading data file {CONFIG['data_file']}: {e}")
    exit()


# %%
# --- Data Preparation ---
def df_to_tensors(df, all_columns):
    non_tabular_cols = ['op_id', 'time_tensors', 'seq_len', 'aki', 'aki_boolean']
    tabular_feature_cols = [col for col in all_columns if col not in non_tabular_cols]
    if tabular_feature_cols and not df[tabular_feature_cols].empty:
        X_tab = torch.tensor(df[tabular_feature_cols].values).float()
    else:
        X_tab = torch.empty((len(df), 0)).float()
    X_time = torch.stack(df['time_tensors'].tolist()).float()
    y = torch.tensor(df['aki'].values).unsqueeze(1).float()
    y_binary = df['aki_boolean'].values.astype(bool)
    sequence_lengths = torch.tensor(df['seq_len'].tolist()).long()
    return X_tab, X_time, sequence_lengths, y, y_binary

def prepare_data(full_df, test_size, random_seed):
    df_train, df_test = train_test_split(full_df, test_size=test_size, random_state=random_seed, stratify=full_df['aki_boolean'])
    print(f"Original train size: {len(df_train)}")
    print(f"Test size: {len(df_test)}")
    all_columns = full_df.columns.tolist()
    num_positive_aki_train = df_train['aki_boolean'].sum()
    num_negative_aki_train = len(df_train) - num_positive_aki_train
    print(f"Training set: Positive AKI cases: {num_positive_aki_train}, Negative AKI cases: {num_negative_aki_train}")
    X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train = df_to_tensors(df_train, all_columns)
    X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test = df_to_tensors(df_test, all_columns)
    
    CONFIG["input_features"] = X_time_train.shape[2]
    print(f"Number of input time-series features: {CONFIG['input_features']}")
    
    return (X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train,
            X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test,
            num_positive_aki_train, num_negative_aki_train)

(X_tab_train_full, X_time_train_full, seq_len_train_full, y_train_full, y_binary_train_full,
 X_tab_test_full, X_time_test_full, seq_len_test_full, y_test_full, y_binary_test_full,
 num_pos_train_full, num_neg_train_full) = prepare_data(
    df_full, CONFIG["test_split_size"], CONFIG["random_seed"]
)
input_features_global = CONFIG["input_features"]


# %%
# --- Utility Functions ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["random_seed"])

# (save_hpo_trial_results can be kept if you want to save individual trial outputs)

# %%
# --- Model Definition (EnhancedLSTM with custom Attention and optional SE) ---
class Attention(nn.Module): # Your custom single-head attention
    def __init__(self, hidden_dim): # Takes the full dimension of BiLSTM output
        super(Attention, self).__init__()
        self.W = nn.Linear(hidden_dim, hidden_dim)
        self.V = nn.Linear(hidden_dim, 1, bias=False)
    def forward(self, lstm_outputs): # lstm_outputs shape: (batch, seq_len, hidden_dim_from_bilstm)
        e = torch.tanh(self.W(lstm_outputs))
        e = self.V(e).squeeze(2); alpha = F.softmax(e, dim=1)
        context = torch.bmm(alpha.unsqueeze(1), lstm_outputs).squeeze(1)
        return context, alpha

class SequenceSEBlock(nn.Module):
    def __init__(self, channels, reduction_ratio=16):
        super(SequenceSEBlock, self).__init__()
        self.channels = channels; self.reduction_ratio = reduction_ratio
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, max(1, channels // reduction_ratio), bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(max(1, channels // reduction_ratio), channels, bias=False),
            nn.Sigmoid())
    def forward(self, x):
        b, s, c = x.shape; y = x.permute(0, 2, 1); y = self.avg_pool(y); y = y.squeeze(-1)
        y = self.fc(y); return x * y.unsqueeze(1)

class EnhancedLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes,
                 bidirectional=True, use_attention=True, attention_dim=None, # attention_dim not used by custom Attention
                 num_attention_heads=0, # Not used by custom single head Attention
                 dropout_rate=0.2, use_se_block=False, se_reduction_ratio=16):
        super(EnhancedLSTM, self).__init__()
        self.input_dim = input_dim; self.hidden_dim = hidden_dim; self.num_layers = num_layers
        self.num_classes = num_classes; self.bidirectional = bidirectional
        self.use_attention = use_attention; self.use_se_block = use_se_block

        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=bidirectional,
                            dropout=dropout_rate if num_layers > 1 else 0)
        
        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        
        self.se_block_instance = None
        if self.use_se_block:
            self.se_block_instance = SequenceSEBlock(channels=lstm_output_dim, reduction_ratio=se_reduction_ratio)
        
        self.attention_instance = None
        if self.use_attention:
            self.attention_instance = Attention(lstm_output_dim) # Using your custom Attention class
            fc_input_dim = lstm_output_dim # Attention output is lstm_output_dim
        else:
            fc_input_dim = lstm_output_dim
            
        self.fc_intermediate_dim = max(1, fc_input_dim // 2)
        self.fc0 = nn.Linear(fc_input_dim, self.fc_intermediate_dim)
        self.relu_fc = nn.ReLU(); self.dropout_fc = nn.Dropout(dropout_rate)
        self.fc_final = nn.Linear(self.fc_intermediate_dim, num_classes)

    def forward(self, x_time, seq_lens):
        packed_input = pack_padded_sequence(x_time, seq_lens.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, (hn, cn) = self.lstm(packed_input)
        lstm_sequence_output, _ = pad_packed_sequence(packed_output, batch_first=True)
        
        processed_sequence_output = lstm_sequence_output
        if self.use_se_block and self.se_block_instance is not None:
            processed_sequence_output = self.se_block_instance(processed_sequence_output)
            
        if self.use_attention and self.attention_instance is not None:
            context_vector, _ = self.attention_instance(processed_sequence_output)
            fc_input = context_vector
        else: 
            if self.bidirectional:
                if self.num_layers == 1:
                    fc_input = torch.cat((hn[0,:,:], hn[1,:,:]), dim=1)
                else: 
                    fc_input = torch.cat((hn[-2,:,:], hn[-1,:,:]), dim=1)
            else: 
                fc_input = hn.squeeze(0) if self.num_layers == 1 else hn[-1,:,:]
                
        out = self.fc0(fc_input); out = self.relu_fc(out); out = self.dropout_fc(out)
        out = self.fc_final(out)
        return out

# %%
# --- Optuna Objective Function for BiLSTM + Single Head Attention ---
def objective(trial, fixed_config, data_globals):
    cfg_trial = {
        "learning_rate": trial.suggest_float("learning_rate", 5e-5, 1e-3, log=True), # Adjusted based on plain LSTM results
        "lstm_hidden_dim": trial.suggest_categorical("lstm_hidden_dim", [32, 48, 64, 96]), # Per direction
        # "num_lstm_layers": fixed_config["num_lstm_layers"], # Kept fixed at 1 for this specific HPO run
        "dropout_rate": trial.suggest_float("dropout_rate", 0.1, 0.4, step=0.05),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-4, log=True),
        # attention_dim for the custom Attention class is implicitly 2*lstm_hidden_dim
        # se_reduction_ratio is not tuned as use_se_block is False
    }

    print(f"\n--- Trial {trial.number} Starting ---")
    print(f"Hyperparameters for BiLSTM+Attention: {cfg_trial}")

    try:
        model = EnhancedLSTM(
            input_dim=data_globals["input_features"],
            hidden_dim=cfg_trial["lstm_hidden_dim"],
            num_layers=fixed_config["num_lstm_layers"], # Fixed (e.g., 1 for single BiLSTM layer)
            num_classes=1,
            bidirectional=fixed_config["bidirectional"], # Fixed True
            use_attention=fixed_config["use_attention"],   # Fixed True
            attention_dim=fixed_config["attention_dim"], # Not directly used by this custom Attention
            num_attention_heads=fixed_config["num_attention_heads"], # Not used
            dropout_rate=cfg_trial["dropout_rate"],
            use_se_block=fixed_config["use_se_block"],     # Fixed False
            se_reduction_ratio=fixed_config["se_reduction_ratio"] # Not used
        ).to(device)

        if data_globals["num_pos_train"] > 0:
            pos_weight_value = data_globals["num_neg_train"] / data_globals["num_pos_train"]
        else: pos_weight_value = 1.0
        pos_weight = torch.tensor([pos_weight_value], device=device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = optim.Adam(model.parameters(), lr=cfg_trial["learning_rate"], weight_decay=cfg_trial["weight_decay"])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=fixed_config["lr_scheduler_factor"],
            patience=fixed_config["lr_scheduler_patience"], verbose=False )

        num_train_samples = data_globals["X_time_train_full"].shape[0]
        train_batch_size = fixed_config["batch_size"]
        num_train_batches = int(np.ceil(num_train_samples / train_batch_size))
        eval_batch_size = fixed_config["eval_batch_size"]
        num_test_samples = data_globals["X_time_test_full"].shape[0]
        num_eval_batches = int(np.ceil(num_test_samples / eval_batch_size))

        best_val_loss_for_trial = float('inf')
        epochs_no_improve_for_trial = 0
        
        for epoch in range(fixed_config["num_epochs_hpo"]):
            model.train()
            epoch_train_loss = 0.0
            for i in range(num_train_batches):
                start_idx = i * train_batch_size; end_idx = min(num_train_samples, (i + 1) * train_batch_size)
                batch_x_time = data_globals["X_time_train_full"][start_idx:end_idx].to(device)
                batch_seq_len = data_globals["seq_len_train_full"][start_idx:end_idx]
                batch_y_binary = torch.tensor(data_globals["y_binary_train_full"][start_idx:end_idx]).unsqueeze(1).float().to(device)
                optimizer.zero_grad(); outputs = model(batch_x_time, batch_seq_len); loss = criterion(outputs, batch_y_binary)
                loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=fixed_config["gradient_clip_value"]); optimizer.step()
                epoch_train_loss += loss.item()
            avg_epoch_train_loss = epoch_train_loss / num_train_batches if num_train_batches > 0 else 0.0

            model.eval()
            all_val_outputs_logits = []; all_val_y_for_loss = []
            with torch.no_grad():
                for i in range(num_eval_batches):
                    start_idx = i * eval_batch_size; end_idx = min(num_test_samples, (i + 1) * eval_batch_size)
                    batch_x_time_val = data_globals["X_time_test_full"][start_idx:end_idx].to(device)
                    batch_seq_len_val = data_globals["seq_len_test_full"][start_idx:end_idx]
                    batch_y_binary_val_for_loss = torch.tensor(data_globals["y_binary_test_full"][start_idx:end_idx]).unsqueeze(1).float().to(device)
                    outputs_batch_logits = model(batch_x_time_val, batch_seq_len_val)
                    all_val_outputs_logits.append(outputs_batch_logits.cpu()); all_val_y_for_loss.append(batch_y_binary_val_for_loss.cpu())
            
            if not all_val_outputs_logits: current_val_loss = float('inf')
            else:
                val_outputs_logits = torch.cat(all_val_outputs_logits, dim=0); val_y_for_loss = torch.cat(all_val_y_for_loss, dim=0)
                current_val_loss = criterion(val_outputs_logits.to(device), val_y_for_loss.to(device)).item()

            scheduler.step(current_val_loss)
            if current_val_loss < best_val_loss_for_trial:
                best_val_loss_for_trial = current_val_loss; epochs_no_improve_for_trial = 0
            else: epochs_no_improve_for_trial += 1
            if epochs_no_improve_for_trial >= fixed_config["early_stopping_patience"]:
                print(f"Trial {trial.number} early stopping at epoch {epoch+1} (val_loss: {current_val_loss:.4f})")
                break
            trial.report(current_val_loss, epoch)
            if trial.should_prune():
                print(f"Trial {trial.number} pruned at epoch {epoch+1} (val_loss: {current_val_loss:.4f}).")
                raise optuna.exceptions.TrialPruned()
            if (epoch + 1) % 10 == 0 or epoch == fixed_config["num_epochs_hpo"] -1 :
                 print(f"Trial {trial.number}, Epoch [{epoch+1}/{fixed_config['num_epochs_hpo']}], Train Loss: {avg_epoch_train_loss:.4f}, Val Loss: {current_val_loss:.4f}")
        
        if 'val_outputs_logits' in locals() and val_outputs_logits is not None and len(val_outputs_logits) > 0:
            val_probs = torch.sigmoid(val_outputs_logits).numpy().squeeze()
            if val_probs.ndim == 0: val_probs = np.array([val_probs])
            val_pred_binary = (val_probs > fixed_config["classification_threshold"])
            current_y_true_for_metric = data_globals["y_binary_test_full"][:len(val_probs)]
            try:
                dic = performance_dict(current_y_true_for_metric, val_pred_binary, val_probs)
                val_auprc = dic.get('pr_auc', 0.0)
            except Exception as e:
                print(f"Warning: Could not compute performance_dict for trial {trial.number} final metrics: {e}"); val_auprc = 0.0
            print(f"Trial {trial.number} finished. Best Val Loss: {best_val_loss_for_trial:.4f}, Final Epoch Val AUPRC: {val_auprc:.4f}")
        else:
             print(f"Trial {trial.number} finished. Best Val Loss: {best_val_loss_for_trial:.4f} (No validation outputs for AUPRC).")
        return best_val_loss_for_trial

    except torch.cuda.OutOfMemoryError:
        print(f"Trial {trial.number} failed due to CUDA OutOfMemoryError. Pruning trial.")
        torch.cuda.empty_cache(); raise optuna.exceptions.TrialPruned()
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"Trial {trial.number} failed due to RuntimeError (OOM). Pruning trial. Error: {e}")
            torch.cuda.empty_cache(); raise optuna.exceptions.TrialPruned()
        else: print(f"Trial {trial.number} failed with other RuntimeError: {e}"); import traceback; traceback.print_exc(); raise

# %%
# --- HPO Study Execution ---
if __name__ == "__main__":
    data_globals_for_objective = {
        "input_features": input_features_global,
        "num_pos_train": num_pos_train_full,
        "num_neg_train": num_neg_train_full,
        "X_time_train_full": X_time_train_full,
        "seq_len_train_full": seq_len_train_full,
        "y_binary_train_full": y_binary_train_full,
        "X_time_test_full": X_time_test_full,
        "seq_len_test_full": seq_len_test_full,
        "y_binary_test_full": y_binary_test_full,
    }
    os.makedirs(CONFIG["results_output_file_base"], exist_ok=True)
    os.makedirs(CONFIG["tensorboard_log_dir_base"], exist_ok=True)

    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=max(1, CONFIG["early_stopping_patience"]//2), interval_steps=1)
    study = optuna.create_study(direction="minimize", pruner=pruner)
    objective_with_config = functools.partial(objective, fixed_config=CONFIG, data_globals=data_globals_for_objective)

    print(f"Starting HPO study for BiLSTM + Attention with {CONFIG['n_hpo_trials']} trials...")
    try:
        study.optimize(objective_with_config, n_trials=CONFIG['n_hpo_trials'])
    except KeyboardInterrupt: print("HPO study interrupted by user.")
    except Exception as e: print(f"An error occurred during HPO study: {e}"); import traceback; traceback.print_exc()

    print("\n--- HPO Study Finished ---")
    print(f"Number of finished trials: {len(study.trials)}")
    if len(study.trials) > 0 and hasattr(study, 'best_trial') and study.best_trial is not None:
        best_trial = study.best_trial
        print("\nBest trial for BiLSTM + Attention:")
        print(f"  Value (Best Validation Loss): {best_trial.value:.4f}")
        print("  Params: "); [print(f"    {key}: {value}") for key, value in best_trial.params.items()]
        best_params_path = os.path.join(CONFIG["results_output_file_base"], "best_bilstm_attention_hpo_params.json")
        import json
        with open(best_params_path, 'w') as f: json.dump(best_trial.params, f, indent=4)
        print(f"Best hyperparameters for BiLSTM + Attention saved to {best_params_path}")
    else: print("No successful trials completed or no best trial found.")
    print("\nTo train a final BiLSTM + Attention model with these best hyperparameters, update your CONFIG dict,")
    print("set num_epochs to a higher value, and run a standard (non-HPO) training script.")
    study_results_df = study.trials_dataframe()
    study_results_path = os.path.join(CONFIG["results_output_file_base"], "hpo_bilstm_attention_study_results.csv")
    study_results_df.to_csv(study_results_path, index=False)
    print(f"Full HPO study results for BiLSTM + Attention saved to {study_results_path}")

print("Script finished.")

Using device: cuda
Original train size: 43271
Test size: 10818
Training set: Positive AKI cases: 2595.0, Negative AKI cases: 40676.0


[I 2025-05-25 16:34:04,679] A new study created in memory with name: no-name-970e2de1-9709-4fa6-aa2c-a2e147797de7


Number of input time-series features: 24
Starting HPO study for BiLSTM + Attention with 50 trials...

--- Trial 0 Starting ---
Hyperparameters for BiLSTM+Attention: {'learning_rate': 0.0002079531119358334, 'lstm_hidden_dim': 96, 'dropout_rate': 0.4, 'weight_decay': 2.147044725057583e-06}


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Trial 0, Epoch [10/50], Train Loss: 1.0807, Val Loss: 1.1356
Trial 0, Epoch [20/50], Train Loss: 0.9935, Val Loss: 1.1700


...

[I 2025-05-25 18:42:24,206] Trial 24 pruned. 


Trial 24 pruned at epoch 6 (val_loss: 1.1449).

--- Trial 25 Starting ---
Hyperparameters for BiLSTM+Attention: {'learning_rate': 0.0005188200431264049, 'lstm_hidden_dim': 48, 'dropout_rate': 0.2, 'weight_decay': 1.129024349582005e-05}
Trial 25, Epoch [10/50], Train Loss: 1.0323, Val Loss: 1.1376


[I 2025-05-25 18:48:02,446] Trial 25 finished with value: 1.1316981315612793 and parameters: {'learning_rate': 0.0005188200431264049, 'lstm_hidden_dim': 48, 'dropout_rate': 0.2, 'weight_decay': 1.129024349582005e-05}. Best is trial 12 with value: 1.1215479373931885.


Trial 25 early stopping at epoch 17 (val_loss: 1.1975)
Trial 25 finished. Best Val Loss: 1.1317, Final Epoch Val AUPRC: 0.2147

--- Trial 26 Starting ---
Hyperparameters for BiLSTM+Attention: {'learning_rate': 0.0002545714256852133, 'lstm_hidden_dim': 48, 'dropout_rate': 0.15000000000000002, 'weight_decay': 3.2998899833624704e-05}


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


# HPO BiLSTM + SAttention + SE

In [4]:
# %%
import os
import numpy as np
import torch
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
import torch.optim as optim
import optuna # <<< --- Optuna import --- >>>
import functools # For passing arguments to objective function

import random
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_curve, auc, precision_recall_curve, average_precision_score)
# Not strictly needed for this script if not plotting here:
# import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pandas as pd
# Ensure utils.py is in the same directory or Python path
from utils import performance_dict, sigmoid
from tqdm import tqdm

# --- Configuration ---
CONFIG = {
    "data_file": '/home/server/Projects/data/AKI/lstm_trainable.pkl', # !!! UPDATE THIS PATH !!!
    "results_output_file_base": '/home/server/Projects/data/AKI/results/hpo_lstm/', # Base for HPO results
    "tensorboard_log_dir_base": '/home/server/Projects/data/AKI/runs/hpo_lstm/',     # Base for TensorBoard logs
    "best_model_save_path_base": '/home/server/Projects/data/AKI/models/hpo_lstm/', # Base for best models (if saving per trial)
    "test_split_size": 0.2,
    "random_seed": 42,
    "num_epochs_hpo": 50,  # Max epochs PER HPO TRIAL (early stopping will often trigger sooner)
    "gradient_clip_value": 1.0,
    "lr_scheduler_patience": 5, 
    "lr_scheduler_factor": 0.2, 
    "early_stopping_patience": 10,
    # Fixed architectural choices for this HPO study (can change for another study)
    "bidirectional": True,
    "use_attention": True,
    "attention_dim": 32, # Fixed attention_dim; can be tuned if Attention class takes it or if using nn.MultiheadAttention
    "use_se_block": True, # Set to True to include SE blocks in the architecture being tuned
    # Default batch sizes (can also be made tunable in Optuna if desired)
    "batch_size": 1024,
    "eval_batch_size": 512,
    "classification_threshold": 0.3, # Primarily for final metrics after HPO, not for HPO objective typically
    "n_hpo_trials": 50, # Number of HPO trials to run
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load data (once)
try:
    df_full = pd.read_pickle(CONFIG["data_file"])
except FileNotFoundError:
    print(f"ERROR: Data file not found at {CONFIG['data_file']}")
    print("Please ensure the path in CONFIG['data_file'] is correct.")
    exit()
except Exception as e:
    print(f"Error loading data file {CONFIG['data_file']}: {e}")
    exit()


# %%
# --- Data Preparation ---
def df_to_tensors(df, all_columns):
    non_tabular_cols = ['op_id', 'time_tensors', 'seq_len', 'aki', 'aki_boolean']
    tabular_feature_cols = [col for col in all_columns if col not in non_tabular_cols]
    
    if tabular_feature_cols and not df[tabular_feature_cols].empty:
        X_tab = torch.tensor(df[tabular_feature_cols].values).float()
    else:
        X_tab = torch.empty((len(df), 0)).float()

    X_time = torch.stack(df['time_tensors'].tolist()).float()
    y = torch.tensor(df['aki'].values).unsqueeze(1).float()
    y_binary = df['aki_boolean'].values.astype(bool)
    sequence_lengths = torch.tensor(df['seq_len'].tolist()).long()
    
    return X_tab, X_time, sequence_lengths, y, y_binary

def prepare_data(full_df, test_size, random_seed):
    df_train, df_test = train_test_split(full_df,
                                         test_size=test_size,
                                         random_state=random_seed,
                                         stratify=full_df['aki_boolean'])
    
    print(f"Original train size: {len(df_train)}")
    print(f"Test size: {len(df_test)}")

    all_columns = full_df.columns.tolist()

    num_positive_aki_train = df_train['aki_boolean'].sum()
    num_negative_aki_train = len(df_train) - num_positive_aki_train
    
    print(f"Training set: Positive AKI cases: {num_positive_aki_train}, Negative AKI cases: {num_negative_aki_train}")

    X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train = df_to_tensors(df_train, all_columns)
    X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test = df_to_tensors(df_test, all_columns)
    
    return (X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train,
            X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test,
            num_positive_aki_train, num_negative_aki_train)

# --- Prepare data once before HPO study ---
(X_tab_train_full, X_time_train_full, seq_len_train_full, y_train_full, y_binary_train_full,
 X_tab_test_full, X_time_test_full, seq_len_test_full, y_test_full, y_binary_test_full,
 num_pos_train_full, num_neg_train_full) = prepare_data(
    df_full, CONFIG["test_split_size"], CONFIG["random_seed"]
)
lstm_input_dim_global = X_time_train_full.shape[2]


# %%
# --- Utility Functions ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["random_seed"]) # Set seed for main script operations like data splitting

def save_hpo_trial_results(trial_number, params, metrics_dict, output_file_base):
    results_to_save = {**params, **metrics_dict}
    df_result = pd.DataFrame([results_to_save])
    df_result['trial_number'] = trial_number
    
    output_file = os.path.join(output_file_base, f"trial_{trial_number}_results.pkl")
    # os.makedirs(os.path.dirname(output_file), exist_ok=True) # Not needed if base dir created once
    df_result.to_pickle(output_file)

# %%
# --- Model Definition ---
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.W = nn.Linear(hidden_dim, hidden_dim)
        self.V = nn.Linear(hidden_dim, 1, bias=False)
    def forward(self, lstm_outputs):
        e = torch.tanh(self.W(lstm_outputs))
        e = self.V(e).squeeze(2); alpha = F.softmax(e, dim=1)
        context = torch.bmm(alpha.unsqueeze(1), lstm_outputs).squeeze(1)
        return context, alpha

class SequenceSEBlock(nn.Module):
    def __init__(self, channels, reduction_ratio=16):
        super(SequenceSEBlock, self).__init__()
        self.channels = channels; self.reduction_ratio = reduction_ratio
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, max(1, channels // reduction_ratio), bias=False), # Ensure internal dim is at least 1
            nn.ReLU(inplace=True),
            nn.Linear(max(1, channels // reduction_ratio), channels, bias=False), 
            nn.Sigmoid())
    def forward(self, x): # x shape: (batch, seq_len, channels)
        b, s, c = x.shape
        y = x.permute(0, 2, 1) # (batch, channels, seq_len)
        y = self.avg_pool(y)   # (batch, channels, 1)
        y = y.squeeze(-1)      # (batch, channels)
        y = self.fc(y)         # (batch, channels)
        return x * y.unsqueeze(1) # Scale original input

class EnhancedLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes,
                 bidirectional=True, use_attention=True, attention_dim=None, dropout_rate=0.2,
                 use_se_block=False, se_reduction_ratio=16):
        super(EnhancedLSTM, self).__init__()
        self.input_dim = input_dim; self.hidden_dim = hidden_dim; self.num_layers = num_layers
        self.num_classes = num_classes; self.bidirectional = bidirectional
        self.use_attention = use_attention; self.use_se_block = use_se_block # Store this

        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=bidirectional, 
                            dropout=dropout_rate if num_layers > 1 else 0)
        
        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        
        self.se_block_instance = None # Initialize to None
        if self.use_se_block: # Check the instance variable
            self.se_block_instance = SequenceSEBlock(channels=lstm_output_dim, reduction_ratio=se_reduction_ratio)
        
        self.attention_instance = None # Initialize to None
        if self.use_attention: # Check the instance variable
            self.attention_instance = Attention(lstm_output_dim)
            fc_input_dim = lstm_output_dim
        else:
            fc_input_dim = lstm_output_dim
            
        self.fc_intermediate_dim = max(1, fc_input_dim // 2)
        self.fc0 = nn.Linear(fc_input_dim, self.fc_intermediate_dim)
        self.relu_fc = nn.ReLU(); self.dropout_fc = nn.Dropout(dropout_rate)
        self.fc_final = nn.Linear(self.fc_intermediate_dim, num_classes)

    def forward(self, x_time, seq_lens):
        packed_input = pack_padded_sequence(x_time, seq_lens.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, (hn, cn) = self.lstm(packed_input)
        lstm_sequence_output, _ = pad_packed_sequence(packed_output, batch_first=True)
        
        processed_sequence_output = lstm_sequence_output
        if self.use_se_block and self.se_block_instance is not None: # Check instance variable
            processed_sequence_output = self.se_block_instance(processed_sequence_output)
            
        if self.use_attention and self.attention_instance is not None: # Check instance variable
            context_vector, _ = self.attention_instance(processed_sequence_output)
            fc_input = context_vector
        else: # Fallback if no attention (or SE not designed for hn path)
            if self.bidirectional:
                fc_input = torch.cat((hn[-2,:,:], hn[-1,:,:]), dim=1)
            else:
                fc_input = hn[-1,:,:]
                
        out = self.fc0(fc_input); out = self.relu_fc(out); out = self.dropout_fc(out)
        out = self.fc_final(out)
        return out

# %%
# --- Optuna Objective Function ---
def objective(trial, fixed_config, data_globals):
    # --- Hyperparameters to be tuned by Optuna ---
    cfg_trial = {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True), # Narrowed LR
        "lstm_hidden_dim": trial.suggest_categorical("lstm_hidden_dim", [32, 64, 96, 128]),
        "num_lstm_layers": trial.suggest_int("num_lstm_layers", 1, 2),
        "dropout_rate": trial.suggest_float("dropout_rate", 0.1, 0.4, step=0.05), # Refined dropout
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-4, log=True), # Refined weight_decay
        "se_reduction_ratio": trial.suggest_categorical("se_reduction_ratio", [8, 16, 32]) \
                              if fixed_config["use_se_block"] else fixed_config.get("se_reduction_ratio", 16),
        # "attention_dim": fixed_config["attention_dim"] # Assuming fixed or not separately tunable in this Attention class
    }

    print(f"\n--- Trial {trial.number} Starting ---")
    print(f"Hyperparameters: {cfg_trial}")

    # Use a specific seed for each trial if you want trial-level reproducibility
    # trial_seed = fixed_config["random_seed"] + trial.number 
    # set_seed(trial_seed)


    model = EnhancedLSTM(
        input_dim=data_globals["lstm_input_dim"],
        hidden_dim=cfg_trial["lstm_hidden_dim"],
        num_layers=cfg_trial["num_lstm_layers"],
        num_classes=1,
        bidirectional=fixed_config["bidirectional"],
        use_attention=fixed_config["use_attention"],
        attention_dim=fixed_config["attention_dim"],
        dropout_rate=cfg_trial["dropout_rate"],
        use_se_block=fixed_config["use_se_block"],
        se_reduction_ratio=cfg_trial["se_reduction_ratio"]
    ).to(device)

    if data_globals["num_pos_train"] > 0:
        pos_weight_value = data_globals["num_neg_train"] / data_globals["num_pos_train"]
    else:
        pos_weight_value = 1.0
    pos_weight = torch.tensor([pos_weight_value], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=cfg_trial["learning_rate"], weight_decay=cfg_trial["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=fixed_config["lr_scheduler_factor"],
        patience=fixed_config["lr_scheduler_patience"], verbose=False
    )

    num_train_samples = data_globals["X_time_train_full"].shape[0]
    train_batch_size = fixed_config["batch_size"]
    num_train_batches = int(np.ceil(num_train_samples / train_batch_size))

    eval_batch_size = fixed_config["eval_batch_size"]
    num_test_samples = data_globals["X_time_test_full"].shape[0]
    num_eval_batches = int(np.ceil(num_test_samples / eval_batch_size))

    best_val_loss_for_trial = float('inf')
    epochs_no_improve_for_trial = 0
    
    for epoch in range(fixed_config["num_epochs_hpo"]):
        model.train()
        epoch_train_loss = 0.0
        for i in range(num_train_batches):
            start_idx = i * train_batch_size
            end_idx = min(num_train_samples, (i + 1) * train_batch_size)
            batch_x_time = data_globals["X_time_train_full"][start_idx:end_idx].to(device)
            batch_seq_len = data_globals["seq_len_train_full"][start_idx:end_idx]
            batch_y_binary = torch.tensor(data_globals["y_binary_train_full"][start_idx:end_idx]).unsqueeze(1).float().to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_x_time, batch_seq_len)
            loss = criterion(outputs, batch_y_binary)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=fixed_config["gradient_clip_value"])
            optimizer.step()
            epoch_train_loss += loss.item()
        
        avg_epoch_train_loss = epoch_train_loss / num_train_batches

        model.eval()
        all_val_outputs_logits = []; all_val_y_for_loss = []
        with torch.no_grad():
            for i in range(num_eval_batches):
                start_idx = i * eval_batch_size; end_idx = min(num_test_samples, (i + 1) * eval_batch_size)
                batch_x_time_val = data_globals["X_time_test_full"][start_idx:end_idx].to(device) # Using test set as validation for HPO
                batch_seq_len_val = data_globals["seq_len_test_full"][start_idx:end_idx]
                batch_y_binary_val_for_loss = torch.tensor(data_globals["y_binary_test_full"][start_idx:end_idx]).unsqueeze(1).float().to(device)
                outputs_batch_logits = model(batch_x_time_val, batch_seq_len_val)
                all_val_outputs_logits.append(outputs_batch_logits.cpu()); all_val_y_for_loss.append(batch_y_binary_val_for_loss.cpu())
        
        val_outputs_logits = torch.cat(all_val_outputs_logits, dim=0)
        val_y_for_loss = torch.cat(all_val_y_for_loss, dim=0)
        current_val_loss = criterion(val_outputs_logits.to(device), val_y_for_loss.to(device)).item()

        scheduler.step(current_val_loss)

        if current_val_loss < best_val_loss_for_trial:
            best_val_loss_for_trial = current_val_loss
            epochs_no_improve_for_trial = 0
            # Optional: Save best model for this specific trial
            # trial_model_path = os.path.join(fixed_config["best_model_save_path_base"], f"trial_{trial.number}_best_model.pth")
            # os.makedirs(os.path.dirname(trial_model_path), exist_ok=True)
            # torch.save(model.state_dict(), trial_model_path)
        else:
            epochs_no_improve_for_trial += 1
        
        if epochs_no_improve_for_trial >= fixed_config["early_stopping_patience"]:
            print(f"Trial {trial.number} early stopping at epoch {epoch+1} (val_loss: {current_val_loss:.4f})")
            break
        
        trial.report(current_val_loss, epoch)
        if trial.should_prune():
            print(f"Trial {trial.number} pruned at epoch {epoch+1} (val_loss: {current_val_loss:.4f}).")
            raise optuna.exceptions.TrialPruned()
        
        if (epoch + 1) % 10 == 0: # Log periodically for long trials
             print(f"Trial {trial.number}, Epoch [{epoch+1}/{fixed_config['num_epochs_hpo']}], Train Loss: {avg_epoch_train_loss:.4f}, Val Loss: {current_val_loss:.4f}")


    # Calculate AUPRC on validation set (using test set as validation for HPO)
    val_probs = torch.sigmoid(val_outputs_logits).numpy().squeeze()
    # Ensure val_pred_binary has same length as val_probs if val_probs is 1D
    if val_probs.ndim == 0: val_probs = np.array([val_probs]) # Ensure it's an array
    val_pred_binary = (val_probs > fixed_config["classification_threshold"])
    
    current_y_true_for_metric = data_globals["y_binary_test_full"][:len(val_probs)]
    
    # Handle case where performance_dict might not be available or fails
    try:
        dic = performance_dict(current_y_true_for_metric, val_pred_binary, val_probs)
        val_auprc = dic.get('pr_auc', 0.0)
    except Exception as e:
        print(f"Warning: Could not compute performance_dict for trial {trial.number}: {e}")
        val_auprc = 0.0 # Default or np.nan

    print(f"Trial {trial.number} finished. Best Val Loss: {best_val_loss_for_trial:.4f}, Final Val AUPRC: {val_auprc:.4f}")
    
    # --- You might want to return a metric that Optuna directly optimizes based on your goal ---
    # E.g., if AUPRC is more important and you want to maximize it: return -val_auprc
    # For now, returning best_val_loss (to be minimized)
    return best_val_loss_for_trial

# %%
# --- HPO Study Execution ---
if __name__ == "__main__":
    # Package necessary global data for the objective function
    data_globals_for_objective = {
        "lstm_input_dim": lstm_input_dim_global,
        "num_pos_train": num_pos_train_full,
        "num_neg_train": num_neg_train_full,
        "X_time_train_full": X_time_train_full,
        "seq_len_train_full": seq_len_train_full,
        "y_binary_train_full": y_binary_train_full,
        "X_time_test_full": X_time_test_full, # Using test set as validation set for HPO
        "seq_len_test_full": seq_len_test_full,
        "y_binary_test_full": y_binary_test_full, # True labels for validation
    }

    # Create necessary directories for HPO outputs
    os.makedirs(CONFIG["results_output_file_base"], exist_ok=True)
    os.makedirs(CONFIG["tensorboard_log_dir_base"], exist_ok=True) # Not used in objective for now
    os.makedirs(CONFIG["best_model_save_path_base"], exist_ok=True) # Not used in objective for now

    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=CONFIG["early_stopping_patience"], interval_steps=1)
    study = optuna.create_study(direction="minimize", pruner=pruner)

    objective_with_config = functools.partial(objective, fixed_config=CONFIG, data_globals=data_globals_for_objective)

    print(f"Starting HPO study with {CONFIG['n_hpo_trials']} trials...")
    try:
        study.optimize(objective_with_config, n_trials=CONFIG['n_hpo_trials'])
    except KeyboardInterrupt:
        print("HPO study interrupted by user.")
    except Exception as e:
        print(f"An error occurred during HPO study: {e}")
        import traceback
        traceback.print_exc()


    print("\n--- HPO Study Finished ---")
    print(f"Number of finished trials: {len(study.trials)}")
    
    if len(study.trials) > 0 and hasattr(study, 'best_trial') and study.best_trial is not None:
        best_trial = study.best_trial
        print("\nBest trial:")
        print(f"  Value (Best Validation Metric, e.g., Loss): {best_trial.value:.4f}")
        print("  Params: ")
        for key, value in best_trial.params.items():
            print(f"    {key}: {value}")

        # Save best parameters
        best_params_path = os.path.join(CONFIG["results_output_file_base"], "best_hpo_params.json")
        import json
        with open(best_params_path, 'w') as f:
            json.dump(best_trial.params, f, indent=4)
        print(f"Best hyperparameters saved to {best_params_path}")

    else:
        print("No successful trials completed or no best trial found.")


    print("\nTo train a final model with these best hyperparameters, update your CONFIG dict with these values,")
    print("increase num_epochs, and run your standard (non-HPO) training script.")

    study_results_df = study.trials_dataframe()
    study_results_path = os.path.join(CONFIG["results_output_file_base"], "hpo_study_results.csv")
    study_results_df.to_csv(study_results_path, index=False)
    print(f"Full HPO study results saved to {study_results_path}")

print("Script finished.")

Using device: cuda
Original train size: 43271
Test size: 10818
Training set: Positive AKI cases: 2595.0, Negative AKI cases: 40676.0


[I 2025-05-23 21:37:25,817] A new study created in memory with name: no-name-26fa0abf-7b9c-4381-a3b1-0d38018bc69c
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Starting HPO study with 50 trials...

--- Trial 0 Starting ---
Hyperparameters: {'learning_rate': 5.705326811262844e-05, 'lstm_hidden_dim': 128, 'num_lstm_layers': 1, 'dropout_rate': 0.2, 'weight_decay': 3.110241883658167e-06, 'se_reduction_ratio': 8}


: 

# Transformer

Working

Epoch [24/100], Train Loss: 1.0109, Val Loss: 1.1579, Val AUC: 0.7525, Val AUPRC: 0.2268
Validation loss did not improve for 15 epoch(s).
Early stopping triggered after 24 epochs.
Training finished.
Loading best model from /home/server/Projects/data/AKI/models/best_transformer_model.pth for final evaluation.

Starting final evaluation with the best/last model...
                                                                         

--- Final Test Performance (Best/Last Model) ---
Precision: 0.0934
Sensitivity: 0.8043
Accuracy: 0.5199
rc_auc: 0.7467
pr_auc: 0.2094
Specificity: 0.5017
Negative Predictive Value: 0.9757
F1 Score: 0.1674
Results saved to /home/server/Projects/data/AKI/results/transformer_model_results.pkl
Script finished.



In [3]:
# %%
import os
import numpy as np
import torch
# from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence # Not used for Transformer
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
import torch.optim as optim

import random
import math # For sinusoidal positional encoding if chosen, or math ops
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_curve, auc, precision_recall_curve, average_precision_score)
from sklearn.model_selection import train_test_split
import pandas as pd
# Ensure utils.py is in the same directory or Python path
from utils import performance_dict, sigmoid
from tqdm import tqdm

# --- Configuration for the Transformer Model ---
CONFIG = {
    "data_file": '/home/server/Projects/data/AKI/lstm_trainable.pkl', # !!! UPDATE THIS PATH !!!
    "results_output_file": '/home/server/Projects/data/AKI/results/transformer_model_results.pkl',
    "tensorboard_log_dir_base": '/home/server/Projects/data/AKI/runs/transformer_model/',
    "best_model_save_path": '/home/server/Projects/data/AKI/models/best_transformer_model.pth',
    "test_split_size": 0.2,
    "random_seed": 42,
    "num_epochs": 100,
    "learning_rate": 0.00042587898767975377, # HPO
    "weight_decay": 3.593168387199891e-05, # HPO
    "gradient_clip_value": 1.0,
    "lr_scheduler_patience": 7,
    "lr_scheduler_factor": 0.2, # Can be more aggressive for Transformers
    "early_stopping_patience": 15,
    
    # --- Architecture Specifics for Transformer ---
    "input_features": -1, # Will be set dynamically from data (e.g., ~24)
    "d_model": 32,             # Embedding dimension / Transformer hidden size - HPO
    "num_encoder_layers": 4,    # Number of Transformer Encoder layers (e.g., 2-6) - HPO    
    "num_attention_heads": 2,   # Number of heads in Multi-Head Attention (must divide d_model) - HPO
    "dim_feedforward": 128,     # Dimension of the feed-forward network within encoder (e.g., 2*d_model or 4*d_model) - HPO by multiplying d_model (32) * ff_multiplier (4)
    "transformer_dropout_rate": 0.2, # Dropout in Transformer layers - HPO
    "max_seq_len_for_pe": 500,  # Max sequence length for learnable positional embeddings (e.g., > typical max surgery length in 5-min steps)
    
    "batch_size": 256, # <<--- Transformers can be memory intensive; start smaller. 256 worked with default architecture, HPO optimized architecture is much smaller, larger batch sizes
    "eval_batch_size": 256,
    "classification_threshold": 0.3, 
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load data (once)
try:
    df_full = pd.read_pickle(CONFIG["data_file"])
except FileNotFoundError:
    print(f"ERROR: Data file not found at {CONFIG['data_file']}")
    print("Please ensure the path in CONFIG['data_file'] is correct.")
    exit()
except Exception as e:
    print(f"Error loading data file {CONFIG['data_file']}: {e}")
    exit()


# %%
# --- Data Preparation ---
def df_to_tensors(df, all_columns):
    non_tabular_cols = ['op_id', 'time_tensors', 'seq_len', 'aki', 'aki_boolean']
    tabular_feature_cols = [col for col in all_columns if col not in non_tabular_cols]
    if tabular_feature_cols and not df[tabular_feature_cols].empty:
        X_tab = torch.tensor(df[tabular_feature_cols].values).float()
    else:
        X_tab = torch.empty((len(df), 0)).float()
    X_time = torch.stack(df['time_tensors'].tolist()).float()
    y = torch.tensor(df['aki'].values).unsqueeze(1).float()
    y_binary = df['aki_boolean'].values.astype(bool)
    sequence_lengths = torch.tensor(df['seq_len'].tolist()).long()
    return X_tab, X_time, sequence_lengths, y, y_binary

def prepare_data(full_df, test_size, random_seed):
    df_train, df_test = train_test_split(full_df, test_size=test_size, random_state=random_seed, stratify=full_df['aki_boolean'])
    print(f"Original train size: {len(df_train)}")
    print(f"Test size: {len(df_test)}")
    all_columns = full_df.columns.tolist()
    num_positive_aki_train = df_train['aki_boolean'].sum()
    num_negative_aki_train = len(df_train) - num_positive_aki_train
    print(f"Training set: Positive AKI cases: {num_positive_aki_train}, Negative AKI cases: {num_negative_aki_train}")
    X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train = df_to_tensors(df_train, all_columns)
    X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test = df_to_tensors(df_test, all_columns)
    
    # Set input_features in CONFIG dynamically
    CONFIG["input_features"] = X_time_train.shape[2]
    print(f"Number of input time-series features: {CONFIG['input_features']}")

    return (X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train,
            X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test,
            num_positive_aki_train, num_negative_aki_train)

(X_tab_train_full, X_time_train_full, seq_len_train_full, y_train_full, y_binary_train_full,
 X_tab_test_full, X_time_test_full, seq_len_test_full, y_test_full, y_binary_test_full,
 num_pos_train_full, num_neg_train_full) = prepare_data(
    df_full, CONFIG["test_split_size"], CONFIG["random_seed"]
)


# %%
# --- Utility Functions ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["random_seed"])

def save_single_run_results(model_name, metrics_dict, output_file):
    # (Same as your previous save_single_run_results function)
    results_to_save = {k: [v] if not isinstance(v, (list, np.ndarray)) else v for k,v in metrics_dict.items()}
    df_result = pd.DataFrame(results_to_save)
    df_result['model_name'] = model_name
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    if os.path.exists(output_file):
        try:
            df_output = pd.read_pickle(output_file)
            df_output = df_output[df_output['model_name'] != model_name]
            df_output = pd.concat([df_output, df_result], ignore_index=True)
        except EOFError: df_output = df_result
    else: df_output = df_result
    df_output.to_pickle(output_file)
    print(f"Results saved to {output_file}")

# %%
# --- Transformer Model Definition ---

class PositionalEncoding(nn.Module):
    # Standard Sinusoidal Positional Encoding
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.transpose(0,1)) # pe shape (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor, shape [batch_size, seq_len, embedding_dim]
        """
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class LearnablePositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 500):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        # nn.Embedding provides learnable lookups
        self.pos_embedding = nn.Embedding(max_len, d_model)
        # self.register_buffer('positions', torch.arange(max_len)) # Not needed if passing positions directly

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor, shape [batch_size, seq_len, embedding_dim]
        """
        # Create positions on the fly based on input sequence length
        # Ensure positions are on the same device as x
        positions = torch.arange(0, x.size(1), device=x.device).unsqueeze(0) # (1, seq_len)
        # If x.size(1) > max_len of embedding, this will error. Ensure max_len is sufficient.
        x = x + self.pos_embedding(positions) # (B, S, D) + (1, S, D) -> (B, S, D)
        return self.dropout(x)


class TransformerClassifier(nn.Module):
    def __init__(self, input_features, d_model, num_heads, num_encoder_layers, dim_feedforward,
                 num_classes, dropout_rate=0.1, max_seq_len_pe=500, use_learnable_pe=True):
        super(TransformerClassifier, self).__init__()
        self.d_model = d_model

        # 1. Input Feature Embedding
        self.input_embedder = nn.Linear(input_features, d_model)
        
        # 2. Positional Encoding
        if use_learnable_pe:
            self.pos_encoder = LearnablePositionalEncoding(d_model, dropout_rate, max_seq_len_pe)
        else:
            self.pos_encoder = PositionalEncoding(d_model, dropout_rate, max_seq_len_pe)
            
        # 3. Transformer Encoder Stack
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout_rate,
            batch_first=True # Crucial for (batch, seq, feature) input
        )
        encoder_norm = nn.LayerNorm(d_model) # Optional: norm after the stack
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers,
            norm=encoder_norm
        )
        
        # 4. Final Classifier Head
        # Output of Transformer Encoder is (batch, seq_len, d_model)
        # We will mean-pool this for classification
        self.fc_intermediate_dim = max(1, d_model // 2)
        self.classifier_head = nn.Sequential(
            nn.Linear(d_model, self.fc_intermediate_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(self.fc_intermediate_dim, num_classes)
        )

    def _generate_padding_mask(self, seq_lens, max_len):
        # seq_lens: (batch_size) tensor of actual lengths
        # max_len: scalar, max sequence length in the current batch (x_time.size(1))
        # Returns: (batch_size, max_len) boolean tensor, True where padded
        batch_size = seq_lens.size(0)
        mask = torch.arange(max_len, device=seq_lens.device).expand(batch_size, max_len) >= seq_lens.unsqueeze(1)
        return mask

    def forward(self, x_time, seq_lens): # x_time: (batch, seq_len, input_features), seq_lens: (batch)
        # 1. Input Embedding
        x_embedded = self.input_embedder(x_time) # (batch, seq_len, d_model)
        x_embedded = x_embedded * math.sqrt(self.d_model) # Scale embedding (common practice)

        # 2. Add Positional Encoding
        x_pos_encoded = self.pos_encoder(x_embedded) # (batch, seq_len, d_model)
        
        # 3. Create Padding Mask for Transformer Encoder
        # True for positions that should be IGNORED by attention
        src_key_padding_mask = self._generate_padding_mask(seq_lens.cpu(), x_time.size(1)).to(x_time.device)
        
        # 4. Pass through Transformer Encoder
        encoder_output = self.transformer_encoder(x_pos_encoded, src_key_padding_mask=src_key_padding_mask)
        # encoder_output shape: (batch, seq_len, d_model)
        
        # 5. Aggregate Encoder Output (Mean Pooling over non-padded elements)
        # Invert mask for summing: True for non-padded elements
        sum_mask = ~src_key_padding_mask # (batch, seq_len)
        # Zero out padded elements before summing
        masked_encoder_output = encoder_output * sum_mask.unsqueeze(-1) # (batch, seq_len, d_model)
        summed_output = masked_encoder_output.sum(dim=1) # (batch, d_model)
        # Count non-padded elements for each sequence to compute mean
        num_non_padded = sum_mask.sum(dim=1, keepdim=True) # (batch, 1)
        # Prevent division by zero for sequences of zero length (though seq_lens should be > 0)
        pooled_output = summed_output / num_non_padded.clamp(min=1e-9) # (batch, d_model)
        
        # 6. Pass through Classifier Head
        logits = self.classifier_head(pooled_output) # (batch, num_classes)
        return logits

# %%
# --- Main Training and Evaluation ---
model = TransformerClassifier(
    input_features=CONFIG["input_features"],
    d_model=CONFIG["d_model"],
    num_heads=CONFIG["num_attention_heads"],
    num_encoder_layers=CONFIG["num_encoder_layers"],
    dim_feedforward=CONFIG["dim_feedforward"],
    num_classes=1, # Binary classification
    dropout_rate=CONFIG["transformer_dropout_rate"],
    max_seq_len_pe=CONFIG["max_seq_len_for_pe"],
    use_learnable_pe=True # Using Learnable PE by default
).to(device)

print(model)
print(f"Transformer Model Parameter Count: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

if num_pos_train_full > 0:
    pos_weight_value = num_neg_train_full / num_pos_train_full
else:
    pos_weight_value = 1.0; print("Warning: No positive samples in train for pos_weight.")
pos_weight = torch.tensor([pos_weight_value], device=device)
print(f"Calculated pos_weight for BCEWithLogitsLoss: {pos_weight.item():.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=CONFIG["lr_scheduler_factor"],
    patience=CONFIG["lr_scheduler_patience"])

run_name = f"Transformer_d{CONFIG['d_model']}_L{CONFIG['num_encoder_layers']}_H{CONFIG['num_attention_heads']}_classWeighted"
tensorboard_run_path = os.path.join(CONFIG["tensorboard_log_dir_base"], run_name)
os.makedirs(tensorboard_run_path, exist_ok=True)
writer = SummaryWriter(tensorboard_run_path)

num_train_samples = X_time_train_full.shape[0]
train_batch_size = CONFIG["batch_size"]
num_train_batches = int(np.ceil(num_train_samples / train_batch_size))
eval_batch_size = CONFIG["eval_batch_size"]
num_test_samples = X_time_test_full.shape[0]
num_eval_batches = int(np.ceil(num_test_samples / eval_batch_size))

best_val_loss = float('inf')
epochs_no_improve = 0
os.makedirs(os.path.dirname(CONFIG["best_model_save_path"]), exist_ok=True)

print(f"Starting training for {CONFIG['num_epochs']} epochs on {num_train_samples} original samples...")
for epoch in tqdm(range(CONFIG["num_epochs"]), desc="Epochs"):
    model.train()
    epoch_train_loss = 0.0
    for i in tqdm(range(num_train_batches), desc="Training Batches", leave=False):
        start_idx = i * train_batch_size
        end_idx = min(num_train_samples, (i + 1) * train_batch_size)
        batch_x_time = X_time_train_full[start_idx:end_idx].to(device)
        batch_seq_len = seq_len_train_full[start_idx:end_idx] # Used for mask
        batch_y_binary = torch.tensor(y_binary_train_full[start_idx:end_idx]).unsqueeze(1).float().to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x_time, batch_seq_len) # Transformer doesn't use pack_padded_sequence
        loss = criterion(outputs, batch_y_binary)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CONFIG["gradient_clip_value"])
        optimizer.step()
        epoch_train_loss += loss.item()
    avg_epoch_train_loss = epoch_train_loss / num_train_batches
    writer.add_scalar('Loss/train', avg_epoch_train_loss, epoch)

    model.eval()
    all_val_outputs_logits = []; all_val_y_for_loss = []
    with torch.no_grad():
        for i in tqdm(range(num_eval_batches), desc="Validation Batches", leave=False):
            start_idx = i * eval_batch_size; end_idx = min(num_test_samples, (i + 1) * eval_batch_size)
            batch_x_time_val = X_time_test_full[start_idx:end_idx].to(device)
            batch_seq_len_val = seq_len_test_full[start_idx:end_idx]
            batch_y_binary_val_for_loss = torch.tensor(y_binary_test_full[start_idx:end_idx]).unsqueeze(1).float().to(device)
            outputs_batch_logits = model(batch_x_time_val, batch_seq_len_val)
            all_val_outputs_logits.append(outputs_batch_logits.cpu()); all_val_y_for_loss.append(batch_y_binary_val_for_loss.cpu())
    val_outputs_logits = torch.cat(all_val_outputs_logits, dim=0); val_y_for_loss = torch.cat(all_val_y_for_loss, dim=0)
    current_val_loss = criterion(val_outputs_logits.to(device), val_y_for_loss.to(device)).item()
    writer.add_scalar('Loss/val', current_val_loss, epoch)
    val_probs = torch.sigmoid(val_outputs_logits).numpy().squeeze()
    if val_probs.ndim == 0: val_probs = np.array([val_probs])
    val_pred_binary = (val_probs > CONFIG["classification_threshold"])
    current_y_true_for_metric = y_binary_test_full[:len(val_pred_binary)]
    
    try:
        dic = performance_dict(current_y_true_for_metric, val_pred_binary, val_probs)
        writer.add_scalar('Val/AUC', dic.get('rc_auc', np.nan), epoch)
        writer.add_scalar('Val/AUPRC', dic.get('pr_auc', np.nan), epoch)
        writer.add_scalar('Val/Accuracy', dic.get('Accuracy', np.nan), epoch)
        # ... (log other metrics from dic)
        print(f"Epoch [{epoch+1}/{CONFIG['num_epochs']}], Train Loss: {avg_epoch_train_loss:.4f}, Val Loss: {current_val_loss:.4f}, Val AUC: {dic.get('rc_auc', np.nan):.4f}, Val AUPRC: {dic.get('pr_auc', np.nan):.4f}")
    except Exception as e:
        print(f"Error calculating performance_dict in validation: {e}")
        print(f"Epoch [{epoch+1}/{CONFIG['num_epochs']}], Train Loss: {avg_epoch_train_loss:.4f}, Val Loss: {current_val_loss:.4f}")

    scheduler.step(current_val_loss)
    if current_val_loss < best_val_loss:
        best_val_loss = current_val_loss; epochs_no_improve = 0
        torch.save(model.state_dict(), CONFIG["best_model_save_path"])
        print(f"Validation loss improved. Saved best model to {CONFIG['best_model_save_path']}")
    else:
        epochs_no_improve += 1
        print(f"Validation loss did not improve for {epochs_no_improve} epoch(s).")
    if epochs_no_improve >= CONFIG["early_stopping_patience"]:
        print(f"Early stopping triggered after {epoch+1} epochs.")
        break
writer.close()
print("Training finished.")

if os.path.exists(CONFIG["best_model_save_path"]):
    print(f"Loading best model from {CONFIG['best_model_save_path']} for final evaluation.")
    model.load_state_dict(torch.load(CONFIG["best_model_save_path"]))
else:
    print("No best model found. Using last model state.")

model.eval()
final_all_test_outputs_logits = []
print("\nStarting final evaluation with the best/last model...")
with torch.no_grad():
    for i in tqdm(range(num_eval_batches), desc="Final Evaluation Batches", leave=False):
        start_idx = i * eval_batch_size; end_idx = min(num_test_samples, (i + 1) * eval_batch_size)
        batch_x_time_test = X_time_test_full[start_idx:end_idx].to(device)
        batch_seq_len_test = seq_len_test_full[start_idx:end_idx]
        outputs_batch_logits = model(batch_x_time_test, batch_seq_len_test)
        final_all_test_outputs_logits.append(outputs_batch_logits.cpu())
final_test_outputs_logits = torch.cat(final_all_test_outputs_logits, dim=0)
final_test_probs = torch.sigmoid(final_test_outputs_logits).numpy().squeeze()
if final_test_probs.ndim == 0: final_test_probs = np.array([final_test_probs])
final_y_pred_binary = (final_test_probs > CONFIG["classification_threshold"])
current_y_true_for_final_metric = y_binary_test_full[:len(final_y_pred_binary)]

try:
    final_metrics = performance_dict(current_y_true_for_final_metric, final_y_pred_binary, final_test_probs)
    print("\n--- Final Test Performance (Best/Last Model) ---")
    for metric, value in final_metrics.items():
        if isinstance(value, (float, int, np.number)): print(f"{metric}: {value:.4f}")
    save_single_run_results(run_name, final_metrics, CONFIG["results_output_file"])
except Exception as e:
    print(f"Error calculating final performance_dict: {e}")

print("Script finished.")

Using device: cuda
Original train size: 43271
Test size: 10818
Training set: Positive AKI cases: 2595.0, Negative AKI cases: 40676.0


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Number of input time-series features: 24
TransformerClassifier(
  (input_embedder): Linear(in_features=24, out_features=32, bias=True)
  (pos_encoder): LearnablePositionalEncoding(
    (dropout): Dropout(p=0.2, inplace=False)
    (pos_embedding): Embedding(500, 32)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
        )
        (linear1): Linear(in_features=32, out_features=128, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=128, out_features=32, bias=True)
        (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
    (norm): LayerNorm((

Epochs:   1%|          | 1/100 [00:47<1:18:12, 47.40s/it]

Epoch [1/100], Train Loss: 1.2376, Val Loss: 1.1843, Val AUC: 0.7023, Val AUPRC: 0.1445
Validation loss improved. Saved best model to /home/server/Projects/data/AKI/models/best_transformer_model.pth


...

Epochs:  23%|██▎       | 23/100 [18:17<1:01:19, 47.79s/it]

Epoch [23/100], Train Loss: 1.0174, Val Loss: 1.1584, Val AUC: 0.7521, Val AUPRC: 0.2261
Validation loss did not improve for 14 epoch(s).


Epochs:  23%|██▎       | 23/100 [19:05<1:03:53, 49.79s/it]
/tmp/ipykernel_152677/363335550.py:373: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.

Epoch [24/100], Train Loss: 1.0109, Val Loss: 1.1579, Val AUC: 0.7525, Val AUPRC: 0.2268
Validation loss did not improve for 15 epoch(s).
Early stopping triggered after 24 epochs.
Training finished.
Loading best model from /home/server/Projects/data/AKI/models/best_transformer_model.pth for final evaluation.

Starting final evaluation with the best/last model...



--- Final Test Performance (Best/Last Model) ---
Precision: 0.0934
Sensitivity: 0.8043
Accuracy: 0.5199
rc_auc: 0.7467
pr_auc: 0.2094
Specificity: 0.5017
Negative Predictive Value: 0.9757
F1 Score: 0.1674
Results saved to /home/server/Projects/data/AKI/results/transformer_model_results.pkl
Script finished.


# HPO Transformer

Working

--- HPO Study Finished ---
Number of finished trials: 50

Best trial for Transformer:
  Value (Best Validation Loss): 1.1163
  Params: 
    d_model: 32
    num_encoder_layers: 4
    num_attention_heads: 2
    ff_multiplier: 4
    learning_rate: 0.00042587898767975377
    transformer_dropout_rate: 0.2
    weight_decay: 3.593168387199891e-05
Best hyperparameters for Transformer saved to /home/server/Projects/data/AKI/results/hpo_transformer/best_transformer_hpo_params.json

To train a final Transformer model with these best hyperparameters, update your CONFIG dict,
set num_epochs to a higher value, and run a standard (non-HPO) training script.
Full HPO study results for Transformer saved to /home/server/Projects/data/AKI/results/hpo_transformer/hpo_transformer_study_results.csv
Script finished.

In [3]:
# %%
import os
import numpy as np
import torch
# No pack_padded_sequence needed for this Transformer version
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
import torch.optim as optim
import optuna # Optuna import
import functools # For passing arguments to objective function
import math # For positional encoding

import random
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_curve, auc, precision_recall_curve, average_precision_score)
from sklearn.model_selection import train_test_split
import pandas as pd
# Ensure utils.py is in the same directory or Python path
from utils import performance_dict, sigmoid
from tqdm import tqdm

# --- Configuration ---
CONFIG = {
    "data_file": '/home/server/Projects/data/AKI/lstm_trainable.pkl', # !!! UPDATE THIS PATH !!!
    "results_output_file_base": '/home/server/Projects/data/AKI/results/hpo_transformer/',
    "tensorboard_log_dir_base": '/home/server/Projects/data/AKI/runs/hpo_transformer/',
    # "best_model_save_path_base": '/home/server/Projects/data/AKI/models/hpo_transformer/', # Optional per-trial model saving
    "test_split_size": 0.2,
    "random_seed": 42,
    "num_epochs_hpo": 50,  # Max epochs PER HPO TRIAL
    "gradient_clip_value": 1.0,
    "lr_scheduler_patience": 5,
    "lr_scheduler_factor": 0.2,
    "early_stopping_patience": 10, # Per-trial early stopping
    
    # --- Fixed Architectural Choices / Parameters for Transformer HPO ---
    "max_seq_len_for_pe": 500, # Max sequence length for positional embeddings
    "use_learnable_pe": True,  # Whether to use learnable or sinusoidal PE
    "input_features": -1,      # Will be set dynamically
    
    "batch_size": 256,         # Batch size for HPO trials (Transformers can be memory hungry)
    "eval_batch_size": 256,
    "classification_threshold": 0.3, # For final metrics, not usually for HPO objective itself
    "n_hpo_trials": 50,        # Number of HPO trials to run
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load data (once)
try:
    df_full = pd.read_pickle(CONFIG["data_file"])
except FileNotFoundError:
    print(f"ERROR: Data file not found at {CONFIG['data_file']}")
    print("Please ensure the path in CONFIG['data_file'] is correct.")
    exit()
except Exception as e:
    print(f"Error loading data file {CONFIG['data_file']}: {e}")
    exit()


# %%
# --- Data Preparation ---
def df_to_tensors(df, all_columns):
    non_tabular_cols = ['op_id', 'time_tensors', 'seq_len', 'aki', 'aki_boolean']
    tabular_feature_cols = [col for col in all_columns if col not in non_tabular_cols]
    if tabular_feature_cols and not df[tabular_feature_cols].empty:
        X_tab = torch.tensor(df[tabular_feature_cols].values).float()
    else:
        X_tab = torch.empty((len(df), 0)).float()
    X_time = torch.stack(df['time_tensors'].tolist()).float()
    y = torch.tensor(df['aki'].values).unsqueeze(1).float()
    y_binary = df['aki_boolean'].values.astype(bool)
    sequence_lengths = torch.tensor(df['seq_len'].tolist()).long()
    return X_tab, X_time, sequence_lengths, y, y_binary

def prepare_data(full_df, test_size, random_seed):
    df_train, df_test = train_test_split(full_df, test_size=test_size, random_state=random_seed, stratify=full_df['aki_boolean'])
    print(f"Original train size: {len(df_train)}")
    print(f"Test size: {len(df_test)}")
    all_columns = full_df.columns.tolist()
    num_positive_aki_train = df_train['aki_boolean'].sum()
    num_negative_aki_train = len(df_train) - num_positive_aki_train
    print(f"Training set: Positive AKI cases: {num_positive_aki_train}, Negative AKI cases: {num_negative_aki_train}")
    X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train = df_to_tensors(df_train, all_columns)
    X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test = df_to_tensors(df_test, all_columns)
    
    CONFIG["input_features"] = X_time_train.shape[2]
    print(f"Number of input time-series features: {CONFIG['input_features']}")
    
    return (X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train,
            X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test,
            num_positive_aki_train, num_negative_aki_train)

(X_tab_train_full, X_time_train_full, seq_len_train_full, y_train_full, y_binary_train_full,
 X_tab_test_full, X_time_test_full, seq_len_test_full, y_test_full, y_binary_test_full,
 num_pos_train_full, num_neg_train_full) = prepare_data(
    df_full, CONFIG["test_split_size"], CONFIG["random_seed"]
)
# Ensure input_features is set globally if data_globals doesn't pass it explicitly
input_features_global = CONFIG["input_features"]


# %%
# --- Utility Functions ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["random_seed"])

# (save_hpo_trial_results function can be kept if you want to save individual trial outputs)

# %%
# --- Transformer Model Definition ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.transpose(0,1))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class LearnablePositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 500):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        self.pos_embedding = nn.Embedding(max_len, d_model)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        positions = torch.arange(0, x.size(1), device=x.device).unsqueeze(0)
        x = x + self.pos_embedding(positions)
        return self.dropout(x)

class TransformerClassifier(nn.Module):
    def __init__(self, input_features, d_model, num_heads, num_encoder_layers, dim_feedforward,
                 num_classes, dropout_rate=0.1, max_seq_len_pe=500, use_learnable_pe=True):
        super(TransformerClassifier, self).__init__()
        self.d_model = d_model
        self.input_embedder = nn.Linear(input_features, d_model)
        if use_learnable_pe:
            self.pos_encoder = LearnablePositionalEncoding(d_model, dropout_rate, max_seq_len_pe)
        else:
            self.pos_encoder = PositionalEncoding(d_model, dropout_rate, max_seq_len_pe)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, dim_feedforward=dim_feedforward,
            dropout=dropout_rate, batch_first=True )
        encoder_norm = nn.LayerNorm(d_model)
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_encoder_layers, norm=encoder_norm )
        self.fc_intermediate_dim = max(1, d_model // 2)
        self.classifier_head = nn.Sequential(
            nn.Linear(d_model, self.fc_intermediate_dim), nn.ReLU(),
            nn.Dropout(dropout_rate), nn.Linear(self.fc_intermediate_dim, num_classes) )
    def _generate_padding_mask(self, seq_lens, max_len):
        batch_size = seq_lens.size(0)
        mask = torch.arange(max_len, device=seq_lens.device).expand(batch_size, max_len) >= seq_lens.unsqueeze(1)
        return mask
    def forward(self, x_time, seq_lens):
        x_embedded = self.input_embedder(x_time) * math.sqrt(self.d_model)
        x_pos_encoded = self.pos_encoder(x_embedded)
        src_key_padding_mask = self._generate_padding_mask(seq_lens.cpu(), x_time.size(1)).to(x_time.device)
        encoder_output = self.transformer_encoder(x_pos_encoded, src_key_padding_mask=src_key_padding_mask)
        sum_mask = ~src_key_padding_mask
        masked_encoder_output = encoder_output * sum_mask.unsqueeze(-1)
        summed_output = masked_encoder_output.sum(dim=1)
        num_non_padded = sum_mask.sum(dim=1, keepdim=True)
        pooled_output = summed_output / num_non_padded.clamp(min=1e-9)
        logits = self.classifier_head(pooled_output)
        return logits

# %%
# --- Optuna Objective Function ---
def objective(trial, fixed_config, data_globals):
    # --- Hyperparameters to be tuned by Optuna ---
    # Suggest d_model first, then constrain other parameters based on it for VRAM
    d_model_val = trial.suggest_categorical("d_model", [32, 64]) # <<--- REDUCED D_MODEL OPTIONS
    
    # Constrain num_encoder_layers based on d_model to manage memory
    if d_model_val <= 64:
        num_encoder_layers_val = trial.suggest_int("num_encoder_layers", 2, 4) # Max 4 layers for smaller d_model
    else: # Should not be hit with current d_model options, but good for future
        num_encoder_layers_val = trial.suggest_int("num_encoder_layers", 2, 3) 

    # Ensure num_attention_heads divides d_model
    possible_heads = [h for h in [1, 2, 4, 8] if d_model_val % h == 0]
    if not possible_heads: # Should not happen if d_model is e.g. 32, 64
        print(f"Trial {trial.number} skipped: No valid num_attention_heads for d_model {d_model_val}.")
        raise optuna.exceptions.TrialPruned()
    num_attention_heads_val = trial.suggest_categorical("num_attention_heads", possible_heads)

    # Constrain dim_feedforward based on d_model
    # E.g., suggest a multiplier for d_model
    ff_multiplier = trial.suggest_categorical("ff_multiplier", [2, 4])
    dim_feedforward_val = d_model_val * ff_multiplier

    cfg_trial = {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True), # Kept original range
        "d_model": d_model_val,
        "num_encoder_layers": num_encoder_layers_val,
        "num_attention_heads": num_attention_heads_val,
        "dim_feedforward": dim_feedforward_val,
        "transformer_dropout_rate": trial.suggest_float("transformer_dropout_rate", 0.1, 0.3, step=0.05), # Slightly reduced range
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-4, log=True),
    }

    print(f"\n--- Trial {trial.number} Starting ---")
    print(f"Hyperparameters for Transformer: {cfg_trial}")

    # <<< --- ADD TRY-EXCEPT BLOCK FOR OOM --- >>>
    try:
        model = TransformerClassifier(
            input_features=data_globals["input_features"],
            d_model=cfg_trial["d_model"],
            num_heads=cfg_trial["num_attention_heads"],
            num_encoder_layers=cfg_trial["num_encoder_layers"],
            dim_feedforward=cfg_trial["dim_feedforward"],
            num_classes=1,
            dropout_rate=cfg_trial["transformer_dropout_rate"],
            max_seq_len_pe=fixed_config["max_seq_len_for_pe"],
            use_learnable_pe=fixed_config["use_learnable_pe"]
        ).to(device)

        if data_globals["num_pos_train"] > 0:
            pos_weight_value = data_globals["num_neg_train"] / data_globals["num_pos_train"]
        else: pos_weight_value = 1.0
        pos_weight = torch.tensor([pos_weight_value], device=device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = optim.Adam(model.parameters(), lr=cfg_trial["learning_rate"], weight_decay=cfg_trial["weight_decay"])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=fixed_config["lr_scheduler_factor"],
            patience=fixed_config["lr_scheduler_patience"], verbose=False )

        num_train_samples = data_globals["X_time_train_full"].shape[0]
        # Use fixed_config for batch sizes as they are memory critical
        train_batch_size = fixed_config["batch_size"] 
        num_train_batches = int(np.ceil(num_train_samples / train_batch_size))
        eval_batch_size = fixed_config["eval_batch_size"]
        num_test_samples = data_globals["X_time_test_full"].shape[0]
        num_eval_batches = int(np.ceil(num_test_samples / eval_batch_size))

        best_val_loss_for_trial = float('inf')
        epochs_no_improve_for_trial = 0
        
        for epoch in range(fixed_config["num_epochs_hpo"]):
            model.train()
            epoch_train_loss = 0.0
            for i in range(num_train_batches):
                start_idx = i * train_batch_size; end_idx = min(num_train_samples, (i + 1) * train_batch_size)
                batch_x_time = data_globals["X_time_train_full"][start_idx:end_idx].to(device)
                batch_seq_len = data_globals["seq_len_train_full"][start_idx:end_idx]
                batch_y_binary = torch.tensor(data_globals["y_binary_train_full"][start_idx:end_idx]).unsqueeze(1).float().to(device)
                
                optimizer.zero_grad()
                outputs = model(batch_x_time, batch_seq_len)
                loss = criterion(outputs, batch_y_binary)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=fixed_config["gradient_clip_value"])
                optimizer.step()
                epoch_train_loss += loss.item()
            avg_epoch_train_loss = epoch_train_loss / num_train_batches if num_train_batches > 0 else 0


            model.eval()
            all_val_outputs_logits = []; all_val_y_for_loss = []
            with torch.no_grad():
                for i in range(num_eval_batches):
                    start_idx = i * eval_batch_size; end_idx = min(num_test_samples, (i + 1) * eval_batch_size)
                    batch_x_time_val = data_globals["X_time_test_full"][start_idx:end_idx].to(device)
                    batch_seq_len_val = data_globals["seq_len_test_full"][start_idx:end_idx]
                    batch_y_binary_val_for_loss = torch.tensor(data_globals["y_binary_test_full"][start_idx:end_idx]).unsqueeze(1).float().to(device)
                    outputs_batch_logits = model(batch_x_time_val, batch_seq_len_val)
                    all_val_outputs_logits.append(outputs_batch_logits.cpu()); all_val_y_for_loss.append(batch_y_binary_val_for_loss.cpu())
            
            if not all_val_outputs_logits: # Handle empty validation set if num_eval_batches is 0
                print(f"Warning: Trial {trial.number}, Epoch {epoch+1} - No validation batches processed.")
                current_val_loss = float('inf') # Or handle as appropriate
            else:
                val_outputs_logits = torch.cat(all_val_outputs_logits, dim=0)
                val_y_for_loss = torch.cat(all_val_y_for_loss, dim=0)
                current_val_loss = criterion(val_outputs_logits.to(device), val_y_for_loss.to(device)).item()

            scheduler.step(current_val_loss)

            if current_val_loss < best_val_loss_for_trial:
                best_val_loss_for_trial = current_val_loss
                epochs_no_improve_for_trial = 0
            else:
                epochs_no_improve_for_trial += 1
            
            if epochs_no_improve_for_trial >= fixed_config["early_stopping_patience"]:
                print(f"Trial {trial.number} early stopping at epoch {epoch+1} (val_loss: {current_val_loss:.4f})")
                break
            
            trial.report(current_val_loss, epoch)
            if trial.should_prune():
                print(f"Trial {trial.number} pruned at epoch {epoch+1} (val_loss: {current_val_loss:.4f}).")
                raise optuna.exceptions.TrialPruned()
            
            if (epoch + 1) % 10 == 0 or epoch == fixed_config["num_epochs_hpo"] -1 : # Log periodically
                 print(f"Trial {trial.number}, Epoch [{epoch+1}/{fixed_config['num_epochs_hpo']}], Train Loss: {avg_epoch_train_loss:.4f}, Val Loss: {current_val_loss:.4f}")
        
        # Final metrics for this trial (optional logging, main return is best_val_loss_for_trial)
        if 'val_outputs_logits' in locals() and val_outputs_logits is not None and len(val_outputs_logits) > 0: # Check if validation was run
            val_probs = torch.sigmoid(val_outputs_logits).numpy().squeeze()
            if val_probs.ndim == 0: val_probs = np.array([val_probs])
            val_pred_binary = (val_probs > fixed_config["classification_threshold"])
            current_y_true_for_metric = data_globals["y_binary_test_full"][:len(val_probs)]
            try:
                dic = performance_dict(current_y_true_for_metric, val_pred_binary, val_probs)
                val_auprc = dic.get('pr_auc', 0.0)
            except Exception as e:
                print(f"Warning: Could not compute performance_dict for trial {trial.number} final metrics: {e}"); val_auprc = 0.0
            print(f"Trial {trial.number} finished. Best Val Loss: {best_val_loss_for_trial:.4f}, Final Epoch Val AUPRC: {val_auprc:.4f}")
        else:
            print(f"Trial {trial.number} finished. Best Val Loss: {best_val_loss_for_trial:.4f} (No validation outputs to calculate AUPRC, possibly pruned early).")

        return best_val_loss_for_trial

    except torch.cuda.OutOfMemoryError: # Specific for CUDA OOM
        print(f"Trial {trial.number} failed due to CUDA OutOfMemoryError. Pruning trial.")
        # Free up memory
        del model
        if 'optimizer' in locals(): del optimizer
        if 'criterion' in locals(): del criterion
        if 'scheduler' in locals(): del scheduler
        torch.cuda.empty_cache()
        raise optuna.exceptions.TrialPruned() # Tell Optuna to prune this trial
    except RuntimeError as e: # Catch other runtime errors too
        if "out of memory" in str(e).lower():
            print(f"Trial {trial.number} failed due to RuntimeError (likely OOM). Pruning trial. Error: {e}")
            del model
            if 'optimizer' in locals(): del optimizer
            if 'criterion' in locals(): del criterion
            if 'scheduler' in locals(): del scheduler
            torch.cuda.empty_cache()
            raise optuna.exceptions.TrialPruned()
        else:
            print(f"Trial {trial.number} failed with other RuntimeError: {e}")
            import traceback
            traceback.print_exc()
            raise # Re-raise other runtime errors
# --- HPO Study Execution ---
if __name__ == "__main__":
    data_globals_for_objective = {
        "input_features": input_features_global, # Use the dynamically set value
        "num_pos_train": num_pos_train_full,
        "num_neg_train": num_neg_train_full,
        "X_time_train_full": X_time_train_full,
        "seq_len_train_full": seq_len_train_full,
        "y_binary_train_full": y_binary_train_full,
        "X_time_test_full": X_time_test_full,
        "seq_len_test_full": seq_len_test_full,
        "y_binary_test_full": y_binary_test_full,
    }
    os.makedirs(CONFIG["results_output_file_base"], exist_ok=True)
    os.makedirs(CONFIG["tensorboard_log_dir_base"], exist_ok=True)
    # os.makedirs(CONFIG["best_model_save_path_base"], exist_ok=True) # Only if saving models per trial

    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=max(1, CONFIG["early_stopping_patience"]//2), interval_steps=1)
    study = optuna.create_study(direction="minimize", pruner=pruner)
    objective_with_config = functools.partial(objective, fixed_config=CONFIG, data_globals=data_globals_for_objective)

    print(f"Starting HPO study for TRANSFORMER with {CONFIG['n_hpo_trials']} trials...")
    try:
        study.optimize(objective_with_config, n_trials=CONFIG['n_hpo_trials'])
    except KeyboardInterrupt: print("HPO study interrupted by user.")
    except Exception as e: print(f"An error occurred during HPO study: {e}"); import traceback; traceback.print_exc()

    print("\n--- HPO Study Finished ---")
    print(f"Number of finished trials: {len(study.trials)}")
    if len(study.trials) > 0 and hasattr(study, 'best_trial') and study.best_trial is not None:
        best_trial = study.best_trial
        print("\nBest trial for Transformer:")
        print(f"  Value (Best Validation Loss): {best_trial.value:.4f}")
        print("  Params: "); [print(f"    {key}: {value}") for key, value in best_trial.params.items()]
        best_params_path = os.path.join(CONFIG["results_output_file_base"], "best_transformer_hpo_params.json")
        import json
        with open(best_params_path, 'w') as f: json.dump(best_trial.params, f, indent=4)
        print(f"Best hyperparameters for Transformer saved to {best_params_path}")
    else: print("No successful trials completed or no best trial found.")
    print("\nTo train a final Transformer model with these best hyperparameters, update your CONFIG dict,")
    print("set num_epochs to a higher value, and run a standard (non-HPO) training script.")
    study_results_df = study.trials_dataframe()
    study_results_path = os.path.join(CONFIG["results_output_file_base"], "hpo_transformer_study_results.csv")
    study_results_df.to_csv(study_results_path, index=False)
    print(f"Full HPO study results for Transformer saved to {study_results_path}")

print("Script finished.")

Using device: cuda
Original train size: 43271
Test size: 10818
Training set: Positive AKI cases: 2595.0, Negative AKI cases: 40676.0


[I 2025-05-24 18:57:52,115] A new study created in memory with name: no-name-72a6b272-4e52-47e0-9c05-3e72529c3c47


Number of input time-series features: 24
Starting HPO study for TRANSFORMER with 50 trials...

--- Trial 0 Starting ---
Hyperparameters for Transformer: {'learning_rate': 0.0003545894833225361, 'd_model': 64, 'num_encoder_layers': 4, 'num_attention_heads': 8, 'dim_feedforward': 128, 'transformer_dropout_rate': 0.1, 'weight_decay': 2.1702727571874046e-05}


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Trial 0, Epoch [10/50], Train Loss: 0.9473, Val Loss: 1.1963


...

Trial 48, Epoch [10/50], Train Loss: 1.0466, Val Loss: 1.1607


[I 2025-05-25 06:14:55,461] Trial 48 finished with value: 1.1318838596343994 and parameters: {'d_model': 64, 'num_encoder_layers': 4, 'num_attention_heads': 8, 'ff_multiplier': 4, 'learning_rate': 0.000617018713120148, 'transformer_dropout_rate': 0.25, 'weight_decay': 7.855573318679067e-05}. Best is trial 34 with value: 1.1163101196289062.


Trial 48 early stopping at epoch 18 (val_loss: 1.2140)
Trial 48 finished. Best Val Loss: 1.1319, Final Epoch Val AUPRC: 0.2390

--- Trial 49 Starting ---
Hyperparameters for Transformer: {'learning_rate': 6.578685404404615e-05, 'd_model': 32, 'num_encoder_layers': 4, 'num_attention_heads': 8, 'dim_feedforward': 64, 'transformer_dropout_rate': 0.15000000000000002, 'weight_decay': 5.0626975328029995e-05}


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
[I 2025-05-25 06:24:44,796] Trial 49 pruned. 


Trial 49 pruned at epoch 6 (val_loss: 1.1848).

--- HPO Study Finished ---
Number of finished trials: 50

Best trial for Transformer:
  Value (Best Validation Loss): 1.1163
  Params: 
    d_model: 32
    num_encoder_layers: 4
    num_attention_heads: 2
    ff_multiplier: 4
    learning_rate: 0.00042587898767975377
    transformer_dropout_rate: 0.2
    weight_decay: 3.593168387199891e-05
Best hyperparameters for Transformer saved to /home/server/Projects/data/AKI/results/hpo_transformer/best_transformer_hpo_params.json

To train a final Transformer model with these best hyperparameters, update your CONFIG dict,
set num_epochs to a higher value, and run a standard (non-HPO) training script.
Full HPO study results for Transformer saved to /home/server/Projects/data/AKI/results/hpo_transformer/hpo_transformer_study_results.csv
Script finished.


# TCN

Working - incredibly slow because utilizing only 37% of cpu due to inability to use causal padding even though pytorch 2.5.1 should support. May be due to cuda mismatch of 11.8 vs 12.4 for building PyTorch. 

Epoch [23/100], Train Loss: 0.9650, Val Loss: 1.2047, Val AUC: 0.7468, Val AUPRC: 0.2255
Validation loss did not improve for 15 epoch(s).
Early stopping triggered after 23 epochs.
Training finished.
Loading best model from /home/server/Projects/data/AKI/models/best_tcn_model.pth for final evaluation.

Starting final evaluation with the best/last model...
                                                                         

--- Final Test Performance (Best/Last Model) ---
Precision: 0.0811
Sensitivity: 0.8921
Accuracy: 0.3870
rc_auc: 0.7365
pr_auc: 0.2170
Specificity: 0.3548
Negative Predictive Value: 0.9810
F1 Score: 0.1487
Results saved to /home/server/Projects/data/AKI/results/tcn_model_results.pkl
Script finished.


In [1]:
# %%
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import weight_norm # For TCN weight normalization
from torch.utils.tensorboard import SummaryWriter
import torch.optim as optim

import random
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_curve, auc, precision_recall_curve, average_precision_score)
from sklearn.model_selection import train_test_split
import pandas as pd
# Ensure utils.py is in the same directory or Python path
from utils import performance_dict, sigmoid
from tqdm import tqdm

# --- Configuration for the TCN Model ---
CONFIG = {
    "data_file": '/home/server/Projects/data/AKI/lstm_trainable.pkl', # !!! UPDATE THIS PATH !!!
    "results_output_file": '/home/server/Projects/data/AKI/results/tcn_model_results.pkl',
    "tensorboard_log_dir_base": '/home/server/Projects/data/AKI/runs/tcn_model/',
    "best_model_save_path": '/home/server/Projects/data/AKI/models/best_tcn_model.pth',
    "test_split_size": 0.2,
    "random_seed": 42,
    "num_epochs": 100,
    "learning_rate": 0.0005, # TCNs can sometimes use slightly higher LRs than Transformers
    "weight_decay": 1e-5,
    "gradient_clip_value": 1.0,
    "lr_scheduler_patience": 7,
    "lr_scheduler_factor": 0.1,
    "early_stopping_patience": 15,
    
    # --- Architecture Specifics for TCN ---
    "input_features": -1, # Will be set dynamically
    "tcn_num_channels": [64, 96, 128, 128], # Number of channels in each TCN layer/block stack
                                           # The last value is the output channels from TCN
    "tcn_kernel_size": 3,
    "tcn_dropout_rate": 0.2,
    
    "batch_size": 1024, # TCNs can also be memory intensive depending on depth/channels
    "eval_batch_size": 512,
    "classification_threshold": 0.3,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load data (once)
try:
    df_full = pd.read_pickle(CONFIG["data_file"])
except FileNotFoundError:
    print(f"ERROR: Data file not found at {CONFIG['data_file']}")
    print("Please ensure the path in CONFIG['data_file'] is correct.")
    exit()
except Exception as e:
    print(f"Error loading data file {CONFIG['data_file']}: {e}")
    exit()

# %%
# --- Data Preparation ---
def df_to_tensors(df, all_columns):
    non_tabular_cols = ['op_id', 'time_tensors', 'seq_len', 'aki', 'aki_boolean']
    tabular_feature_cols = [col for col in all_columns if col not in non_tabular_cols]
    if tabular_feature_cols and not df[tabular_feature_cols].empty:
        X_tab = torch.tensor(df[tabular_feature_cols].values).float()
    else:
        X_tab = torch.empty((len(df), 0)).float()
    X_time = torch.stack(df['time_tensors'].tolist()).float()
    y = torch.tensor(df['aki'].values).unsqueeze(1).float()
    y_binary = df['aki_boolean'].values.astype(bool)
    sequence_lengths = torch.tensor(df['seq_len'].tolist()).long() # We might not use this directly for TCN if using global pooling
    return X_tab, X_time, sequence_lengths, y, y_binary

def prepare_data(full_df, test_size, random_seed):
    df_train, df_test = train_test_split(full_df, test_size=test_size, random_state=random_seed, stratify=full_df['aki_boolean'])
    print(f"Original train size: {len(df_train)}")
    print(f"Test size: {len(df_test)}")
    all_columns = full_df.columns.tolist()
    num_positive_aki_train = df_train['aki_boolean'].sum()
    num_negative_aki_train = len(df_train) - num_positive_aki_train
    print(f"Training set: Positive AKI cases: {num_positive_aki_train}, Negative AKI cases: {num_negative_aki_train}")
    X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train = df_to_tensors(df_train, all_columns)
    X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test = df_to_tensors(df_test, all_columns)
    
    CONFIG["input_features"] = X_time_train.shape[2]
    print(f"Number of input time-series features: {CONFIG['input_features']}")
    
    return (X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train,
            X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test,
            num_positive_aki_train, num_negative_aki_train)

(X_tab_train_full, X_time_train_full, seq_len_train_full, y_train_full, y_binary_train_full,
 X_tab_test_full, X_time_test_full, seq_len_test_full, y_test_full, y_binary_test_full,
 num_pos_train_full, num_neg_train_full) = prepare_data(
    df_full, CONFIG["test_split_size"], CONFIG["random_seed"]
)
input_features_global = CONFIG["input_features"]

# %%
# --- Utility Functions ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["random_seed"])

def save_single_run_results(model_name, metrics_dict, output_file):
    results_to_save = {k: [v] if not isinstance(v, (list, np.ndarray)) else v for k,v in metrics_dict.items()}
    df_result = pd.DataFrame(results_to_save)
    df_result['model_name'] = model_name
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    if os.path.exists(output_file):
        try:
            df_output = pd.read_pickle(output_file)
            df_output = df_output[df_output['model_name'] != model_name]
            df_output = pd.concat([df_output, df_result], ignore_index=True)
        except EOFError: df_output = df_result
    else: df_output = df_result
    df_output.to_pickle(output_file)
    print(f"Results saved to {output_file}")

# %%
# --- TCN Model Definition ---

class CausalConv1d(nn.Conv1d):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, dilation=1, groups=1, bias=True):
        super(CausalConv1d, self).__init__(in_channels, out_channels, kernel_size, stride, 0, dilation, groups, bias)
        # Calculate left padding: (kernel_size - 1) * dilation
        self.left_padding = (kernel_size - 1) * dilation

    def forward(self, input):
        # Pad input on the left for causality
        x = F.pad(input, (self.left_padding, 0))
        return super(CausalConv1d, self).forward(x)

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, dropout=0.2):
        super(TemporalBlock, self).__init__()
        self.conv1 = weight_norm(CausalConv1d(n_inputs, n_outputs, kernel_size,
                                           stride=stride, dilation=dilation))
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = weight_norm(CausalConv1d(n_outputs, n_outputs, kernel_size,
                                           stride=stride, dilation=dilation))
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.conv1, self.relu1, self.dropout1,
                                 self.conv2, self.relu2, self.dropout2)
        
        # 1x1 convolution for residual connection if #channels change
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU() # Final ReLU for the block

    def forward(self, x):
        res = x if self.downsample is None else self.downsample(x)
        out = self.net(x)
        return self.relu(out + res) # Add residual

class TCNModel(nn.Module):
    def __init__(self, input_size, num_classes, num_channels, kernel_size, dropout):
        super(TCNModel, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i # Dilation increases exponentially with depth
            in_channels = input_size if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size, stride=1,
                                     dilation=dilation_size, dropout=dropout)]
        
        self.network = nn.Sequential(*layers)
        
        # Classifier Head
        # The output of TCN is (batch, num_channels[-1], seq_len)
        # We will use global average pooling over the sequence length
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1) # Output (batch, num_channels[-1], 1)
        
        tcn_output_dim = num_channels[-1]
        self.fc_intermediate_dim = max(1, tcn_output_dim // 2)
        self.classifier_head = nn.Sequential(
            nn.Linear(tcn_output_dim, self.fc_intermediate_dim),
            nn.ReLU(),
            nn.Dropout(dropout), # Use the same TCN dropout for classifier or define a new one
            nn.Linear(self.fc_intermediate_dim, num_classes)
        )

    def forward(self, x_time, seq_lens=None): # seq_lens is not directly used by TCN with global pooling
        # Input x_time expected shape: (batch, seq_len, input_features)
        # nn.Conv1d expects: (batch, input_features, seq_len)
        x = x_time.permute(0, 2, 1) # (batch, input_features, seq_len)
        
        tcn_output = self.network(x) # (batch, num_channels[-1], seq_len)
        
        # Global Average Pooling over the sequence length dimension
        pooled_output = self.global_avg_pool(tcn_output) # (batch, num_channels[-1], 1)
        pooled_output = pooled_output.squeeze(-1) # (batch, num_channels[-1])
        
        logits = self.classifier_head(pooled_output) # (batch, num_classes)
        return logits

# %%
# --- Main Training and Evaluation ---
model = TCNModel(
    input_size=input_features_global,
    num_classes=1, # Binary classification
    num_channels=CONFIG["tcn_num_channels"],
    kernel_size=CONFIG["tcn_kernel_size"],
    dropout=CONFIG["tcn_dropout_rate"]
).to(device)

print(model)
print(f"TCN Model Parameter Count: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

if num_pos_train_full > 0:
    pos_weight_value = num_neg_train_full / num_pos_train_full
else:
    pos_weight_value = 1.0; print("Warning: No positive samples in train for pos_weight.")
pos_weight = torch.tensor([pos_weight_value], device=device)
print(f"Calculated pos_weight for BCEWithLogitsLoss: {pos_weight.item():.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=CONFIG["lr_scheduler_factor"],
    patience=CONFIG["lr_scheduler_patience"])

run_name = f"TCN_k{CONFIG['tcn_kernel_size']}_chan{'_'.join(map(str, CONFIG['tcn_num_channels']))}_classWeighted"
tensorboard_run_path = os.path.join(CONFIG["tensorboard_log_dir_base"], run_name)
os.makedirs(tensorboard_run_path, exist_ok=True)
writer = SummaryWriter(tensorboard_run_path)

num_train_samples = X_time_train_full.shape[0]
train_batch_size = CONFIG["batch_size"]
num_train_batches = int(np.ceil(num_train_samples / train_batch_size))
eval_batch_size = CONFIG["eval_batch_size"]
num_test_samples = X_time_test_full.shape[0]
num_eval_batches = int(np.ceil(num_test_samples / eval_batch_size))

best_val_loss = float('inf')
epochs_no_improve = 0
os.makedirs(os.path.dirname(CONFIG["best_model_save_path"]), exist_ok=True)

print(f"Starting training for {CONFIG['num_epochs']} epochs on {num_train_samples} original samples...")
for epoch in tqdm(range(CONFIG["num_epochs"]), desc="Epochs"):
    model.train()
    epoch_train_loss = 0.0
    for i in tqdm(range(num_train_batches), desc="Training Batches", leave=False):
        start_idx = i * train_batch_size
        end_idx = min(num_train_samples, (i + 1) * train_batch_size)
        batch_x_time = X_time_train_full[start_idx:end_idx].to(device)
        # batch_seq_len = seq_len_train_full[start_idx:end_idx] # Not directly used by TCN forward if global pooling
        batch_y_binary = torch.tensor(y_binary_train_full[start_idx:end_idx]).unsqueeze(1).float().to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x_time) # TCNModel might not need seq_lens if using global pooling
        loss = criterion(outputs, batch_y_binary)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CONFIG["gradient_clip_value"])
        optimizer.step()
        epoch_train_loss += loss.item()
    avg_epoch_train_loss = epoch_train_loss / num_train_batches
    writer.add_scalar('Loss/train', avg_epoch_train_loss, epoch)

    model.eval()
    all_val_outputs_logits = []; all_val_y_for_loss = []
    with torch.no_grad():
        for i in tqdm(range(num_eval_batches), desc="Validation Batches", leave=False):
            start_idx = i * eval_batch_size; end_idx = min(num_test_samples, (i + 1) * eval_batch_size)
            batch_x_time_val = X_time_test_full[start_idx:end_idx].to(device)
            # batch_seq_len_val = seq_len_test_full[start_idx:end_idx] # Not directly used
            batch_y_binary_val_for_loss = torch.tensor(y_binary_test_full[start_idx:end_idx]).unsqueeze(1).float().to(device)
            outputs_batch_logits = model(batch_x_time_val)
            all_val_outputs_logits.append(outputs_batch_logits.cpu()); all_val_y_for_loss.append(batch_y_binary_val_for_loss.cpu())
    val_outputs_logits = torch.cat(all_val_outputs_logits, dim=0); val_y_for_loss = torch.cat(all_val_y_for_loss, dim=0)
    current_val_loss = criterion(val_outputs_logits.to(device), val_y_for_loss.to(device)).item()
    writer.add_scalar('Loss/val', current_val_loss, epoch)
    val_probs = torch.sigmoid(val_outputs_logits).numpy().squeeze()
    if val_probs.ndim == 0: val_probs = np.array([val_probs])
    val_pred_binary = (val_probs > CONFIG["classification_threshold"])
    current_y_true_for_metric = y_binary_test_full[:len(val_pred_binary)]
    
    try:
        dic = performance_dict(current_y_true_for_metric, val_pred_binary, val_probs)
        writer.add_scalar('Val/AUC', dic.get('rc_auc', np.nan), epoch)
        writer.add_scalar('Val/AUPRC', dic.get('pr_auc', np.nan), epoch)
        writer.add_scalar('Val/Accuracy', dic.get('Accuracy', np.nan), epoch)
        # ... (log other metrics from dic)
        print(f"Epoch [{epoch+1}/{CONFIG['num_epochs']}], Train Loss: {avg_epoch_train_loss:.4f}, Val Loss: {current_val_loss:.4f}, Val AUC: {dic.get('rc_auc', np.nan):.4f}, Val AUPRC: {dic.get('pr_auc', np.nan):.4f}")
    except Exception as e:
        print(f"Error calculating performance_dict in validation: {e}")
        print(f"Epoch [{epoch+1}/{CONFIG['num_epochs']}], Train Loss: {avg_epoch_train_loss:.4f}, Val Loss: {current_val_loss:.4f}")

    scheduler.step(current_val_loss)
    if current_val_loss < best_val_loss:
        best_val_loss = current_val_loss; epochs_no_improve = 0
        torch.save(model.state_dict(), CONFIG["best_model_save_path"])
        print(f"Validation loss improved. Saved best model to {CONFIG['best_model_save_path']}")
    else:
        epochs_no_improve += 1
        print(f"Validation loss did not improve for {epochs_no_improve} epoch(s).")
    if epochs_no_improve >= CONFIG["early_stopping_patience"]:
        print(f"Early stopping triggered after {epoch+1} epochs.")
        break
writer.close()
print("Training finished.")

if os.path.exists(CONFIG["best_model_save_path"]):
    print(f"Loading best model from {CONFIG['best_model_save_path']} for final evaluation.")
    model.load_state_dict(torch.load(CONFIG["best_model_save_path"]))
else:
    print("No best model found. Using last model state.")

model.eval()
final_all_test_outputs_logits = []
print("\nStarting final evaluation with the best/last model...")
with torch.no_grad():
    for i in tqdm(range(num_eval_batches), desc="Final Evaluation Batches", leave=False):
        start_idx = i * eval_batch_size; end_idx = min(num_test_samples, (i + 1) * eval_batch_size)
        batch_x_time_test = X_time_test_full[start_idx:end_idx].to(device)
        # batch_seq_len_test = seq_len_test_full[start_idx:end_idx] # Not directly used
        outputs_batch_logits = model(batch_x_time_test)
        final_all_test_outputs_logits.append(outputs_batch_logits.cpu())
final_test_outputs_logits = torch.cat(final_all_test_outputs_logits, dim=0)
final_test_probs = torch.sigmoid(final_test_outputs_logits).numpy().squeeze()
if final_test_probs.ndim == 0: final_test_probs = np.array([final_test_probs])
final_y_pred_binary = (final_test_probs > CONFIG["classification_threshold"])
current_y_true_for_final_metric = y_binary_test_full[:len(final_y_pred_binary)]

try:
    final_metrics = performance_dict(current_y_true_for_final_metric, final_y_pred_binary, final_test_probs)
    print("\n--- Final Test Performance (Best/Last Model) ---")
    for metric, value in final_metrics.items():
        if isinstance(value, (float, int, np.number)): print(f"{metric}: {value:.4f}")
    save_single_run_results(run_name, final_metrics, CONFIG["results_output_file"])
except Exception as e:
    print(f"Error calculating final performance_dict: {e}")

print("Script finished.")

Using device: cuda
Original train size: 43271
Test size: 10818
Training set: Positive AKI cases: 2595.0, Negative AKI cases: 40676.0
Number of input time-series features: 24
TCNModel(
  (network): Sequential(
    (0): TemporalBlock(
      (conv1): CausalConv1d(24, 64, kernel_size=(3,), stride=(1,))
      (relu1): ReLU()
      (dropout1): Dropout(p=0.2, inplace=False)
      (conv2): CausalConv1d(64, 64, kernel_size=(3,), stride=(1,))
      (relu2): ReLU()
      (dropout2): Dropout(p=0.2, inplace=False)
      (net): Sequential(
        (0): CausalConv1d(24, 64, kernel_size=(3,), stride=(1,))
        (1): ReLU()
        (2): Dropout(p=0.2, inplace=False)
        (3): CausalConv1d(64, 64, kernel_size=(3,), stride=(1,))
        (4): ReLU()
        (5): Dropout(p=0.2, inplace=False)
      )
      (downsample): Conv1d(24, 64, kernel_size=(1,), stride=(1,))
      (relu): ReLU()
    )
    (1): TemporalBlock(
      (conv1): CausalConv1d(64, 96, kernel_size=(3,), stride=(1,), dilation=(2,))
     

/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Starting training for 100 epochs on 43271 original samples...


Epochs:   0%|          | 0/100 [00:20<?, ?it/s]


KeyboardInterrupt: 

# HPO TCN

In [ ]:
# %%
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import weight_norm # For TCN weight normalization
from torch.utils.tensorboard import SummaryWriter
import torch.optim as optim
import optuna # Optuna import
import functools # For passing arguments to objective function
import math

import random
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_curve, auc, precision_recall_curve, average_precision_score)
from sklearn.model_selection import train_test_split
import pandas as pd
# Ensure utils.py is in the same directory or Python path
from utils import performance_dict, sigmoid
from tqdm import tqdm

# --- Configuration ---
CONFIG = {
    "data_file": '/home/server/Projects/data/AKI/lstm_trainable.pkl', # !!! UPDATE THIS PATH !!!
    "results_output_file_base": '/home/server/Projects/data/AKI/results/hpo_tcn/',
    "tensorboard_log_dir_base": '/home/server/Projects/data/AKI/runs/hpo_tcn/',
    # "best_model_save_path_base": '/home/server/Projects/data/AKI/models/hpo_tcn/', # Optional per-trial model saving
    "test_split_size": 0.2,
    "random_seed": 42,
    "num_epochs_hpo": 40,  # Max epochs PER HPO TRIAL (TCNs with manual pad can be slow)
    "gradient_clip_value": 1.0,
    "lr_scheduler_patience": 5,
    "lr_scheduler_factor": 0.2,
    "early_stopping_patience": 10, # Per-trial early stopping
    
    "input_features": -1, # Will be set dynamically
    
    "batch_size": 256, # Adjusted for TCN memory with manual padding
    "eval_batch_size": 256,
    "classification_threshold": 0.3,
    "n_hpo_trials": 30, # Number of HPO trials (adjust based on time)
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

# Load data (once)
try:
    df_full = pd.read_pickle(CONFIG["data_file"])
except FileNotFoundError:
    print(f"ERROR: Data file not found at {CONFIG['data_file']}")
    print("Please ensure the path in CONFIG['data_file'] is correct.")
    exit()
except Exception as e:
    print(f"Error loading data file {CONFIG['data_file']}: {e}")
    exit()


# %%
# --- Data Preparation ---
def df_to_tensors(df, all_columns):
    non_tabular_cols = ['op_id', 'time_tensors', 'seq_len', 'aki', 'aki_boolean']
    tabular_feature_cols = [col for col in all_columns if col not in non_tabular_cols]
    if tabular_feature_cols and not df[tabular_feature_cols].empty:
        X_tab = torch.tensor(df[tabular_feature_cols].values).float()
    else:
        X_tab = torch.empty((len(df), 0)).float()
    X_time = torch.stack(df['time_tensors'].tolist()).float()
    y = torch.tensor(df['aki'].values).unsqueeze(1).float()
    y_binary = df['aki_boolean'].values.astype(bool)
    sequence_lengths = torch.tensor(df['seq_len'].tolist()).long()
    return X_tab, X_time, sequence_lengths, y, y_binary

def prepare_data(full_df, test_size, random_seed):
    df_train, df_test = train_test_split(full_df, test_size=test_size, random_state=random_seed, stratify=full_df['aki_boolean'])
    print(f"Original train size: {len(df_train)}")
    print(f"Test size: {len(df_test)}")
    all_columns = full_df.columns.tolist()
    num_positive_aki_train = df_train['aki_boolean'].sum()
    num_negative_aki_train = len(df_train) - num_positive_aki_train
    print(f"Training set: Positive AKI cases: {num_positive_aki_train}, Negative AKI cases: {num_negative_aki_train}")
    X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train = df_to_tensors(df_train, all_columns)
    X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test = df_to_tensors(df_test, all_columns)
    
    CONFIG["input_features"] = X_time_train.shape[2]
    print(f"Number of input time-series features: {CONFIG['input_features']}")
    
    return (X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train,
            X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test,
            num_positive_aki_train, num_negative_aki_train)

(X_tab_train_full, X_time_train_full, seq_len_train_full, y_train_full, y_binary_train_full,
 X_tab_test_full, X_time_test_full, seq_len_test_full, y_test_full, y_binary_test_full,
 num_pos_train_full, num_neg_train_full) = prepare_data(
    df_full, CONFIG["test_split_size"], CONFIG["random_seed"]
)
input_features_global = CONFIG["input_features"]


# %%
# --- Utility Functions ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["random_seed"])

# (save_hpo_trial_results can be added if needed for detailed trial logging)

# %%
# --- TCN Model Definition (Using Manual Padding CausalConv1d) ---

class CausalConv1d(nn.Conv1d):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, dilation=1, groups=1, bias=True):
        super(CausalConv1d, self).__init__(in_channels, out_channels, kernel_size, stride, 0, dilation, groups, bias)
        self.left_padding = (kernel_size - 1) * dilation

    def forward(self, input):
        x = F.pad(input, (self.left_padding, 0))
        return super(CausalConv1d, self).forward(x)

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, dropout=0.2):
        super(TemporalBlock, self).__init__()
        self.conv1 = weight_norm(CausalConv1d(n_inputs, n_outputs, kernel_size,
                                           stride=stride, dilation=dilation))
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = weight_norm(CausalConv1d(n_outputs, n_outputs, kernel_size,
                                           stride=stride, dilation=dilation))
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.conv1, self.relu1, self.dropout1,
                                 self.conv2, self.relu2, self.dropout2)
        
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()

    def forward(self, x):
        res = x if self.downsample is None else self.downsample(x)
        out = self.net(x)
        return self.relu(out + res)

class TCNModel(nn.Module):
    def __init__(self, input_size, num_classes, num_channels_list, kernel_size, dropout):
        super(TCNModel, self).__init__()
        layers = []
        num_levels = len(num_channels_list)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = input_size if i == 0 else num_channels_list[i-1]
            out_channels = num_channels_list[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size, stride=1,
                                     dilation=dilation_size, dropout=dropout)]
        
        self.network = nn.Sequential(*layers)
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        
        tcn_output_dim = num_channels_list[-1]
        self.fc_intermediate_dim = max(1, tcn_output_dim // 2)
        self.classifier_head = nn.Sequential(
            nn.Linear(tcn_output_dim, self.fc_intermediate_dim),
            nn.ReLU(),
            nn.Dropout(dropout), # Use the main dropout for classifier as well
            nn.Linear(self.fc_intermediate_dim, num_classes)
        )

    def forward(self, x_time, seq_lens=None): # seq_lens not actively used here
        x = x_time.permute(0, 2, 1)
        tcn_output = self.network(x)
        pooled_output = self.global_avg_pool(tcn_output)
        pooled_output = pooled_output.squeeze(-1)
        logits = self.classifier_head(pooled_output)
        return logits

# %%
# --- Optuna Objective Function for TCN ---
def objective(trial, fixed_config, data_globals):
    cfg_trial = {
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True), # TCNs might tolerate slightly higher LRs
        "tcn_num_layers": trial.suggest_int("tcn_num_layers", 2, 4), # Number of TemporalBlocks
        "tcn_base_channels": trial.suggest_categorical("tcn_base_channels", [32, 48, 64, 80]), # Base num channels
        "tcn_channel_multiplier": trial.suggest_float("tcn_channel_multiplier", 1.0, 1.5, step=0.25), # Multiplier for deeper layers
        "tcn_kernel_size": trial.suggest_categorical("tcn_kernel_size", [2, 3, 5]),
        "tcn_dropout_rate": trial.suggest_float("tcn_dropout_rate", 0.1, 0.4, step=0.05),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-4, log=True),
    }

    # Construct tcn_num_channels list based on suggestions
    num_channels_list = []
    current_channels = cfg_trial["tcn_base_channels"]
    for i in range(cfg_trial["tcn_num_layers"]):
        num_channels_list.append(int(current_channels))
        current_channels *= cfg_trial["tcn_channel_multiplier"]
        current_channels = max(16, int(current_channels)) # Ensure channels don't get too small if multiplier < 1
        current_channels = min(128, int(current_channels))# Cap max channels to prevent excessive memory

    # Ensure the list isn't empty if num_layers was small
    if not num_channels_list:
        num_channels_list = [cfg_trial["tcn_base_channels"]]


    print(f"\n--- Trial {trial.number} Starting ---")
    print(f"Hyperparameters for TCN: {cfg_trial}")
    print(f"Generated tcn_num_channels: {num_channels_list}")


    try:
        model = TCNModel(
            input_size=data_globals["input_features"],
            num_classes=1,
            num_channels_list=num_channels_list,
            kernel_size=cfg_trial["tcn_kernel_size"],
            dropout=cfg_trial["tcn_dropout_rate"]
        ).to(device)

        if data_globals["num_pos_train"] > 0:
            pos_weight_value = data_globals["num_neg_train"] / data_globals["num_pos_train"]
        else: pos_weight_value = 1.0
        pos_weight = torch.tensor([pos_weight_value], device=device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = optim.Adam(model.parameters(), lr=cfg_trial["learning_rate"], weight_decay=cfg_trial["weight_decay"])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=fixed_config["lr_scheduler_factor"],
            patience=fixed_config["lr_scheduler_patience"], verbose=False )

        num_train_samples = data_globals["X_time_train_full"].shape[0]
        train_batch_size = fixed_config["batch_size"]
        num_train_batches = int(np.ceil(num_train_samples / train_batch_size))
        eval_batch_size = fixed_config["eval_batch_size"]
        num_test_samples = data_globals["X_time_test_full"].shape[0]
        num_eval_batches = int(np.ceil(num_test_samples / eval_batch_size))

        best_val_loss_for_trial = float('inf')
        epochs_no_improve_for_trial = 0
        
        for epoch in range(fixed_config["num_epochs_hpo"]):
            model.train()
            epoch_train_loss = 0.0
            for i in range(num_train_batches):
                start_idx = i * train_batch_size; end_idx = min(num_train_samples, (i + 1) * train_batch_size)
                batch_x_time = data_globals["X_time_train_full"][start_idx:end_idx].to(device)
                # batch_seq_len = data_globals["seq_len_train_full"][start_idx:end_idx] # Not used by TCNModel.forward
                batch_y_binary = torch.tensor(data_globals["y_binary_train_full"][start_idx:end_idx]).unsqueeze(1).float().to(device)
                optimizer.zero_grad(); outputs = model(batch_x_time); loss = criterion(outputs, batch_y_binary)
                loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=fixed_config["gradient_clip_value"]); optimizer.step()
                epoch_train_loss += loss.item()
            avg_epoch_train_loss = epoch_train_loss / num_train_batches if num_train_batches > 0 else 0.0

            model.eval()
            all_val_outputs_logits = []; all_val_y_for_loss = []
            with torch.no_grad():
                for i in range(num_eval_batches):
                    start_idx = i * eval_batch_size; end_idx = min(num_test_samples, (i + 1) * eval_batch_size)
                    batch_x_time_val = data_globals["X_time_test_full"][start_idx:end_idx].to(device)
                    # batch_seq_len_val = data_globals["seq_len_test_full"][start_idx:end_idx]
                    batch_y_binary_val_for_loss = torch.tensor(data_globals["y_binary_test_full"][start_idx:end_idx]).unsqueeze(1).float().to(device)
                    outputs_batch_logits = model(batch_x_time_val)
                    all_val_outputs_logits.append(outputs_batch_logits.cpu()); all_val_y_for_loss.append(batch_y_binary_val_for_loss.cpu())
            
            if not all_val_outputs_logits: current_val_loss = float('inf')
            else:
                val_outputs_logits = torch.cat(all_val_outputs_logits, dim=0); val_y_for_loss = torch.cat(all_val_y_for_loss, dim=0)
                current_val_loss = criterion(val_outputs_logits.to(device), val_y_for_loss.to(device)).item()

            scheduler.step(current_val_loss)
            if current_val_loss < best_val_loss_for_trial:
                best_val_loss_for_trial = current_val_loss; epochs_no_improve_for_trial = 0
            else: epochs_no_improve_for_trial += 1
            if epochs_no_improve_for_trial >= fixed_config["early_stopping_patience"]:
                print(f"Trial {trial.number} early stopping at epoch {epoch+1} (val_loss: {current_val_loss:.4f})")
                break
            trial.report(current_val_loss, epoch)
            if trial.should_prune():
                print(f"Trial {trial.number} pruned at epoch {epoch+1} (val_loss: {current_val_loss:.4f}).")
                raise optuna.exceptions.TrialPruned()
            if (epoch + 1) % 10 == 0 or epoch == fixed_config["num_epochs_hpo"] -1 :
                 print(f"Trial {trial.number}, Epoch [{epoch+1}/{fixed_config['num_epochs_hpo']}], Train Loss: {avg_epoch_train_loss:.4f}, Val Loss: {current_val_loss:.4f}")

        # Final metric calculation for logging (optional, Optuna uses returned value)
        if 'val_outputs_logits' in locals() and val_outputs_logits is not None and len(val_outputs_logits) > 0:
            val_probs = torch.sigmoid(val_outputs_logits).numpy().squeeze()
            if val_probs.ndim == 0: val_probs = np.array([val_probs])
            val_pred_binary = (val_probs > fixed_config["classification_threshold"])
            current_y_true_for_metric = data_globals["y_binary_test_full"][:len(val_probs)]
            try:
                dic = performance_dict(current_y_true_for_metric, val_pred_binary, val_probs)
                val_auprc = dic.get('pr_auc', 0.0)
            except Exception as e:
                print(f"Warning: Could not compute performance_dict for trial {trial.number} final metrics: {e}"); val_auprc = 0.0
            print(f"Trial {trial.number} finished. Best Val Loss: {best_val_loss_for_trial:.4f}, Final Epoch Val AUPRC: {val_auprc:.4f}")
        else:
             print(f"Trial {trial.number} finished. Best Val Loss: {best_val_loss_for_trial:.4f} (No validation outputs for AUPRC).")
        
        return best_val_loss_for_trial

    except torch.cuda.OutOfMemoryError:
        print(f"Trial {trial.number} failed due to CUDA OutOfMemoryError. Pruning trial.")
        torch.cuda.empty_cache()
        raise optuna.exceptions.TrialPruned()
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"Trial {trial.number} failed due to RuntimeError (likely OOM). Pruning trial. Error: {e}")
            torch.cuda.empty_cache()
            raise optuna.exceptions.TrialPruned()
        else:
            print(f"Trial {trial.number} failed with other RuntimeError: {e}"); import traceback; traceback.print_exc(); raise

# %%
# --- HPO Study Execution ---
if __name__ == "__main__":
    data_globals_for_objective = {
        "input_features": input_features_global,
        "num_pos_train": num_pos_train_full,
        "num_neg_train": num_neg_train_full,
        "X_time_train_full": X_time_train_full,
        "seq_len_train_full": seq_len_train_full, # Not strictly used by this TCNModel.forward
        "y_binary_train_full": y_binary_train_full,
        "X_time_test_full": X_time_test_full,
        "seq_len_test_full": seq_len_test_full,   # Not strictly used
        "y_binary_test_full": y_binary_test_full,
    }
    os.makedirs(CONFIG["results_output_file_base"], exist_ok=True)
    os.makedirs(CONFIG["tensorboard_log_dir_base"], exist_ok=True)

    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=max(1, CONFIG["early_stopping_patience"]//2), interval_steps=1)
    study = optuna.create_study(direction="minimize", pruner=pruner)
    objective_with_config = functools.partial(objective, fixed_config=CONFIG, data_globals=data_globals_for_objective)

    print(f"Starting HPO study for TCN with {CONFIG['n_hpo_trials']} trials...")
    try:
        study.optimize(objective_with_config, n_trials=CONFIG['n_hpo_trials'])
    except KeyboardInterrupt: print("HPO study interrupted by user.")
    except Exception as e: print(f"An error occurred during HPO study: {e}"); import traceback; traceback.print_exc()

    print("\n--- HPO Study Finished ---")
    print(f"Number of finished trials: {len(study.trials)}")
    if len(study.trials) > 0 and hasattr(study, 'best_trial') and study.best_trial is not None:
        best_trial = study.best_trial
        print("\nBest trial for TCN:"); print(f"  Value (Best Validation Loss): {best_trial.value:.4f}")
        print("  Params: "); [print(f"    {key}: {value}") for key, value in best_trial.params.items()]
        best_params_path = os.path.join(CONFIG["results_output_file_base"], "best_tcn_hpo_params.json")
        import json
        with open(best_params_path, 'w') as f: json.dump(best_trial.params, f, indent=4)
        print(f"Best hyperparameters for TCN saved to {best_params_path}")
    else: print("No successful trials completed or no best trial found.")
    print("\nTo train a final TCN model with these best hyperparameters, update your CONFIG dict,")
    print("set num_epochs to a higher value, and run a standard (non-HPO) training script.")
    study_results_df = study.trials_dataframe()
    study_results_path = os.path.join(CONFIG["results_output_file_base"], "hpo_tcn_study_results.csv")
    study_results_df.to_csv(study_results_path, index=False)
    print(f"Full HPO study results for TCN saved to {study_results_path}")

print("Script finished.")

Using device: cuda
PyTorch version: 2.5.1+cu124
Original train size: 43271
Test size: 10818
Training set: Positive AKI cases: 2595.0, Negative AKI cases: 40676.0


[I 2025-05-25 06:41:23,852] A new study created in memory with name: no-name-8d862b45-935b-4d5e-a663-392d5c9a8ec6


Number of input time-series features: 24
Starting HPO study for TCN with 30 trials...

--- Trial 0 Starting ---
Hyperparameters for TCN: {'learning_rate': 0.004354488348877075, 'tcn_num_layers': 2, 'tcn_base_channels': 32, 'tcn_channel_multiplier': 1.25, 'tcn_kernel_size': 3, 'tcn_dropout_rate': 0.30000000000000004, 'weight_decay': 4.63437343388264e-06}
Generated tcn_num_channels: [32, 40]


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Trial 0, Epoch [10/40], Train Loss: 1.0689, Val Loss: 1.1332
Trial 0, Epoch [20/40], Train Loss: nan, Val Loss: nan


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` paramete

Trial 0 early stopping at epoch 21 (val_loss: nan)
Trial 0 finished. Best Val Loss: 1.1269, Final Epoch Val AUPRC: 0.0000

--- Trial 1 Starting ---
Hyperparameters for TCN: {'learning_rate': 0.00010016105639519034, 'tcn_num_layers': 2, 'tcn_base_channels': 80, 'tcn_channel_multiplier': 1.5, 'tcn_kernel_size': 2, 'tcn_dropout_rate': 0.25, 'weight_decay': 4.743673052500304e-05}
Generated tcn_num_channels: [80, 120]


/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Trial 1, Epoch [10/40], Train Loss: 1.1221, Val Loss: 1.1512


# MLP + LSTM Hybrid Testing

Optimized MLP + LSTM Hybrid Model. Runs once

In [3]:
#!/usr/bin/env python3
import os
import time
import copy
import numpy as np
import pandas as pd
from tqdm import tqdm

# Scikit-learn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import TensorDataset, DataLoader

# =============================================================================
# SCRIPT CONFIGURATION
# =============================================================================
TARGET = 'aki_boolean'
RANDOM_STATE = 42
USE_BOOTSTRAPPING = False # Set to True to run the full 25-iteration cross-validation
N_BOOTSTRAP_ITERATIONS = 25

# --- I/O Configuration ---
BASE_DATA_DIR = '/home/server/Projects/data/AKI/'
LSTM_INPUT_PKL = os.path.join(BASE_DATA_DIR, 'lstm_trainable.pkl')
MLP_INPUT_CSV = os.path.join(BASE_DATA_DIR, 'tabular_preop.csv')

# Main consolidated results file
RESULTS_PKL = os.path.join(BASE_DATA_DIR, 'results/lstm_hybrid_test_optimized.pkl')

# ADDED: Additional output files for specific models
INTRAOP_RESULTS_PKL = os.path.join(BASE_DATA_DIR, 'results/tabular_intraop_test.pkl')
COMBINED_RESULTS_PKL = os.path.join(BASE_DATA_DIR, 'results/tabular_combined_test.pkl')


# --- Model Run Toggles ---
model_configs = {
    'lstm_only': True,
    'mlp_only': False,
    'hybrid': True,
}

# =============================================================================
# HYPERPARAMETER CONFIGURATION
# =============================================================================

# --- Default Hyperparameters (Used if HPO dictionaries are empty) ---
default_hyperparameters = {
    'learning_rate': 0.001,
    'epochs': 1000,
    'batch_size': 1024,
    'patience': 20,
    'es_check_interval': 5,
    'lr_scheduler_patience': 7,
    'lr_scheduler_factor': 0.1,
    'gradient_clip_value': 1.0,
    # Default architecture
    'lstm_hidden_size': 128,
    'lstm_num_layers': 2,
    'mlp_dims': [256, 128, 64],
    'dropout_rate': 0.4
}

# --- HPO RESULTS: PASTE YOUR OPTIMIZED PARAMETERS HERE ---
# After running the HPO script, copy the output dictionary and paste it here.
# If a dictionary is left empty, the script will use the default_hyperparameters above for that model.

hpo_params_lstm_only = {
    'lr': 0.000015,
    'dropout_rate': 0.363015,
    'lstm_hidden_size': 50,
    'lstm_num_layers': 4,
}

hpo_params_hybrid = {
    'mlp_dims': [120, 155],
    'lr': 0.000017,
    'dropout_rate': 0.467438,
    'lstm_hidden_size': 66,
    'lstm_num_layers': 2,
}


# Set seed for reproducibility
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
def performance_dict(y_true, y_pred_binary, y_prob):
    """Calculates a comprehensive dictionary of performance metrics."""
    if len(np.unique(y_true)) < 2:
        tn, fp, fn, tp = (len(y_true), 0, 0, 0) if np.all(y_true==0) else (0,0,0,len(y_true))
    else:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred_binary).ravel()
    return {
        'roc_auc': roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.5,
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred_binary),
        'f1': f1_score(y_true, y_pred_binary, zero_division=0),
        'recall': recall_score(y_true, y_pred_binary, zero_division=0),
        'precision': precision_score(y_true, y_pred_binary, zero_division=0),
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'y_true': y_true,
        'y_pred_binary': y_pred_binary,
        'y_prob': y_prob,
    }

def save_results(model_name, df_results, output_pkl):
    """Saves the results DataFrame to a pickle file."""
    os.makedirs(os.path.dirname(output_pkl), exist_ok=True)
    df_collapsed = pd.DataFrame({col: [df_results[col].values] for col in df_results.columns})
    df_collapsed['model_name'] = model_name
    
    if os.path.exists(output_pkl):
        try:
            df_output = pd.read_pickle(output_pkl)
            if not df_output.empty:
                df_output = df_output[df_output['model_name'] != model_name]
            df_output = pd.concat([df_output, df_collapsed], ignore_index=True)
        except (EOFError, FileNotFoundError):
            df_output = df_collapsed
    else:
        df_output = df_collapsed
    df_output.to_pickle(output_pkl)
    print(f"Results for '{model_name}' saved to {output_pkl}")

# =============================================================================
# DATA SPLITTING CLASS
# =============================================================================
class BootstrapSplitter:
    def __init__(self, df, use_bootstrapping=False, n_iterations=25):
        self.df = df
        self.use_bootstrapping = use_bootstrapping
        self.n_iterations = n_iterations if use_bootstrapping else 1
        self.i = 0
        self.i_df = 0
        self.df_fifths = []

    def __iter__(self):
        return self

    def __next__(self):
        if self.i >= self.n_iterations:
            raise StopIteration
        if self.use_bootstrapping:
            if self.i % 5 == 0:
                self.i_df = 0
                self.df_fifths = []
                df_remainder = self.df.copy()
                for remaining_fifths in range(5, 1, -1):
                    rest_df, fold_df = train_test_split(df_remainder, test_size=(1.0/remaining_fifths), random_state=RANDOM_STATE + (self.i // 5), stratify=df_remainder[TARGET])
                    self.df_fifths.append(fold_df)
                    df_remainder = rest_df
                self.df_fifths.append(df_remainder)
            test_df = self.df_fifths[self.i_df]
            train_dfs = [df for j, df in enumerate(self.df_fifths) if j != self.i_df]
            train_df = pd.concat(train_dfs)
            self.i_df += 1
        else:
            train_df, test_df = train_test_split(self.df, test_size=0.2, random_state=RANDOM_STATE, stratify=self.df[TARGET])
        self.i += 1
        return train_df, test_df

# =============================================================================
# UNIFIED PYTORCH MODEL
# =============================================================================
class HybridModel(nn.Module):
    def __init__(self, tabular_input_size, lstm_input_size, lstm_hidden_size, lstm_num_layers, mlp_dims, dropout_rate, mode='hybrid'):
        super(HybridModel, self).__init__()
        self.mode = mode
        if self.mode in ['lstm_only', 'hybrid']:
            lstm_dropout = dropout_rate if lstm_num_layers > 1 else 0
            self.lstm = nn.LSTM(
                input_size=lstm_input_size, 
                hidden_size=lstm_hidden_size, 
                num_layers=lstm_num_layers, 
                batch_first=True,
                dropout=lstm_dropout
            )
        if self.mode in ['mlp_only', 'hybrid']:
            mlp_layers = []
            in_features = tabular_input_size
            if mlp_dims:
                for dim in mlp_dims:
                    mlp_layers.append(nn.Linear(in_features, dim))
                    mlp_layers.append(nn.ReLU())
                    mlp_layers.append(nn.Dropout(dropout_rate))
                    in_features = dim
                self.mlp = nn.Sequential(*mlp_layers)
        
        if self.mode == 'hybrid':
            classifier_input_size = lstm_hidden_size + (mlp_dims[-1] if mlp_dims else 0)
        elif self.mode == 'lstm_only':
            classifier_input_size = lstm_hidden_size
        elif self.mode == 'mlp_only':
            classifier_input_size = mlp_dims[-1] if mlp_dims else 0
        
        # Add a guard for zero-sized inputs
        if classifier_input_size > 0:
            self.classifier = nn.Sequential(
                nn.Linear(classifier_input_size, (classifier_input_size // 2)),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.Linear((classifier_input_size // 2), 1)
            )
        else:
            # Create a dummy layer if input size is 0 to avoid errors
            self.classifier = nn.Identity()


    def forward(self, x_tab=None, x_time=None, seq_len=None):
        if self.mode == 'hybrid':
            packed_input = pack_padded_sequence(x_time, seq_len.cpu(), batch_first=True, enforce_sorted=False)
            _, (hn, _) = self.lstm(packed_input)
            lstm_out = hn[-1]
            mlp_out = self.mlp(x_tab)
            combined = torch.cat((lstm_out, mlp_out), dim=1)
        elif self.mode == 'lstm_only':
            packed_input = pack_padded_sequence(x_time, seq_len.cpu(), batch_first=True, enforce_sorted=False)
            _, (hn, _) = self.lstm(packed_input)
            combined = hn[-1]
        elif self.mode == 'mlp_only':
            combined = self.mlp(x_tab)
        return self.classifier(combined)

# =============================================================================
# TRAINING & EVALUATION FUNCTIONS
# =============================================================================

def train_evaluate_lstm_hybrid(model, train_loader, val_loader, test_loader, h_params, device):
    """
    Training loop for LSTM and Hybrid models using mini-batching (DataLoader).
    """
    optimizer = optim.Adam(model.parameters(), lr=h_params['learning_rate'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=h_params['lr_scheduler_factor'], patience=h_params['lr_scheduler_patience'])
    
    y_train_full = train_loader.dataset.tensors[3].cpu().numpy()
    pos_weight_val = np.sum(y_train_full == 0) / np.sum(y_train_full == 1) if np.sum(y_train_full == 1) > 0 else 1.0
    pos_weight = torch.tensor([pos_weight_val], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None

    for epoch in range(h_params['epochs']):
        model.train()
        for batch in train_loader:
            x_tab_batch, x_time_batch, seq_len_batch, y_batch = [b.to(device) for b in batch]
            optimizer.zero_grad()
            outputs = model(x_tab_batch, x_time_batch, seq_len_batch)
            loss = criterion(outputs, y_batch.unsqueeze(1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=h_params['gradient_clip_value'])
            optimizer.step()
        
        if (epoch + 1) % h_params['es_check_interval'] == 0:
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for batch in val_loader:
                    x_tab_batch, x_time_batch, seq_len_batch, y_batch = [b.to(device) for b in batch]
                    val_outputs = model(x_tab_batch, x_time_batch, seq_len_batch)
                    val_loss += criterion(val_outputs, y_batch.unsqueeze(1)).item()
            
            avg_val_loss = val_loss / len(val_loader)
            scheduler.step(avg_val_loss)
            
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                patience_counter = 0
                best_model_state = copy.deepcopy(model.state_dict())
            else:
                patience_counter += 1
            
            if patience_counter >= h_params['patience']:
                print(f"  Stopping early at epoch {epoch + 1} due to no improvement.")
                break
    
    if best_model_state:
        model.load_state_dict(best_model_state)
    
    model.eval()
    all_probs, all_true = [], []
    with torch.no_grad():
        for batch in test_loader:
            x_tab_batch, x_time_batch, seq_len_batch, y_batch = [b.to(device) for b in batch]
            test_outputs = model(x_tab_batch, x_time_batch, seq_len_batch)
            all_probs.append(torch.sigmoid(test_outputs).cpu())
            all_true.append(y_batch.cpu())
            
    y_prob = torch.cat(all_probs).numpy().flatten()
    y_true = torch.cat(all_true).numpy()
    y_pred = (y_prob >= 0.5).astype(int)
    
    return performance_dict(y_true, y_pred, y_prob)

def train_evaluate_mlp(model, train_tensors, val_tensors, test_tensors, h_params, device):
    """
    A lightweight, fast training loop for the MLP-only model using full-batch updates.
    """
    X_train_tensor, _, _, y_train_tensor = train_tensors
    X_val_tensor, _, _, y_val_tensor = val_tensors
    X_test_tensor, _, _, y_test_tensor = test_tensors
    
    # Move all data to GPU at once
    X_train_tensor = X_train_tensor.to(device)
    y_train_tensor = y_train_tensor.to(device).unsqueeze(1)
    X_val_tensor = X_val_tensor.to(device)
    X_test_tensor = X_test_tensor.to(device)

    optimizer = optim.Adam(model.parameters(), lr=h_params['learning_rate'])
    pos_weight = torch.tensor([torch.sum(y_train_tensor == 0) / torch.sum(y_train_tensor == 1)], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_val_auc, patience_counter, best_model_state = 0, 0, None
    
    for epoch in range(5000): 
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train_tensor)
        loss = criterion(outputs, y_train_tensor)
        loss.backward()
        optimizer.step()
        
        if (epoch + 1) % 10 == 0:
            model.eval()
            with torch.no_grad():
                val_probs = torch.sigmoid(model(X_val_tensor)).cpu().numpy().flatten()
            
            current_val_auc = roc_auc_score(y_val_tensor.cpu().numpy(), val_probs)
            if current_val_auc > best_val_auc:
                best_val_auc = current_val_auc
                patience_counter = 0
                best_model_state = copy.deepcopy(model.state_dict())
            else:
                patience_counter += 1
            if patience_counter >= 20: 
                print(f"  Stopping early at epoch {epoch + 1} due to no improvement in validation AUC.")
                break
    
    if best_model_state:
        model.load_state_dict(best_model_state)

    model.eval()
    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_test_tensor)).cpu().numpy().flatten()
    
    y_true = y_test_tensor.cpu().numpy()
    y_pred = (y_prob >= 0.5).astype(int)

    return performance_dict(y_true, y_pred, y_prob)


# =============================================================================
# MAIN EXECUTION BLOCK
# =============================================================================
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True
    print(f"Using device: {device}")

    # --- Load Data Sources ---
    try:
        print(f"Loading time-series data from {LSTM_INPUT_PKL}...")
        df_lstm_source = pd.read_pickle(LSTM_INPUT_PKL)
        num_ts_features = df_lstm_source['time_tensors'].iloc[0].shape[1]
        print(f"--> Found {num_ts_features} time-series features.")
    except FileNotFoundError:
        print(f"ERROR: Time-series data file not found at {LSTM_INPUT_PKL}. Exiting.")
        return
        
    try:
        print(f"Loading tabular data from {MLP_INPUT_CSV}...")
        df_mlp_source = pd.read_csv(MLP_INPUT_CSV)
    except FileNotFoundError:
        print(f"ERROR: Tabular data file not found at {MLP_INPUT_CSV}. Exiting.")
        return
    
    base_results_saved = False # Flag to ensure base model is only saved once

    # --- Main Loop for Models ---
    for model_name, should_run in model_configs.items():
        if not should_run:
            print(f"\nSkipping {model_name.upper()} model.")
            continue
        
        # ADDED: try-except block for tmux reliability
        try:
            print(f"\n{'='*25} RUNNING MODEL: {model_name.upper()} {'='*25}")

            # --- Select Hyperparameters for the current model ---
            h_params = default_hyperparameters.copy()
            if model_name == 'lstm_only' and hpo_params_lstm_only:
                h_params.update(hpo_params_lstm_only)
                print("--> Using HPO parameters for LSTM_ONLY model.")
            elif model_name == 'hybrid' and hpo_params_hybrid:
                h_params.update(hpo_params_hybrid)
                print("--> Using HPO parameters for HYBRID model.")
            else:
                print("--> Using DEFAULT parameters.")


            # --- Data Preparation based on Model Type ---
            if model_name == 'mlp_only':
                df_for_run = df_mlp_source
                feature_cols_tab = [col for col in df_for_run.columns if col not in ['op_id', TARGET]]
            else:
                df_lstm_subset = df_lstm_source[['op_id', 'time_tensors', 'seq_len', TARGET]]
                df_for_run = pd.merge(df_lstm_subset, df_mlp_source.drop(columns=[TARGET], errors='ignore'), on='op_id', how='inner')
                feature_cols_tab = [col for col in df_mlp_source.columns if col not in ['op_id', TARGET]]
                print(f"For {model_name}, found {len(df_for_run)} patients common to both data sources.")

            if df_for_run.empty:
                print(f"--> ERROR: No data available for model '{model_name}'. Check for matching 'op_id's if merging. Skipping.")
                continue

            def df_to_tensors(sub_df):
                X_tab = torch.tensor(sub_df[feature_cols_tab].values, dtype=torch.float32)
                y = torch.tensor(sub_df[TARGET].values, dtype=torch.float32)
                if 'time_tensors' in sub_df.columns:
                    X_time = torch.stack([t.clone().detach() for t in sub_df['time_tensors']]).to(torch.float32)
                    seq_len = torch.tensor(sub_df['seq_len'].tolist(), dtype=torch.long)
                else:
                    n_samples = len(sub_df)
                    X_time = torch.zeros(n_samples, 1, 1, dtype=torch.float32)
                    seq_len = torch.zeros(n_samples, dtype=torch.long)
                return X_tab, X_time, seq_len, y

            df_results = pd.DataFrame()
            splitter = BootstrapSplitter(df_for_run, use_bootstrapping=USE_BOOTSTRAPPING, n_iterations=N_BOOTSTRAP_ITERATIONS)
            
            for i, (train_df, test_df) in enumerate(splitter, 1):
                print(f"--- Starting Run {i}/{splitter.n_iterations} for {model_name.upper()} ---")
                start_time = time.time()
                
                train_df, val_df = train_test_split(train_df, test_size=0.15, random_state=RANDOM_STATE + i, stratify=train_df[TARGET])

                scaler = StandardScaler()
                train_df.loc[:, feature_cols_tab] = scaler.fit_transform(train_df[feature_cols_tab])
                val_df.loc[:, feature_cols_tab] = scaler.transform(val_df[feature_cols_tab])
                test_df.loc[:, feature_cols_tab] = scaler.transform(test_df[feature_cols_tab])

                # --- Model Initialization ---
                train_tensors = df_to_tensors(train_df)
                val_tensors = df_to_tensors(val_df)
                test_tensors = df_to_tensors(test_df)

                lstm_input_size = train_tensors[1].shape[2] if train_tensors[1].numel() > 1 else 0
                
                model = HybridModel(
                    tabular_input_size=len(feature_cols_tab),
                    lstm_input_size=lstm_input_size,
                    lstm_hidden_size=h_params['lstm_hidden_size'],
                    lstm_num_layers=h_params['lstm_num_layers'],
                    mlp_dims=h_params.get('mlp_dims', []), # Use .get for safety
                    dropout_rate=h_params['dropout_rate'],
                    mode=model_name
                ).to(device)

                # --- Select appropriate training function ---
                if model_name == 'mlp_only':
                    perf = train_evaluate_mlp(model, train_tensors, val_tensors, test_tensors, h_params, device)
                else:
                    train_dataset = TensorDataset(*train_tensors)
                    val_dataset = TensorDataset(*val_tensors)
                    test_dataset = TensorDataset(*test_tensors)
                    
                    train_loader = DataLoader(train_dataset, batch_size=h_params['batch_size'], shuffle=True, pin_memory=True, num_workers=4)
                    val_loader = DataLoader(val_dataset, batch_size=h_params['batch_size'], shuffle=False, pin_memory=True, num_workers=4)
                    test_loader = DataLoader(test_dataset, batch_size=h_params['batch_size'], shuffle=False, pin_memory=True, num_workers=4)
                    
                    perf = train_evaluate_lstm_hybrid(model, train_loader, val_loader, test_loader, h_params, device)

                df_results = pd.concat([df_results, pd.DataFrame([perf])], ignore_index=True)
                
                end_time = time.time()
                print(f"--- Run {i} Finished in {end_time - start_time:.2f} seconds ---")
                print(f"    AUROC: {perf['roc_auc']:.4f}, F1: {perf['f1']:.4f}, Recall: {perf['recall']:.4f}, Precision: {perf['precision']:.4f}\n")

            if not df_results.empty:
                # Save to the main consolidated results file
                save_results(f"lstm_{model_name}", df_results, RESULTS_PKL)

                # Save results to additional files for specific models
                if model_name == 'lstm_only':
                    save_results('lstm', df_results, INTRAOP_RESULTS_PKL)
                elif model_name == 'hybrid':
                    save_results('hybrid', df_results, COMBINED_RESULTS_PKL)

                if len(df_results) > 1:
                    print(f"--- Overall {model_name.upper()} Performance Summary (Mean +/- Std Dev) ---")
                    for metric in ['roc_auc', 'f1', 'recall', 'precision']:
                        mean_val = df_results[metric].mean()
                        std_val = df_results[metric].std()
                        print(f"  {metric.capitalize()}: {mean_val:.4f} +/- {std_val:.4f}")

            # --- Save base model results after the first model run ---
            if not base_results_saved and not df_results.empty:
                print("\n--- Creating and saving 'base_54k' ground truth model ---")
                # Create a DataFrame in the format expected by the plotting script
                # It stores the true labels from the test set in the 'y_pred_binary' column
                base_df = pd.DataFrame({'y_pred_binary': df_results['y_true']})
                
                # Save this base model to all three results files
                save_results('base_54k', base_df, RESULTS_PKL)
                save_results('base_54k', base_df, INTRAOP_RESULTS_PKL)
                save_results('base_54k', base_df, COMBINED_RESULTS_PKL)
                
                base_results_saved = True # Set flag to prevent this from running again
        
        except Exception as e:
            print(f"\n{'!'*25} ERROR ENCOUNTERED {'!'*25}")
            print(f"An error occurred while running model: {model_name}")
            print(f"Error details: {e}")
            print(f"Skipping to next model.")
            print(f"{'!'*65}\n")


    print("\n\nScript finished.")

if __name__ == "__main__":
    main()


Using device: cuda
Loading time-series data from /home/server/Projects/data/AKI/lstm_trainable.pkl...
--> Found 24 time-series features.
Loading tabular data from /home/server/Projects/data/AKI/tabular_preop.csv...

========================= RUNNING MODEL: LSTM_ONLY =========================
--> Using HPO parameters for LSTM_ONLY model.
For lstm_only, found 54085 patients common to both data sources.
--- Starting Run 1/1 for LSTM_ONLY ---
  Stopping early at epoch 110 due to no improvement.
--- Run 1 Finished in 434.30 seconds ---
    AUROC: 0.7296, F1: 0.2014, Recall: 0.6487, Precision: 0.1192

Results for 'lstm_lstm_only' saved to /home/server/Projects/data/AKI/results/lstm_hybrid_test_optimized.pkl
Results for 'lstm' saved to /home/server/Projects/data/AKI/results/tabular_intraop_test.pkl

--- Creating and saving 'base_54k' ground truth model ---
Results for 'base_54k' saved to /home/server/Projects/data/AKI/results/lstm_hybrid_test_optimized.pkl
Results for 'base_54k' saved to /hom

KeyboardInterrupt: 

: 

# HPO Hybrid

In [3]:
#!/usr/bin/env python3
import os
import time
import copy
import numpy as np
import pandas as pd
import optuna

# Scikit-learn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import TensorDataset, DataLoader

# =============================================================================
# SCRIPT CONFIGURATION
# =============================================================================
TARGET = 'aki_boolean'
RANDOM_STATE = 42
N_TRIALS = 50  # Number of HPO trials to run for each model

# --- I/O Configuration ---
BASE_DATA_DIR = '/home/server/Projects/data/AKI/'
LSTM_INPUT_PKL = os.path.join(BASE_DATA_DIR, 'lstm_trainable.pkl')
MLP_INPUT_CSV = os.path.join(BASE_DATA_DIR, 'tabular_preop.csv')
RESULTS_FILE_PATH = os.path.join(BASE_DATA_DIR, 'results/hybrid_hpo_results.txt')

# --- Model HPO Toggles ---
hpo_configs = {
    'lstm_only': True,
    'hybrid': True,
}

# --- Search Space Definitions ---
# These define the range of hyperparameters Optuna will search over.
search_spaces = {
    'lstm_only': {
        'lr': (1e-5, 1e-2),
        'lstm_hidden_size': (16, 256),
        'lstm_num_layers': (1, 4),
        'dropout_rate': (0.1, 0.6)
    },
    'hybrid': {
        'lr': (1e-5, 1e-2),
        'lstm_hidden_size': (16, 128),
        'lstm_num_layers': (1, 3),
        'n_mlp_layers': (1, 4),
        'mlp_layer_size': (16, 256), # Range for individual MLP layer sizes
        'dropout_rate': (0.1, 0.6)
    }
}

# =============================================================================
# HYBRID MODEL DEFINITION (Copied from main training script)
# =============================================================================
class HybridModel(nn.Module):
    def __init__(self, tabular_input_size, lstm_input_size, lstm_hidden_size, lstm_num_layers, mlp_dims, dropout_rate, mode='hybrid'):
        super(HybridModel, self).__init__()
        self.mode = mode
        if self.mode in ['lstm_only', 'hybrid']:
            lstm_dropout = dropout_rate if lstm_num_layers > 1 else 0
            self.lstm = nn.LSTM(
                input_size=lstm_input_size,
                hidden_size=lstm_hidden_size,
                num_layers=lstm_num_layers,
                batch_first=True,
                dropout=lstm_dropout
            )
        if self.mode in ['mlp_only', 'hybrid']:
            mlp_layers = []
            in_features = tabular_input_size
            # Handle empty mlp_dims for lstm_only case
            if mlp_dims:
                for dim in mlp_dims:
                    mlp_layers.append(nn.Linear(in_features, dim))
                    mlp_layers.append(nn.ReLU())
                    mlp_layers.append(nn.Dropout(dropout_rate))
                    in_features = dim
                self.mlp = nn.Sequential(*mlp_layers)
        
        if self.mode == 'hybrid':
            # Guard against empty mlp_dims list
            classifier_input_size = lstm_hidden_size + (mlp_dims[-1] if mlp_dims else 0)
        elif self.mode == 'lstm_only':
            classifier_input_size = lstm_hidden_size
        elif self.mode == 'mlp_only':
            classifier_input_size = mlp_dims[-1]
        
        self.classifier = nn.Sequential(
            nn.Linear(classifier_input_size, (classifier_input_size // 2)),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear((classifier_input_size // 2), 1)
        )

    def forward(self, x_tab=None, x_time=None, seq_len=None):
        if self.mode == 'hybrid':
            packed_input = pack_padded_sequence(x_time, seq_len.cpu(), batch_first=True, enforce_sorted=False)
            _, (hn, _) = self.lstm(packed_input)
            lstm_out = hn[-1]
            mlp_out = self.mlp(x_tab)
            combined = torch.cat((lstm_out, mlp_out), dim=1)
        elif self.mode == 'lstm_only':
            packed_input = pack_padded_sequence(x_time, seq_len.cpu(), batch_first=True, enforce_sorted=False)
            _, (hn, _) = self.lstm(packed_input)
            combined = hn[-1]
        elif self.mode == 'mlp_only':
            combined = self.mlp(x_tab)
        return self.classifier(combined)

# =============================================================================
# HPO HELPER FUNCTIONS
# =============================================================================

def check_search_space_boundaries(study, search_space):
    """Checks if the best parameters are at the boundaries of the search space."""
    warnings = []
    best_params = study.best_params

    for param, value in best_params.items():
        # Handle dynamic mlp layer sizes, which all use the 'mlp_layer_size' search space
        search_key = 'mlp_layer_size' if param.startswith('mlp_layer_') else param
        
        if search_key not in search_space:
            continue

        min_val, max_val = search_space[search_key]

        if isinstance(value, (int, float)) and (np.isclose(value, min_val) or np.isclose(value, max_val)):
            warnings.append(
                f"  - WARNING for {study.study_name}: Best value for '{param}' ({value}) is at the boundary "
                f"of its search space [{min_val}, {max_val}]. Consider expanding the range."
            )
    
    if warnings:
        print("\n" + "="*20 + " BOUNDARY WARNINGS " + "="*20)
        for warning in warnings:
            print(warning)
        print("="*63 + "\n")


def save_hpo_results_to_file(filepath, results_dict):
    """Formats and saves the final HPO results to a text file."""
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w') as f:
        f.write("# Hyperparameter Optimization Results for LSTM/Hybrid Models\n")
        f.write(f"# Generated on: {time.ctime()}\n\n")
        f.write("# --- COPY-PASTE A DICTIONARY BELOW INTO THE MAIN SCRIPT ---\n")
        
        for model_name, params in results_dict.items():
            f.write(f"\n# --- Best for {model_name.upper()} ---\n")
            f.write(f"hpo_params_{model_name} = {{\n")
            for key, value in params.items():
                if isinstance(value, list):
                     f.write(f"    '{key}': {value},\n")
                elif isinstance(value, float):
                     f.write(f"    '{key}': {value:.6f},\n")
                else:
                     f.write(f"    '{key}': {value},\n")
            f.write("}\n")
    print(f"\nFinal HPO results saved to: {filepath}")

def objective_builder(model_name, train_loader, val_loader, tabular_input_size, lstm_input_size, device):
    """Builds the Optuna objective function for a given model."""
    
    def objective(trial):
        # --- Suggest Hyperparameters ---
        lr = trial.suggest_float('lr', *search_spaces[model_name]['lr'], log=True)
        dropout_rate = trial.suggest_float('dropout_rate', *search_spaces[model_name]['dropout_rate'])
        
        # Model-specific architecture parameters
        if model_name == 'lstm_only':
            lstm_hidden_size = trial.suggest_int('lstm_hidden_size', *search_spaces[model_name]['lstm_hidden_size'])
            lstm_num_layers = trial.suggest_int('lstm_num_layers', *search_spaces[model_name]['lstm_num_layers'])
            mlp_dims = [] # No MLP for this model
        elif model_name == 'hybrid':
            lstm_hidden_size = trial.suggest_int('lstm_hidden_size', *search_spaces[model_name]['lstm_hidden_size'])
            lstm_num_layers = trial.suggest_int('lstm_num_layers', *search_spaces[model_name]['lstm_num_layers'])
            n_mlp_layers = trial.suggest_int('n_mlp_layers', *search_spaces[model_name]['n_mlp_layers'])
            
            mlp_dims = []
            for i in range(n_mlp_layers):
                layer_size = trial.suggest_int(f'mlp_layer_{i}_size', *search_spaces[model_name]['mlp_layer_size'])
                mlp_dims.append(layer_size)
        else:
            raise ValueError(f"Unsupported model_name for HPO: {model_name}")

        # --- Setup Model and Training ---
        model = HybridModel(
            tabular_input_size=tabular_input_size,
            lstm_input_size=lstm_input_size,
            lstm_hidden_size=lstm_hidden_size,
            lstm_num_layers=lstm_num_layers,
            mlp_dims=mlp_dims,
            dropout_rate=dropout_rate,
            mode=model_name
        ).to(device)

        optimizer = optim.Adam(model.parameters(), lr=lr)
        y_train_numpy = train_loader.dataset.tensors[3].numpy()
        pos_weight_val = np.sum(y_train_numpy == 0) / np.sum(y_train_numpy == 1) if np.sum(y_train_numpy == 1) > 0 else 1.0
        pos_weight = torch.tensor([pos_weight_val], device=device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        best_val_metric = 0 # Now represents balanced accuracy
        patience_counter = 0

        # --- Simplified Training Loop for HPO ---
        for epoch in range(150): # Run for a max of 150 epochs per trial
            model.train()
            for batch in train_loader:
                x_tab_batch, x_time_batch, seq_len_batch, y_batch = [b.to(device) for b in batch]
                optimizer.zero_grad()
                outputs = model(x_tab_batch, x_time_batch, seq_len_batch)
                loss = criterion(outputs, y_batch.unsqueeze(1))
                loss.backward()
                optimizer.step()

            # --- Validation and Early Stopping ---
            model.eval()
            all_val_probs = []
            with torch.no_grad():
                for batch in val_loader:
                    x_tab_batch, x_time_batch, seq_len_batch, _ = [b.to(device) for b in batch]
                    val_outputs = model(x_tab_batch, x_time_batch, seq_len_batch)
                    all_val_probs.append(torch.sigmoid(val_outputs).cpu())

            val_probs = torch.cat(all_val_probs).numpy().flatten()
            
            # --- Use Balanced Accuracy ---
            val_pred_binary = (val_probs >= 0.5).astype(int)
            current_val_metric = balanced_accuracy_score(val_loader.dataset.tensors[3].numpy(), val_pred_binary)

            if current_val_metric > best_val_metric:
                best_val_metric = current_val_metric
                patience_counter = 0
            else:
                patience_counter += 1
            
            # Pruning and early stopping
            trial.report(best_val_metric, epoch)
            if trial.should_prune() or patience_counter >= 15:
                raise optuna.exceptions.TrialPruned()

        return best_val_metric
    
    return objective

# =============================================================================
# MAIN EXECUTION BLOCK
# =============================================================================
def main():
    """Main function to run the HPO process."""
    torch.manual_seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # --- Load Data Sources ---
    try:
        df_lstm_source = pd.read_pickle(LSTM_INPUT_PKL)
    except FileNotFoundError:
        print(f"ERROR: LSTM data not found at {LSTM_INPUT_PKL}. Exiting.")
        return
        
    try:
        df_mlp_source = pd.read_csv(MLP_INPUT_CSV)
    except FileNotFoundError:
        print(f"ERROR: Tabular data not found at {MLP_INPUT_CSV}. Exiting.")
        return

    all_best_hpo_results = {}
    
    for model_name, should_run in hpo_configs.items():
        if not should_run:
            continue

        print(f"\n{'='*25} STARTING HPO FOR: {model_name.upper()} {'='*25}")
        
        # --- Prepare Data for the specific model ---
        if model_name == 'hybrid':
            df_lstm_subset = df_lstm_source[['op_id', 'time_tensors', 'seq_len', TARGET]]
            df_for_run = pd.merge(df_lstm_subset, df_mlp_source.drop(columns=[TARGET], errors='ignore'), on='op_id', how='inner')
        else: # lstm_only
            df_for_run = df_lstm_source
        
        feature_cols_tab = [col for col in df_mlp_source.columns if col not in ['op_id', TARGET]]

        def df_to_tensors(sub_df):
            # CORRECTED: Check if tabular columns exist before trying to create a tensor
            if feature_cols_tab and all(col in sub_df.columns for col in feature_cols_tab):
                X_tab = torch.tensor(sub_df[feature_cols_tab].values, dtype=torch.float32)
            else:
                X_tab = torch.empty(len(sub_df), 0)

            y = torch.tensor(sub_df[TARGET].values, dtype=torch.float32)
            
            # Time-series data is assumed to be present for both models in this HPO script
            X_time = torch.stack([t.clone().detach() for t in sub_df['time_tensors']]).to(torch.float32)
            seq_len = torch.tensor(sub_df['seq_len'].tolist(), dtype=torch.long)
            
            return X_tab, X_time, seq_len, y

        # --- Create a single train/validation split for the HPO process ---
        train_val_df, _ = train_test_split(df_for_run, test_size=0.2, random_state=RANDOM_STATE, stratify=df_for_run[TARGET])
        train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=RANDOM_STATE, stratify=train_val_df[TARGET])

        scaler = StandardScaler()
        # CORRECTED: Only scale if the model is 'hybrid' and thus has the tabular columns
        if model_name == 'hybrid':
             train_df.loc[:, feature_cols_tab] = scaler.fit_transform(train_df[feature_cols_tab])
             val_df.loc[:, feature_cols_tab] = scaler.transform(val_df[feature_cols_tab])
        
        train_dataset = TensorDataset(*df_to_tensors(train_df))
        val_dataset = TensorDataset(*df_to_tensors(val_df))
        
        # MODIFIED: Batch size reduced to 1024 to prevent OOM errors
        train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=1024)

        # --- Run Optuna Study ---
        study = optuna.create_study(direction='maximize', study_name=f"{model_name}_hpo")
        objective_func = objective_builder(
            model_name, train_loader, val_loader, 
            tabular_input_size=len(feature_cols_tab) if model_name == 'hybrid' else 0,
            lstm_input_size=train_dataset.tensors[1].shape[2],
            device=device
        )
        study.optimize(objective_func, n_trials=N_TRIALS, show_progress_bar=True)

        print(f"Best trial for {model_name.upper()}: Balanced Accuracy = {study.best_value:.4f}")
        
        # --- Console printout and boundary check ---
        print("\n--- Best Parameters Found ---")
        best_params = study.best_params
        # Re-create mlp_dims for clean output if necessary
        if 'n_mlp_layers' in best_params:
            mlp_dims = [best_params[f'mlp_layer_{i}_size'] for i in range(best_params['n_mlp_layers'])]
            final_params = {'mlp_dims': mlp_dims}
            # Add other non-layer-specific params
            for key, val in best_params.items():
                if not key.startswith('mlp_layer_') and key != 'n_mlp_layers':
                    final_params[key] = val
            all_best_hpo_results[model_name] = final_params
        else:
             all_best_hpo_results[model_name] = best_params

        # Print final formatted parameters
        for key, value in all_best_hpo_results[model_name].items():
            print(f"  - {key}: {value}")
        
        check_search_space_boundaries(study, search_spaces[model_name])


    # --- Save Final Results ---
    if all_best_hpo_results:
        save_hpo_results_to_file(RESULTS_FILE_PATH, all_best_hpo_results)

if __name__ == "__main__":
    main()


Using device: cuda

========================= STARTING HPO FOR: LSTM_ONLY =========================


  0%|          | 0/50 [00:01<?, ?it/s]


[W 2025-06-14 20:48:14,338] Trial 0 failed with parameters: {'lr': 0.001197197044887897, 'dropout_rate': 0.1942826487007748, 'lstm_hidden_size': 212, 'lstm_num_layers': 1} because of the following error: OutOfMemoryError('CUDA out of memory. Tried to allocate 350.00 MiB. GPU 0 has a total capacity of 2.94 GiB of which 181.44 MiB is free. Process 224648 has 2.56 GiB memory in use. Including non-PyTorch memory, this process has 198.00 MiB memory in use. Of the allocated memory 137.25 MiB is allocated by PyTorch, and 6.75 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)').
Traceback (most recent call last):
  File "/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/optuna/study/_optimize.py", line 197, in _run_trial


OutOfMemoryError: CUDA out of memory. Tried to allocate 350.00 MiB. GPU 0 has a total capacity of 2.94 GiB of which 181.44 MiB is free. Process 224648 has 2.56 GiB memory in use. Including non-PyTorch memory, this process has 198.00 MiB memory in use. Of the allocated memory 137.25 MiB is allocated by PyTorch, and 6.75 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)